In [ ]:
# Notebook 13 — SHAP Explainability Analysis

## Purpose

This notebook performs post-hoc explainability analysis of the final development-selected XGBoost model using SHapley Additive exPlanations (SHAP).

The purpose of the analysis is to determine how the frozen XGBoost model attributes its stack-voltage predictions to the selected operational and engineered predictors, with particular emphasis on generalisation to the unseen later durability stages of 900 h, 950 h, and 1000 h.

The explainability analysis is conducted only after completion of feature selection, hyperparameter tuning, chronological model evaluation, model-family comparison, and final later-stage predictive assessment. Therefore, SHAP is used strictly as a post-hoc interpretation method and does not feed back into model development or model selection.

The primary SHAP design is:

- Primary model: frozen XGBoost model selected during development-stage chronological validation.
- Primary explanation regime: representative observations from the unseen 900–1000 h later-stage holdout.
- Background/reference regime: representative observations from the 50–850 h development data.
- Primary TreeSHAP formulation: interventional TreeSHAP.
- Sensitivity formulation: tree-path-dependent TreeSHAP.
- Interpretation: predictive model attribution rather than causal or uniquely physical feature effects.

Because several PEMFC predictors are strongly correlated or algebraically related, individual SHAP values will be interpreted cautiously and, where appropriate, alongside related subsystem variables and established PEMFC behaviour.

In [ ]:
## 13.1 Setup and Reproducibility

This section establishes the computational environment required for the SHAP analysis.

The objective is to ensure that all subsequent explanation-sampling, background-sampling, TreeSHAP computation, stability assessment, and figure generation steps are reproducible.

A fixed random seed is used throughout the notebook for all study-specific sampling procedures. Package versions are also recorded because SHAP behaviour and TreeExplainer options can vary between software versions.

Dedicated output directories are created for:

- SHAP figures,
- SHAP numerical results,
- robustness and stability outputs,
- and intermediate reproducibility artefacts.

No model training, feature selection, hyperparameter tuning, or model re-selection is performed in this notebook.

In [3]:
# ============================================================
# 13.1 Setup and Reproducibility
# ============================================================

from pathlib import Path
import sys
import platform
import random

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb
import shap
import joblib


# ------------------------------------------------------------
# 1. Reproducibility settings
# ------------------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# ------------------------------------------------------------
# 2. Display settings
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")


# ------------------------------------------------------------
# 3. Project and output directories
# ------------------------------------------------------------

# Notebook is expected to run from:
# PEMFC_Dissertation/notebooks/
#
# Therefore, the project root is one level above the current
# working directory.

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

OUTPUT_DIR = PROJECT_ROOT / "results" / "shap_analysis"
FIGURE_DIR = PROJECT_ROOT / "figures" / "shap_analysis"
ROBUSTNESS_DIR = OUTPUT_DIR / "robustness"
EXPORT_DIR = OUTPUT_DIR / "exports"

for directory in [
    OUTPUT_DIR,
    FIGURE_DIR,
    ROBUSTNESS_DIR,
    EXPORT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)


print("=" * 70)
print("PROJECT DIRECTORY CHECK")
print("=" * 70)

print(f"\nCurrent working directory:")
print(f"  {CURRENT_DIR}")

print(f"\nProject root:")
print(f"  {PROJECT_ROOT}")

print("\nSHAP output directories:")
print(f"  Results:    {OUTPUT_DIR}")
print(f"  Figures:    {FIGURE_DIR}")
print(f"  Robustness: {ROBUSTNESS_DIR}")
print(f"  Exports:    {EXPORT_DIR}")


# ------------------------------------------------------------
# 4. Record software environment
# ------------------------------------------------------------

environment_info = pd.DataFrame(
    {
        "Component": [
            "Python",
            "Platform",
            "NumPy",
            "pandas",
            "scikit-learn",
            "XGBoost",
            "SHAP",
            "Matplotlib",
            "joblib",
        ],
        "Version": [
            sys.version.split()[0],
            platform.platform(),
            np.__version__,
            pd.__version__,
            sklearn.__version__,
            xgb.__version__,
            shap.__version__,
            matplotlib.__version__,
            joblib.__version__,
        ],
    }
)

print("=" * 70)
print("NOTEBOOK 13 — SHAP EXPLAINABILITY ANALYSIS")
print("=" * 70)

print(f"\nRandom seed: {RANDOM_SEED}")

print("\nSoftware environment:")
display(environment_info)


# ------------------------------------------------------------
# 5. Save software environment for reproducibility
# ------------------------------------------------------------

environment_path = OUTPUT_DIR / "shap_software_environment.csv"
environment_info.to_csv(environment_path, index=False)

print("\nReproducibility record saved to:")
print(environment_path)


# ------------------------------------------------------------
# 6. Methodological safeguard
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("METHODOLOGICAL SAFEGUARD")
print("=" * 70)

print(
    "Notebook 13 is a post-hoc explainability notebook.\n"
    "No feature selection, hyperparameter tuning, model training,\n"
    "or model-family re-selection will be performed here."
)

PROJECT DIRECTORY CHECK

Current working directory:
  C:\Users\usman\Desktop\PEMFC_Dissertation\notebooks

Project root:
  C:\Users\usman\Desktop\PEMFC_Dissertation

SHAP output directories:
  Results:    C:\Users\usman\Desktop\PEMFC_Dissertation\results\shap_analysis
  Figures:    C:\Users\usman\Desktop\PEMFC_Dissertation\figures\shap_analysis
  Robustness: C:\Users\usman\Desktop\PEMFC_Dissertation\results\shap_analysis\robustness
  Exports:    C:\Users\usman\Desktop\PEMFC_Dissertation\results\shap_analysis\exports
NOTEBOOK 13 — SHAP EXPLAINABILITY ANALYSIS

Random seed: 42

Software environment:


,Component,Version
0,Python,3.13.11
1,Platform,Windows-11-10.0.26200-SP0
2,NumPy,2.4.6
3,pandas,3.0.3
4,scikit-learn,1.9.0
5,XGBoost,3.4.1
6,SHAP,0.52.0
7,Matplotlib,3.11.0
8,joblib,1.5.3



Reproducibility record saved to:
C:\Users\usman\Desktop\PEMFC_Dissertation\results\shap_analysis\shap_software_environment.csv

METHODOLOGICAL SAFEGUARD
Notebook 13 is a post-hoc explainability notebook.
No feature selection, hyperparameter tuning, model training,
or model-family re-selection will be performed here.


In [ ]:
## 13.2 Load and Verify the Frozen XGBoost Model and Required Data

This section loads the final XGBoost model and the processed dataset required for the explainability analysis.

The XGBoost model was previously selected as the primary predictive model using development-stage chronological validation. It is treated as frozen throughout this notebook; therefore, no model fitting, hyperparameter optimisation, feature selection, or model-family re-selection is permitted.

The data are reconstructed using the same temporal boundary used during model development:

- Development regime: 50–850 h
- Later-stage holdout regime: 900, 950, and 1000 h
- Target variable: stack voltage
- Predictor set: the same 20 predictors used by the final XGBoost model

The development data will subsequently provide the candidate background/reference observations for interventional TreeSHAP, whereas the later-stage holdout will provide the primary explanation observations.

At this stage, the objective is only to load the required artefacts and verify their availability. Predictor order, stage composition, and prediction reproducibility will be checked separately before any SHAP values are calculated.

In [4]:
# ============================================================
# 13.2 Load and Verify the Frozen XGBoost Model and Required Data
# ============================================================

# ------------------------------------------------------------
# 13.2.1 Locate Required Artefacts
# ------------------------------------------------------------

MODEL_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"


def list_files(directory, patterns):
    """
    Return files matching one or more patterns within a directory tree.
    """
    matches = []

    if directory.exists():
        for pattern in patterns:
            matches.extend(directory.rglob(pattern))

    return sorted(set(matches))


# ------------------------------------------------------------
# 1. Search for saved model artefacts
# ------------------------------------------------------------

model_files = list_files(
    MODEL_DIR,
    [
        "*.joblib",
        "*.pkl",
        "*.pickle",
        "*.json",
        "*.ubj",
    ],
)


# ------------------------------------------------------------
# 2. Search for processed data artefacts
# ------------------------------------------------------------

data_files = list_files(
    PROCESSED_DATA_DIR,
    [
        "*.csv",
        "*.parquet",
        "*.feather",
        "*.pkl",
        "*.pickle",
        "*.joblib",
    ],
)


# ------------------------------------------------------------
# 3. Display results
# ------------------------------------------------------------

print("=" * 70)
print("REQUIRED ARTEFACT SEARCH")
print("=" * 70)

print(f"\nModel directory:")
print(f"  {MODEL_DIR}")

print(f"\nProcessed data directory:")
print(f"  {PROCESSED_DATA_DIR}")


print("\n" + "-" * 70)
print("MODEL ARTEFACTS FOUND")
print("-" * 70)

if model_files:
    for i, path in enumerate(model_files, start=1):
        print(f"{i:>3}. {path.relative_to(PROJECT_ROOT)}")
else:
    print("No model artefacts found in the models directory.")


print("\n" + "-" * 70)
print("PROCESSED DATA ARTEFACTS FOUND")
print("-" * 70)

if data_files:
    for i, path in enumerate(data_files, start=1):
        print(f"{i:>3}. {path.relative_to(PROJECT_ROOT)}")
else:
    print("No processed data artefacts found in the processed-data directory.")


print("\n" + "=" * 70)
print("SEARCH SUMMARY")
print("=" * 70)

print(f"Model artefacts found: {len(model_files):,}")
print(f"Processed data artefacts found: {len(data_files):,}")

REQUIRED ARTEFACT SEARCH

Model directory:
  C:\Users\usman\Desktop\PEMFC_Dissertation\models

Processed data directory:
  C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed

----------------------------------------------------------------------
MODEL ARTEFACTS FOUND
----------------------------------------------------------------------
  1. models\final_ridge_model.joblib
  2. models\final_xgboost_model.joblib

----------------------------------------------------------------------
PROCESSED DATA ARTEFACTS FOUND
----------------------------------------------------------------------
  1. data\processed\dataset_structure_summary.csv
  2. data\processed\ml_variable_classification.csv
  3. data\processed\operational_cleaned.csv
  4. data\processed\operational_merged_raw.csv
  5. data\processed\overall_summary.csv
  6. data\processed\pemfc_feature_engineered.csv
  7. data\processed\variable_dictionary.csv

SEARCH SUMMARY
Model artefacts found: 2
Processed data artefacts found: 7


In [ ]:
### 13.2.2 Load the Frozen XGBoost Model and Feature-Engineered Dataset

The previously saved final XGBoost model is loaded directly from the model-development stage rather than being refitted in this notebook. This preserves the separation between predictive model development and post-hoc explainability analysis.

The feature-engineered operational dataset is also loaded because it contains the original and derived predictors used during model development.

Immediately after loading, basic integrity checks are performed to confirm:

- the model object is available and has not been retrained;
- the feature-engineered dataset has the expected dimensional structure;
- the voltage target and operating-hour variable are present;
- the required 20 predictors are available;
- and the durability stages required for development-background and later-stage explanation analysis are present.

No SHAP values are calculated at this stage.

In [5]:
# ============================================================
# 13.2.2 Load Frozen Model and Feature-Engineered Dataset
# ============================================================

# ------------------------------------------------------------
# 1. Define exact artefact paths
# ------------------------------------------------------------

XGB_MODEL_PATH = MODEL_DIR / "final_xgboost_model.joblib"
FEATURE_DATA_PATH = PROCESSED_DATA_DIR / "pemfc_feature_engineered.csv"


# ------------------------------------------------------------
# 2. Verify required files exist
# ------------------------------------------------------------

required_paths = {
    "Frozen XGBoost model": XGB_MODEL_PATH,
    "Feature-engineered dataset": FEATURE_DATA_PATH,
}

print("=" * 70)
print("ARTEFACT PATH VERIFICATION")
print("=" * 70)

for name, path in required_paths.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:<30}: {status}")
    print(f"  {path}")


missing_files = [
    name
    for name, path in required_paths.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Required artefact(s) missing: "
        + ", ".join(missing_files)
    )


# ------------------------------------------------------------
# 3. Load frozen XGBoost model
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING FROZEN XGBOOST MODEL")
print("=" * 70)

final_xgb_model = joblib.load(XGB_MODEL_PATH)

print(f"\nLoaded model type:")
print(f"  {type(final_xgb_model)}")

print("\nModel successfully loaded.")
print("No model fitting has been performed in Notebook 13.")


# ------------------------------------------------------------
# 4. Load feature-engineered operational dataset
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOADING FEATURE-ENGINEERED DATASET")
print("=" * 70)

pemfc_data = pd.read_csv(FEATURE_DATA_PATH)

print("\nDataset successfully loaded.")

print(f"\nDataset shape:")
print(f"  Rows:    {pemfc_data.shape[0]:,}")
print(f"  Columns: {pemfc_data.shape[1]:,}")

print("\nDataset columns:")
for i, column in enumerate(pemfc_data.columns, start=1):
    print(f"{i:>2}. {column}")


# ------------------------------------------------------------
# 5. Define frozen modelling variables
# ------------------------------------------------------------

TARGET = "voltage"
STAGE_COLUMN = "operating_hour"

selected_features = [
    "anode_pressure_diff",
    "anode_temp_diff",
    "cathode_pressure_diff",
    "current",
    "pressure_anode_outlet",
    "pressure_cathode_outlet",
    "temp_anode_dewpoint_water",
    "temp_anode_endplate",
    "temp_cathode_inlet",
    "total_anode_stack_flow",
    "total_cathode_stack_flow",
    "cathode_dewpoint_offset",
    "temp_cathode_dewpoint_water",
    "temp_anode_outlet",
    "pressure_cathode_inlet",
    "temp_cathode_outlet",
    "pressure_anode_inlet",
    "temp_anode_inlet",
    "cathode_temp_diff",
    "anode_dewpoint_offset",
]


DEVELOPMENT_STAGES = list(range(50, 851, 50))
HOLDOUT_STAGES = [900, 950, 1000]


# ------------------------------------------------------------
# 6. Required-column integrity check
# ------------------------------------------------------------

required_columns = (
    [TARGET, STAGE_COLUMN]
    + selected_features
)

missing_columns = [
    column
    for column in required_columns
    if column not in pemfc_data.columns
]

print("\n" + "=" * 70)
print("REQUIRED COLUMN CHECK")
print("=" * 70)

print(f"\nFrozen predictors expected: {len(selected_features)}")
print(f"Frozen predictors found:    "
      f"{sum(feature in pemfc_data.columns for feature in selected_features)}")

if missing_columns:
    print("\nMissing required columns:")
    for column in missing_columns:
        print(f"  - {column}")

    raise KeyError(
        "The feature-engineered dataset is missing required modelling columns."
    )

print("\nAll required target, stage, and predictor columns are present.")


# ------------------------------------------------------------
# 7. Durability-stage integrity check
# ------------------------------------------------------------

available_stages = sorted(
    pemfc_data[STAGE_COLUMN]
    .dropna()
    .unique()
    .tolist()
)

missing_development_stages = [
    stage
    for stage in DEVELOPMENT_STAGES
    if stage not in available_stages
]

missing_holdout_stages = [
    stage
    for stage in HOLDOUT_STAGES
    if stage not in available_stages
]


print("\n" + "=" * 70)
print("DURABILITY-STAGE CHECK")
print("=" * 70)

print("\nAvailable durability stages:")
print(available_stages)

print(f"\nExpected development stages:")
print(DEVELOPMENT_STAGES)

print(f"\nExpected holdout stages:")
print(HOLDOUT_STAGES)


if missing_development_stages or missing_holdout_stages:
    raise ValueError(
        "Expected development and/or holdout durability stages are missing."
    )

print("\nAll required development and holdout stages are present.")


# ------------------------------------------------------------
# 8. Stage-level observation counts
# ------------------------------------------------------------

stage_counts = (
    pemfc_data
    .groupby(STAGE_COLUMN)
    .size()
    .rename("n_observations")
    .reset_index()
)

print("\n" + "=" * 70)
print("OBSERVATIONS BY DURABILITY STAGE")
print("=" * 70)

display(stage_counts)


# ------------------------------------------------------------
# 9. Final loading safeguard
# ------------------------------------------------------------

print("=" * 70)
print("13.2.2 LOADING CHECK COMPLETE")
print("=" * 70)

print(
    "\nFrozen XGBoost model and feature-engineered data are available.\n"
    "No SHAP explainer has been created and no SHAP values have been\n"
    "calculated at this stage."
)

ARTEFACT PATH VERIFICATION
Frozen XGBoost model          : FOUND
  C:\Users\usman\Desktop\PEMFC_Dissertation\models\final_xgboost_model.joblib
Feature-engineered dataset    : FOUND
  C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed\pemfc_feature_engineered.csv

LOADING FROZEN XGBOOST MODEL

Loaded model type:
  <class 'xgboost.sklearn.XGBRegressor'>

Model successfully loaded.
No model fitting has been performed in Notebook 13.

LOADING FEATURE-ENGINEERED DATASET

Dataset successfully loaded.

Dataset shape:
  Rows:    3,629,680
  Columns: 24

Dataset columns:
 1. operating_hour
 2. time
 3. current
 4. voltage
 5. power
 6. pressure_anode_inlet
 7. pressure_anode_outlet
 8. pressure_cathode_inlet
 9. pressure_cathode_outlet
10. temp_anode_endplate
11. temp_anode_dewpoint_water
12. temp_anode_inlet
13. temp_anode_outlet
14. temp_cathode_dewpoint_water
15. temp_cathode_inlet
16. temp_cathode_outlet
17. total_anode_stack_flow
18. total_cathode_stack_flow
19. anode_pressure_diff
2

,operating_hour,n_observations
0,50,179360
1,100,179360
2,150,179360
3,200,179360
4,250,179360
5,300,179360
6,350,179360
7,400,179360
8,450,179360
9,500,179360


13.2.2 LOADING CHECK COMPLETE

Frozen XGBoost model and feature-engineered data are available.
No SHAP explainer has been created and no SHAP values have been
calculated at this stage.


In [ ]:
### 13.2.3 Verify and Recover the Frozen Predictor Order

SHAP explanations must use the same predictor identities and ordering as the fitted XGBoost model.

The 20 predictors reconstructed from the feature-selection stage are first compared with the feature metadata stored within the frozen XGBoost model. Predictor membership and predictor order are assessed separately because a selected-feature list may contain the correct variables without preserving the exact DataFrame column sequence used during final model fitting.

Where predictor membership is identical but ordering differs, the feature order stored within the frozen XGBoost model is treated as the authoritative training order. This does not alter or retrain the model; it reconstructs the exact input structure already used during model development.

The resulting model-derived predictor sequence is frozen for all subsequent prediction and SHAP calculations.

In [7]:
# ============================================================
# 13.2.3 Verify and Recover the Frozen Predictor Order
# ============================================================

# ------------------------------------------------------------
# 1. Retrieve feature information from frozen XGBoost model
# ------------------------------------------------------------

model_feature_names = None

# Preferred source: scikit-learn XGBRegressor metadata
if hasattr(final_xgb_model, "feature_names_in_"):
    model_feature_names = list(final_xgb_model.feature_names_in_)

# Secondary source: underlying XGBoost Booster
if model_feature_names is None:
    booster_feature_names = final_xgb_model.get_booster().feature_names

    if booster_feature_names is not None:
        model_feature_names = list(booster_feature_names)


if model_feature_names is None:
    raise ValueError(
        "Feature names could not be recovered from the frozen XGBoost model. "
        "The SHAP analysis should not proceed until predictor identity and "
        "order can be verified."
    )


# ------------------------------------------------------------
# 2. Basic dimensional comparison
# ------------------------------------------------------------

print("=" * 70)
print("FROZEN XGBOOST PREDICTOR VERIFICATION")
print("=" * 70)

print(f"\nPredictors reconstructed from Notebook 12 : {len(selected_features)}")
print(f"Predictors stored in frozen model          : {len(model_feature_names)}")


# ------------------------------------------------------------
# 3. Compare predictor membership
# ------------------------------------------------------------

missing_from_model = [
    feature
    for feature in selected_features
    if feature not in model_feature_names
]

unexpected_in_model = [
    feature
    for feature in model_feature_names
    if feature not in selected_features
]

same_feature_count = (
    len(selected_features)
    == len(model_feature_names)
)

same_feature_membership = (
    len(missing_from_model) == 0
    and len(unexpected_in_model) == 0
)


print("\nPredictor membership check:")
print(f"  Same predictor count      : {same_feature_count}")
print(f"  Missing from frozen model : {len(missing_from_model)}")
print(f"  Unexpected in frozen model: {len(unexpected_in_model)}")


if missing_from_model:
    print("\nMissing predictors:")
    for feature in missing_from_model:
        print(f"  - {feature}")

if unexpected_in_model:
    print("\nUnexpected predictors:")
    for feature in unexpected_in_model:
        print(f"  - {feature}")


# Membership must match before proceeding
if not (same_feature_count and same_feature_membership):
    raise ValueError(
        "The reconstructed predictor set does not match the predictor "
        "membership stored in the frozen XGBoost model."
    )


# ------------------------------------------------------------
# 4. Compare reconstructed order with model training order
# ------------------------------------------------------------

feature_order_check = pd.DataFrame(
    {
        "Position": np.arange(1, len(model_feature_names) + 1),
        "Reconstructed_Feature": selected_features,
        "Frozen_Model_Feature": model_feature_names,
    }
)

feature_order_check["Exact_Match"] = (
    feature_order_check["Reconstructed_Feature"]
    == feature_order_check["Frozen_Model_Feature"]
)

exact_feature_order = feature_order_check["Exact_Match"].all()


print("\n" + "-" * 70)
print("POSITIONAL FEATURE-ORDER CHECK")
print("-" * 70)

display(feature_order_check)


# ------------------------------------------------------------
# 5. Establish authoritative frozen predictor order
# ------------------------------------------------------------

FROZEN_FEATURES = model_feature_names.copy()

print("=" * 70)
print("PREDICTOR INTEGRITY SUMMARY")
print("=" * 70)

print(f"\nSame predictor count      : {same_feature_count}")
print(f"Same predictor membership : {same_feature_membership}")
print(f"Same reconstructed order  : {exact_feature_order}")


if not exact_feature_order:
    print(
        "\nNOTE:"
        "\nThe reconstructed feature list contains the correct 20 predictors, "
        "but its order differs from the sequence stored in the fitted model."
        "\nThe model-stored feature sequence is therefore adopted as the "
        "authoritative frozen training order."
    )


print("\nAuthoritative frozen predictor order:")

for i, feature in enumerate(FROZEN_FEATURES, start=1):
    print(f"{i:>2}. {feature}")


# ------------------------------------------------------------
# 6. Verify frozen features exist in current dataset
# ------------------------------------------------------------

missing_from_dataset = [
    feature
    for feature in FROZEN_FEATURES
    if feature not in pemfc_data.columns
]

if missing_from_dataset:
    raise KeyError(
        "One or more frozen XGBoost predictors are missing from "
        "the feature-engineered dataset."
    )


# ------------------------------------------------------------
# 7. Final safeguard
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VERIFICATION PASSED")
print("=" * 70)

print(
    "\nPredictor membership exactly matches the frozen XGBoost model."
)

print(
    "The feature order stored within the fitted model has been recovered "
    "and is now frozen for all subsequent prediction and SHAP calculations."
)

FROZEN XGBOOST PREDICTOR VERIFICATION

Predictors reconstructed from Notebook 12 : 20
Predictors stored in frozen model          : 20

Predictor membership check:
  Same predictor count      : True
  Missing from frozen model : 0
  Unexpected in frozen model: 0

----------------------------------------------------------------------
POSITIONAL FEATURE-ORDER CHECK
----------------------------------------------------------------------


,Position,Reconstructed_Feature,Frozen_Model_Feature,Exact_Match
0,1,anode_pressure_diff,current,False
1,2,anode_temp_diff,pressure_anode_inlet,False
2,3,cathode_pressure_diff,pressure_anode_outlet,False
3,4,current,pressure_cathode_inlet,False
4,5,pressure_anode_outlet,pressure_cathode_outlet,False
5,6,pressure_cathode_outlet,temp_anode_endplate,False
6,7,temp_anode_dewpoint_water,temp_anode_dewpoint_water,True
7,8,temp_anode_endplate,temp_anode_inlet,False
8,9,temp_cathode_inlet,temp_anode_outlet,False
9,10,total_anode_stack_flow,temp_cathode_dewpoint_water,False


PREDICTOR INTEGRITY SUMMARY

Same predictor count      : True
Same predictor membership : True
Same reconstructed order  : False

NOTE:
The reconstructed feature list contains the correct 20 predictors, but its order differs from the sequence stored in the fitted model.
The model-stored feature sequence is therefore adopted as the authoritative frozen training order.

Authoritative frozen predictor order:
 1. current
 2. pressure_anode_inlet
 3. pressure_anode_outlet
 4. pressure_cathode_inlet
 5. pressure_cathode_outlet
 6. temp_anode_endplate
 7. temp_anode_dewpoint_water
 8. temp_anode_inlet
 9. temp_anode_outlet
10. temp_cathode_dewpoint_water
11. temp_cathode_inlet
12. temp_cathode_outlet
13. total_anode_stack_flow
14. total_cathode_stack_flow
15. anode_pressure_diff
16. cathode_pressure_diff
17. anode_temp_diff
18. cathode_temp_diff
19. anode_dewpoint_offset
20. cathode_dewpoint_offset

VERIFICATION PASSED

Predictor membership exactly matches the frozen XGBoost model.
The featur

In [ ]:
## 13.3 Reconstruct and Verify the Development and Later-Stage Holdout Regimes

This section reconstructs the development and later-stage holdout regimes using the same temporal boundary applied during model development.

The data are partitioned as follows:

- Development regime: durability stages 50–850 h
- Later-stage holdout regime: durability stages 900, 950, and 1000 h

The development regime will later be used only as the source of candidate SHAP background/reference observations.

The later-stage holdout regime will provide the primary SHAP explanation observations because the dissertation's main predictive objective concerns generalisation to unseen later durability stages.

For both regimes, predictor matrices are reconstructed using the authoritative feature order recovered directly from the frozen XGBoost model.

No sampling or SHAP computation is performed in this section. The objective is to verify that the temporal partitions, predictor matrices, target vectors, and stage counts exactly reproduce the intended modelling design.

In [8]:
# ============================================================
# 13.3 Reconstruct and Verify Development and Holdout Regimes
# ============================================================

# ------------------------------------------------------------
# 1. Reconstruct development and holdout datasets
# ------------------------------------------------------------

development_data = (
    pemfc_data[
        pemfc_data[STAGE_COLUMN].isin(DEVELOPMENT_STAGES)
    ]
    .copy()
)

holdout_data = (
    pemfc_data[
        pemfc_data[STAGE_COLUMN].isin(HOLDOUT_STAGES)
    ]
    .copy()
)


# ------------------------------------------------------------
# 2. Construct predictor matrices using frozen model order
# ------------------------------------------------------------

X_development = development_data[FROZEN_FEATURES].copy()
y_development = development_data[TARGET].copy()

X_holdout = holdout_data[FROZEN_FEATURES].copy()
y_holdout = holdout_data[TARGET].copy()


# ------------------------------------------------------------
# 3. Display regime dimensions
# ------------------------------------------------------------

print("=" * 70)
print("TEMPORAL REGIME RECONSTRUCTION")
print("=" * 70)

print("\nDevelopment regime:")
print(f"  Durability stages : {DEVELOPMENT_STAGES[0]}–{DEVELOPMENT_STAGES[-1]} h")
print(f"  Observations      : {len(development_data):,}")
print(f"  Predictors        : {X_development.shape[1]}")

print("\nLater-stage holdout regime:")
print(f"  Durability stages : {HOLDOUT_STAGES}")
print(f"  Observations      : {len(holdout_data):,}")
print(f"  Predictors        : {X_holdout.shape[1]}")


# ------------------------------------------------------------
# 4. Stage-level observation counts
# ------------------------------------------------------------

development_stage_counts = (
    development_data
    .groupby(STAGE_COLUMN)
    .size()
    .rename("n_observations")
    .reset_index()
)

holdout_stage_counts = (
    holdout_data
    .groupby(STAGE_COLUMN)
    .size()
    .rename("n_observations")
    .reset_index()
)


print("\n" + "-" * 70)
print("DEVELOPMENT STAGE COUNTS")
print("-" * 70)

display(development_stage_counts)


print("\n" + "-" * 70)
print("HOLDOUT STAGE COUNTS")
print("-" * 70)

display(holdout_stage_counts)


# ------------------------------------------------------------
# 5. Verify predictor order
# ------------------------------------------------------------

development_order_match = (
    X_development.columns.tolist() == FROZEN_FEATURES
)

holdout_order_match = (
    X_holdout.columns.tolist() == FROZEN_FEATURES
)


# ------------------------------------------------------------
# 6. Verify missing values
# ------------------------------------------------------------

development_missing = int(
    X_development.isna().sum().sum()
)

holdout_missing = int(
    X_holdout.isna().sum().sum()
)

target_development_missing = int(
    y_development.isna().sum()
)

target_holdout_missing = int(
    y_holdout.isna().sum()
)


# ------------------------------------------------------------
# 7. Integrity summary
# ------------------------------------------------------------

print("=" * 70)
print("TEMPORAL PARTITION INTEGRITY SUMMARY")
print("=" * 70)

print(f"\nDevelopment predictor order correct : {development_order_match}")
print(f"Holdout predictor order correct     : {holdout_order_match}")

print(f"\nMissing predictor values:")
print(f"  Development : {development_missing:,}")
print(f"  Holdout     : {holdout_missing:,}")

print(f"\nMissing target values:")
print(f"  Development : {target_development_missing:,}")
print(f"  Holdout     : {target_holdout_missing:,}")


# ------------------------------------------------------------
# 8. Expected observation-count checks
# ------------------------------------------------------------

expected_development_rows = sum(
    int(
        (pemfc_data[STAGE_COLUMN] == stage).sum()
    )
    for stage in DEVELOPMENT_STAGES
)

expected_holdout_rows = sum(
    int(
        (pemfc_data[STAGE_COLUMN] == stage).sum()
    )
    for stage in HOLDOUT_STAGES
)

development_count_match = (
    len(development_data) == expected_development_rows
)

holdout_count_match = (
    len(holdout_data) == expected_holdout_rows
)


print(f"\nDevelopment row-count check : {development_count_match}")
print(f"Holdout row-count check     : {holdout_count_match}")


# ------------------------------------------------------------
# 9. Final safeguard
# ------------------------------------------------------------

all_checks_pass = all(
    [
        development_order_match,
        holdout_order_match,
        development_missing == 0,
        holdout_missing == 0,
        target_development_missing == 0,
        target_holdout_missing == 0,
        development_count_match,
        holdout_count_match,
    ]
)

if not all_checks_pass:
    raise ValueError(
        "One or more development/holdout reconstruction checks failed."
    )


print("\n" + "=" * 70)
print("VERIFICATION PASSED")
print("=" * 70)

print(
    "\nDevelopment and later-stage holdout regimes have been "
    "reconstructed successfully using the frozen predictor order."
)

print(
    "These partitions are now fixed for subsequent SHAP background "
    "and explanation-sampling procedures."
)

TEMPORAL REGIME RECONSTRUCTION

Development regime:
  Durability stages : 50–850 h
  Observations      : 3,049,120
  Predictors        : 20

Later-stage holdout regime:
  Durability stages : [900, 950, 1000]
  Observations      : 580,560
  Predictors        : 20

----------------------------------------------------------------------
DEVELOPMENT STAGE COUNTS
----------------------------------------------------------------------


,operating_hour,n_observations
0,50,179360
1,100,179360
2,150,179360
3,200,179360
4,250,179360
5,300,179360
6,350,179360
7,400,179360
8,450,179360
9,500,179360



----------------------------------------------------------------------
HOLDOUT STAGE COUNTS
----------------------------------------------------------------------


,operating_hour,n_observations
0,900,179360
1,950,179360
2,1000,221840


TEMPORAL PARTITION INTEGRITY SUMMARY

Development predictor order correct : True
Holdout predictor order correct     : True

Missing predictor values:
  Development : 0
  Holdout     : 0

Missing target values:
  Development : 0
  Holdout     : 0

Development row-count check : True
Holdout row-count check     : True

VERIFICATION PASSED

Development and later-stage holdout regimes have been reconstructed successfully using the frozen predictor order.
These partitions are now fixed for subsequent SHAP background and explanation-sampling procedures.


In [ ]:
## 13.4 Verify Frozen XGBoost Prediction Reproducibility

Before computing SHAP values, the frozen XGBoost model is evaluated on the reconstructed 900–1000 h holdout to verify that the loaded model and reconstructed predictor matrix reproduce the predictive performance reported during model development.

This is a reproducibility safeguard rather than a new model-evaluation stage. The model is not refitted, tuned, or re-selected.

The following metrics are recalculated:

- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)
- Coefficient of determination (R²)

Metrics are calculated for:

- the complete 900–1000 h holdout;
- 900 h;
- 950 h;
- 1000 h.

The reproduced values should closely match the previously reported final XGBoost holdout results. Any meaningful discrepancy would indicate a mismatch in the loaded model, predictor ordering, or reconstructed dataset and would need to be resolved before SHAP analysis proceeds.

In [9]:
# ============================================================
# 13.4 Verify Frozen XGBoost Prediction Reproducibility
# ============================================================

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)


# ------------------------------------------------------------
# 1. Generate predictions using the frozen XGBoost model
# ------------------------------------------------------------

print("=" * 70)
print("FROZEN XGBOOST HOLDOUT REPRODUCIBILITY CHECK")
print("=" * 70)

holdout_predictions = final_xgb_model.predict(X_holdout)


# ------------------------------------------------------------
# 2. Overall holdout metrics
# ------------------------------------------------------------

overall_rmse = np.sqrt(
    mean_squared_error(y_holdout, holdout_predictions)
)

overall_mae = mean_absolute_error(
    y_holdout,
    holdout_predictions
)

overall_r2 = r2_score(
    y_holdout,
    holdout_predictions
)


print("\nOverall 900–1000 h holdout performance:")

print(f"  RMSE : {overall_rmse:.6f} V")
print(f"         {overall_rmse * 1000:.3f} mV")

print(f"  MAE  : {overall_mae:.6f} V")
print(f"         {overall_mae * 1000:.3f} mV")

print(f"  R²   : {overall_r2:.6f}")


# ------------------------------------------------------------
# 3. Stage-wise holdout metrics
# ------------------------------------------------------------

holdout_evaluation = holdout_data[
    [STAGE_COLUMN, TARGET]
].copy()

holdout_evaluation["prediction"] = holdout_predictions
holdout_evaluation["error"] = (
    holdout_evaluation[TARGET]
    - holdout_evaluation["prediction"]
)

holdout_evaluation["absolute_error"] = (
    holdout_evaluation["error"].abs()
)


stage_metric_rows = []

for stage in HOLDOUT_STAGES:

    stage_mask = (
        holdout_evaluation[STAGE_COLUMN] == stage
    )

    y_stage = holdout_evaluation.loc[
        stage_mask,
        TARGET
    ]

    pred_stage = holdout_evaluation.loc[
        stage_mask,
        "prediction"
    ]

    stage_rmse = np.sqrt(
        mean_squared_error(
            y_stage,
            pred_stage
        )
    )

    stage_mae = mean_absolute_error(
        y_stage,
        pred_stage
    )

    stage_r2 = r2_score(
        y_stage,
        pred_stage
    )

    stage_metric_rows.append(
        {
            "operating_hour": stage,
            "n_observations": int(stage_mask.sum()),
            "RMSE_V": stage_rmse,
            "RMSE_mV": stage_rmse * 1000,
            "MAE_V": stage_mae,
            "MAE_mV": stage_mae * 1000,
            "R2": stage_r2,
        }
    )


stage_metrics = pd.DataFrame(stage_metric_rows)


print("\n" + "-" * 70)
print("STAGE-WISE HOLDOUT PERFORMANCE")
print("-" * 70)

display(stage_metrics)


# ------------------------------------------------------------
# 4. Compare against previously reported Notebook 12 results
# ------------------------------------------------------------

reported_xgb_results = {
    "RMSE_V": 0.008788,
    "MAE_V": 0.006357,
    "R2": 0.991834,
}

reported_stage_rmse_mV = {
    900: 6.034,
    950: 8.012,
    1000: 10.990,
}


print("=" * 70)
print("COMPARISON WITH PREVIOUSLY REPORTED RESULTS")
print("=" * 70)


comparison = pd.DataFrame(
    {
        "Metric": [
            "RMSE_V",
            "MAE_V",
            "R2",
        ],
        "Previously_Reported": [
            reported_xgb_results["RMSE_V"],
            reported_xgb_results["MAE_V"],
            reported_xgb_results["R2"],
        ],
        "Reproduced": [
            overall_rmse,
            overall_mae,
            overall_r2,
        ],
    }
)

comparison["Absolute_Difference"] = (
    comparison["Reproduced"]
    - comparison["Previously_Reported"]
).abs()

display(comparison)


stage_rmse_comparison = stage_metrics[
    ["operating_hour", "RMSE_mV"]
].copy()

stage_rmse_comparison[
    "Previously_Reported_RMSE_mV"
] = stage_rmse_comparison[
    "operating_hour"
].map(reported_stage_rmse_mV)

stage_rmse_comparison[
    "Absolute_Difference_mV"
] = (
    stage_rmse_comparison["RMSE_mV"]
    - stage_rmse_comparison[
        "Previously_Reported_RMSE_mV"
    ]
).abs()


print("\nStage-wise RMSE comparison:")

display(stage_rmse_comparison)


# ------------------------------------------------------------
# 5. Reproducibility tolerance
# ------------------------------------------------------------

METRIC_TOLERANCE = 1e-5
STAGE_RMSE_TOLERANCE_MV = 0.02


overall_reproduced = bool(
    (
        comparison["Absolute_Difference"]
        <= METRIC_TOLERANCE
    ).all()
)

stage_reproduced = bool(
    (
        stage_rmse_comparison[
            "Absolute_Difference_mV"
        ]
        <= STAGE_RMSE_TOLERANCE_MV
    ).all()
)


print("=" * 70)
print("REPRODUCIBILITY SUMMARY")
print("=" * 70)

print(
    f"\nOverall metrics reproduced within tolerance : "
    f"{overall_reproduced}"
)

print(
    f"Stage RMSE values reproduced within tolerance: "
    f"{stage_reproduced}"
)


if not (
    overall_reproduced
    and stage_reproduced
):
    raise ValueError(
        "Frozen XGBoost predictions do not reproduce the "
        "previously reported Notebook 12 holdout performance."
    )


print("\nVERIFICATION PASSED")

print(
    "The loaded frozen XGBoost model reproduces the previously "
    "reported later-stage holdout performance."
)

print(
    "The model and reconstructed holdout predictor matrix are "
    "therefore suitable for subsequent SHAP analysis."
)

FROZEN XGBOOST HOLDOUT REPRODUCIBILITY CHECK

Overall 900–1000 h holdout performance:
  RMSE : 0.008788 V
         8.788 mV
  MAE  : 0.006357 V
         6.357 mV
  R²   : 0.991834

----------------------------------------------------------------------
STAGE-WISE HOLDOUT PERFORMANCE
----------------------------------------------------------------------


,operating_hour,n_observations,RMSE_V,RMSE_mV,MAE_V,MAE_mV,R2
0,900,179360,0.006034,6.034272,0.004596,4.595675,0.996210
1,950,179360,0.008012,8.011713,0.005976,5.975518,0.993094
2,1000,221840,0.010990,10.990396,0.008089,8.088791,0.987238


COMPARISON WITH PREVIOUSLY REPORTED RESULTS


,Metric,Previously_Reported,Reproduced,Absolute_Difference
0,RMSE_V,0.008788,0.008788,0.000000
1,MAE_V,0.006357,0.006357,0.000000
2,R2,0.991834,0.991834,0.000000



Stage-wise RMSE comparison:


,operating_hour,RMSE_mV,Previously_Reported_RMSE_mV,Absolute_Difference_mV
0,900,6.034272,6.034000,0.000272
1,950,8.011713,8.012000,0.000287
2,1000,10.990396,10.990000,0.000396


REPRODUCIBILITY SUMMARY

Overall metrics reproduced within tolerance : True
Stage RMSE values reproduced within tolerance: True

VERIFICATION PASSED
The loaded frozen XGBoost model reproduces the previously reported later-stage holdout performance.
The model and reconstructed holdout predictor matrix are therefore suitable for subsequent SHAP analysis.


In [ ]:
## 13.5 Development Background Sampling Design

Interventional TreeSHAP requires an explicit background/reference dataset representing the distribution against which model predictions are explained.

For the primary analysis, background observations are drawn exclusively from the 50–850 h development regime. This preserves the temporal interpretation of the explainability framework: predictions at unseen later durability stages are explained relative to a representative reference drawn from the earlier model-development regime.

Because the development dataset contains more than three million temporally dense observations, using every development observation as the TreeSHAP background is neither computationally necessary nor desirable. A representative subset will therefore be constructed.

A stage-balanced sampling strategy is adopted so that each development durability stage contributes equally to the background rather than allowing the reference distribution to be determined solely by the number of observations available at each stage. Within each stage, observations are sampled randomly without deliberately balancing Current/load or other operating variables, thereby preserving their natural within-stage distributions.

Stage balancing is a study-specific analytical design choice motivated by the scientific importance of durability stage; it is not treated as a universal SHAP requirement.

The exact background sample size is not fixed a priori. Candidate background sizes will subsequently be evaluated empirically by examining the stability of the SHAP expected value, global mean absolute SHAP values, and feature rankings. The final background will be selected only after this stability assessment.

In [11]:
# ============================================================
# 13.5 Development Background Sampling Design
# ============================================================

# ------------------------------------------------------------
# 13.5.1 Stage-Balanced Background Sampling Function
# ------------------------------------------------------------

def sample_stage_balanced(
    data,
    stages,
    total_n,
    stage_column,
    random_seed=42,
):
    """
    Draw an approximately equal number of observations from each
    specified durability stage.

    If total_n is not exactly divisible by the number of stages,
    the remainder is distributed deterministically across the
    earliest stages in the supplied stage list.

    Sampling is random without replacement within each stage.
    """

    stages = list(stages)
    n_stages = len(stages)

    if total_n < n_stages:
        raise ValueError(
            "total_n must be at least equal to the number of stages."
        )

    base_n = total_n // n_stages
    remainder = total_n % n_stages

    sampled_parts = []
    allocation_records = []

    for position, stage in enumerate(stages):

        stage_n = base_n + (
            1 if position < remainder else 0
        )

        stage_data = data[
            data[stage_column] == stage
        ]

        if stage_n > len(stage_data):
            raise ValueError(
                f"Requested {stage_n} observations from stage {stage}, "
                f"but only {len(stage_data)} are available."
            )

        # Stage-specific deterministic seed prevents every stage
        # from using an identical pseudo-random sequence.
        stage_seed = int(
            random_seed + position
        )

        stage_sample = stage_data.sample(
            n=stage_n,
            replace=False,
            random_state=stage_seed,
        )

        sampled_parts.append(stage_sample)

        allocation_records.append(
            {
                stage_column: stage,
                "sample_n": stage_n,
            }
        )

    sampled_data = pd.concat(
        sampled_parts,
        axis=0,
    )

    # Shuffle the combined sample reproducibly while retaining
    # original dataframe indices for traceability.
    sampled_data = sampled_data.sample(
        frac=1.0,
        random_state=random_seed,
    )

    allocation = pd.DataFrame(
        allocation_records
    )

    return sampled_data, allocation

In [12]:
# ============================================================
# 13.5.2 Validate Stage-Balanced Background Sampling
# ============================================================

DIAGNOSTIC_BACKGROUND_N = 500


diagnostic_background, diagnostic_allocation = (
    sample_stage_balanced(
        data=development_data,
        stages=DEVELOPMENT_STAGES,
        total_n=DIAGNOSTIC_BACKGROUND_N,
        stage_column=STAGE_COLUMN,
        random_seed=RANDOM_SEED,
    )
)


print("=" * 70)
print("STAGE-BALANCED BACKGROUND SAMPLING CHECK")
print("=" * 70)

print(
    f"\nRequested diagnostic background size : "
    f"{DIAGNOSTIC_BACKGROUND_N:,}"
)

print(
    f"Returned diagnostic background size  : "
    f"{len(diagnostic_background):,}"
)

print(
    f"Number of development stages          : "
    f"{len(DEVELOPMENT_STAGES)}"
)


print("\n" + "-" * 70)
print("STAGE ALLOCATION")
print("-" * 70)

display(diagnostic_allocation)


# ------------------------------------------------------------
# Verify realised allocation
# ------------------------------------------------------------

realised_allocation = (
    diagnostic_background
    .groupby(STAGE_COLUMN)
    .size()
    .rename("realised_n")
    .reset_index()
)

allocation_check = diagnostic_allocation.merge(
    realised_allocation,
    on=STAGE_COLUMN,
    how="left",
)

allocation_check["Exact_Match"] = (
    allocation_check["sample_n"]
    == allocation_check["realised_n"]
)


print("\n" + "-" * 70)
print("ALLOCATION VERIFICATION")
print("-" * 70)

display(allocation_check)


# ------------------------------------------------------------
# Sampling integrity checks
# ------------------------------------------------------------

correct_total = (
    len(diagnostic_background)
    == DIAGNOSTIC_BACKGROUND_N
)

all_stages_present = (
    set(diagnostic_background[STAGE_COLUMN].unique())
    == set(DEVELOPMENT_STAGES)
)

allocation_correct = (
    allocation_check["Exact_Match"].all()
)

duplicate_original_indices = (
    diagnostic_background.index.duplicated().sum()
)


print("=" * 70)
print("SAMPLING INTEGRITY SUMMARY")
print("=" * 70)

print(f"\nCorrect total sample size : {correct_total}")
print(f"All development stages    : {all_stages_present}")
print(f"Allocation reproduced     : {allocation_correct}")

print(
    f"Duplicate original indices: "
    f"{duplicate_original_indices:,}"
)


if not (
    correct_total
    and all_stages_present
    and allocation_correct
    and duplicate_original_indices == 0
):
    raise ValueError(
        "Stage-balanced background sampling validation failed."
    )


print("\nVERIFICATION PASSED")

print(
    "The stage-balanced sampling function produces a reproducible "
    "development-background candidate with approximately equal "
    "representation across durability stages."
)

print(
    "\nThe 500-row sample is diagnostic only and has NOT been "
    "selected as the final SHAP background."
)

STAGE-BALANCED BACKGROUND SAMPLING CHECK

Requested diagnostic background size : 500
Returned diagnostic background size  : 500
Number of development stages          : 17

----------------------------------------------------------------------
STAGE ALLOCATION
----------------------------------------------------------------------


,operating_hour,sample_n
0,50,30
1,100,30
2,150,30
3,200,30
4,250,30
5,300,30
6,350,30
7,400,29
8,450,29
9,500,29



----------------------------------------------------------------------
ALLOCATION VERIFICATION
----------------------------------------------------------------------


,operating_hour,sample_n,realised_n,Exact_Match
0,50,30,30,True
1,100,30,30,True
2,150,30,30,True
3,200,30,30,True
4,250,30,30,True
5,300,30,30,True
6,350,30,30,True
7,400,29,29,True
8,450,29,29,True
9,500,29,29,True


SAMPLING INTEGRITY SUMMARY

Correct total sample size : True
All development stages    : True
Allocation reproduced     : True
Duplicate original indices: 0

VERIFICATION PASSED
The stage-balanced sampling function produces a reproducible development-background candidate with approximately equal representation across durability stages.

The 500-row sample is diagnostic only and has NOT been selected as the final SHAP background.


In [ ]:
## 13.6 Background Sample-Size Stability Analysis

The exact number of development observations required for the interventional TreeSHAP background is not fixed by a universal methodological standard. Background size is therefore determined empirically.

The purpose of this analysis is to evaluate whether substantive SHAP conclusions remain stable as the number of background/reference observations increases.

Candidate background sizes of 100, 250, 500, and 1,000 observations are evaluated. Each candidate is sampled from the 50–850 h development regime using the stage-balanced procedure established in Section 13.5.

To isolate the effect of background size, all candidate backgrounds are evaluated against the same fixed diagnostic explanation sample from the 900–1000 h holdout. The explanation observations are therefore held constant while only the development-background size changes.

Background stability will subsequently be assessed using:

- the TreeSHAP expected value;
- global mean absolute SHAP magnitude for each predictor;
- complete feature-ranking agreement;
- top-feature overlap;
- and rank correlation between candidate backgrounds.

The candidate sizes are study-specific diagnostic values rather than universal SHAP thresholds. The objective is to determine whether increasing the development background materially changes the substantive explanation produced by the frozen XGBoost model.

In [13]:
# ============================================================
# 13.6 Background Sample-Size Stability Analysis
# ============================================================

# ------------------------------------------------------------
# 13.6.1 Construct Fixed Diagnostic Explanation Sample
# ------------------------------------------------------------

DIAGNOSTIC_EXPLANATION_N = 900


diagnostic_explanation, diagnostic_explanation_allocation = (
    sample_stage_balanced(
        data=holdout_data,
        stages=HOLDOUT_STAGES,
        total_n=DIAGNOSTIC_EXPLANATION_N,
        stage_column=STAGE_COLUMN,
        random_seed=RANDOM_SEED,
    )
)


# Predictor matrix must follow frozen XGBoost feature order
X_diagnostic_explanation = (
    diagnostic_explanation[FROZEN_FEATURES]
    .copy()
)


print("=" * 70)
print("FIXED DIAGNOSTIC EXPLANATION SAMPLE")
print("=" * 70)

print(
    f"\nRequested diagnostic explanation size : "
    f"{DIAGNOSTIC_EXPLANATION_N:,}"
)

print(
    f"Returned diagnostic explanation size  : "
    f"{len(diagnostic_explanation):,}"
)

print(
    f"Number of holdout stages               : "
    f"{len(HOLDOUT_STAGES)}"
)


print("\n" + "-" * 70)
print("HOLDOUT STAGE ALLOCATION")
print("-" * 70)

display(diagnostic_explanation_allocation)


# ------------------------------------------------------------
# Verify realised allocation
# ------------------------------------------------------------

diagnostic_realised_allocation = (
    diagnostic_explanation
    .groupby(STAGE_COLUMN)
    .size()
    .rename("realised_n")
    .reset_index()
)

diagnostic_allocation_check = (
    diagnostic_explanation_allocation.merge(
        diagnostic_realised_allocation,
        on=STAGE_COLUMN,
        how="left",
    )
)

diagnostic_allocation_check["Exact_Match"] = (
    diagnostic_allocation_check["sample_n"]
    == diagnostic_allocation_check["realised_n"]
)


print("\n" + "-" * 70)
print("ALLOCATION VERIFICATION")
print("-" * 70)

display(diagnostic_allocation_check)


# ------------------------------------------------------------
# Predictor and sampling integrity checks
# ------------------------------------------------------------

correct_diagnostic_total = (
    len(diagnostic_explanation)
    == DIAGNOSTIC_EXPLANATION_N
)

all_holdout_stages_present = (
    set(
        diagnostic_explanation[
            STAGE_COLUMN
        ].unique()
    )
    == set(HOLDOUT_STAGES)
)

diagnostic_allocation_correct = (
    diagnostic_allocation_check[
        "Exact_Match"
    ].all()
)

diagnostic_order_correct = (
    X_diagnostic_explanation.columns.tolist()
    == FROZEN_FEATURES
)

diagnostic_missing = int(
    X_diagnostic_explanation
    .isna()
    .sum()
    .sum()
)

diagnostic_duplicate_indices = int(
    diagnostic_explanation
    .index
    .duplicated()
    .sum()
)


print("=" * 70)
print("DIAGNOSTIC SAMPLE INTEGRITY SUMMARY")
print("=" * 70)

print(
    f"\nCorrect total sample size : "
    f"{correct_diagnostic_total}"
)

print(
    f"All holdout stages present: "
    f"{all_holdout_stages_present}"
)

print(
    f"Allocation reproduced     : "
    f"{diagnostic_allocation_correct}"
)

print(
    f"Frozen predictor order    : "
    f"{diagnostic_order_correct}"
)

print(
    f"Missing predictor values  : "
    f"{diagnostic_missing:,}"
)

print(
    f"Duplicate original indices: "
    f"{diagnostic_duplicate_indices:,}"
)


if not all(
    [
        correct_diagnostic_total,
        all_holdout_stages_present,
        diagnostic_allocation_correct,
        diagnostic_order_correct,
        diagnostic_missing == 0,
        diagnostic_duplicate_indices == 0,
    ]
):
    raise ValueError(
        "Diagnostic explanation-sample validation failed."
    )


print("\nVERIFICATION PASSED")

print(
    "A fixed stage-balanced diagnostic explanation sample has "
    "been constructed successfully."
)

print(
    "\nThis same set of holdout observations will be used for every "
    "candidate background size so that background size is the "
    "quantity being varied."
)

print(
    "\nThe 900-row diagnostic sample is NOT the final primary "
    "SHAP explanation sample."
)

FIXED DIAGNOSTIC EXPLANATION SAMPLE

Requested diagnostic explanation size : 900
Returned diagnostic explanation size  : 900
Number of holdout stages               : 3

----------------------------------------------------------------------
HOLDOUT STAGE ALLOCATION
----------------------------------------------------------------------


,operating_hour,sample_n
0,900,300
1,950,300
2,1000,300



----------------------------------------------------------------------
ALLOCATION VERIFICATION
----------------------------------------------------------------------


,operating_hour,sample_n,realised_n,Exact_Match
0,900,300,300,True
1,950,300,300,True
2,1000,300,300,True


DIAGNOSTIC SAMPLE INTEGRITY SUMMARY

Correct total sample size : True
All holdout stages present: True
Allocation reproduced     : True
Frozen predictor order    : True
Missing predictor values  : 0
Duplicate original indices: 0

VERIFICATION PASSED
A fixed stage-balanced diagnostic explanation sample has been constructed successfully.

This same set of holdout observations will be used for every candidate background size so that background size is the quantity being varied.

The 900-row diagnostic sample is NOT the final primary SHAP explanation sample.


In [ ]:
### 13.6.2 Construct Candidate Development Backgrounds

Four candidate background sizes are constructed from the 50–850 h development regime: 100, 250, 500, and 1,000 observations.

Each candidate uses the same reproducible stage-balanced sampling procedure. Because the candidate totals are not necessarily divisible by the 17 development stages, observations are allocated as evenly as mathematically possible, with a maximum difference of one observation between stages.

Only background sample size is varied. The source regime, sampling strategy, random seed, predictor set, predictor order, and subsequently explained holdout observations remain fixed.

These candidate backgrounds are diagnostic objects used to evaluate background-size sensitivity. None is designated as the final SHAP background at this stage.

In [14]:
# ============================================================
# 13.6.2 Construct Candidate Development Backgrounds
# ============================================================

BACKGROUND_CANDIDATE_SIZES = [
    100,
    250,
    500,
    1000,
]


candidate_backgrounds = {}
candidate_background_allocations = {}

background_summary_rows = []


print("=" * 70)
print("CANDIDATE DEVELOPMENT BACKGROUND CONSTRUCTION")
print("=" * 70)


# ------------------------------------------------------------
# 1. Construct each candidate background
# ------------------------------------------------------------

for background_n in BACKGROUND_CANDIDATE_SIZES:

    sampled_background, allocation = (
        sample_stage_balanced(
            data=development_data,
            stages=DEVELOPMENT_STAGES,
            total_n=background_n,
            stage_column=STAGE_COLUMN,
            random_seed=RANDOM_SEED,
        )
    )

    # Retain complete sampled rows for traceability
    candidate_backgrounds[
        background_n
    ] = sampled_background

    candidate_background_allocations[
        background_n
    ] = allocation


    # --------------------------------------------------------
    # Predictor matrix in exact frozen model order
    # --------------------------------------------------------

    X_background_candidate = (
        sampled_background[
            FROZEN_FEATURES
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    actual_n = len(sampled_background)

    stage_counts = (
        sampled_background
        .groupby(STAGE_COLUMN)
        .size()
    )

    all_stages_present = (
        set(stage_counts.index.tolist())
        == set(DEVELOPMENT_STAGES)
    )

    min_stage_n = int(
        stage_counts.min()
    )

    max_stage_n = int(
        stage_counts.max()
    )

    max_stage_difference = (
        max_stage_n - min_stage_n
    )

    order_correct = (
        X_background_candidate
        .columns
        .tolist()
        == FROZEN_FEATURES
    )

    missing_values = int(
        X_background_candidate
        .isna()
        .sum()
        .sum()
    )

    duplicate_indices = int(
        sampled_background
        .index
        .duplicated()
        .sum()
    )


    background_summary_rows.append(
        {
            "Background_N": background_n,
            "Actual_N": actual_n,
            "Stages_Present": len(stage_counts),
            "Min_Per_Stage": min_stage_n,
            "Max_Per_Stage": max_stage_n,
            "Max_Stage_Difference": max_stage_difference,
            "All_Stages_Present": all_stages_present,
            "Frozen_Order": order_correct,
            "Missing_Values": missing_values,
            "Duplicate_Indices": duplicate_indices,
        }
    )


# ------------------------------------------------------------
# 2. Create summary table
# ------------------------------------------------------------

background_candidate_summary = pd.DataFrame(
    background_summary_rows
)


print("\nCandidate-background integrity summary:")

display(background_candidate_summary)


# ------------------------------------------------------------
# 3. Display exact stage allocations
# ------------------------------------------------------------

for background_n in BACKGROUND_CANDIDATE_SIZES:

    print("\n" + "-" * 70)
    print(
        f"BACKGROUND N = {background_n:,} "
        f"— STAGE ALLOCATION"
    )
    print("-" * 70)

    display(
        candidate_background_allocations[
            background_n
        ]
    )


# ------------------------------------------------------------
# 4. Global validation
# ------------------------------------------------------------

correct_sizes = bool(
    (
        background_candidate_summary[
            "Background_N"
        ]
        == background_candidate_summary[
            "Actual_N"
        ]
    ).all()
)

all_stage_checks = bool(
    background_candidate_summary[
        "All_Stages_Present"
    ].all()
)

all_balanced = bool(
    (
        background_candidate_summary[
            "Max_Stage_Difference"
        ]
        <= 1
    ).all()
)

all_orders_correct = bool(
    background_candidate_summary[
        "Frozen_Order"
    ].all()
)

no_missing = bool(
    (
        background_candidate_summary[
            "Missing_Values"
        ]
        == 0
    ).all()
)

no_duplicates = bool(
    (
        background_candidate_summary[
            "Duplicate_Indices"
        ]
        == 0
    ).all()
)


print("=" * 70)
print("CANDIDATE BACKGROUND INTEGRITY SUMMARY")
print("=" * 70)

print(
    f"\nAll requested sample sizes correct : "
    f"{correct_sizes}"
)

print(
    f"All 17 stages present             : "
    f"{all_stage_checks}"
)

print(
    f"Maximum stage imbalance <= 1      : "
    f"{all_balanced}"
)

print(
    f"Frozen predictor order preserved  : "
    f"{all_orders_correct}"
)

print(
    f"No missing predictor values       : "
    f"{no_missing}"
)

print(
    f"No duplicate source indices       : "
    f"{no_duplicates}"
)


all_candidate_checks = all(
    [
        correct_sizes,
        all_stage_checks,
        all_balanced,
        all_orders_correct,
        no_missing,
        no_duplicates,
    ]
)


if not all_candidate_checks:
    raise ValueError(
        "One or more candidate development backgrounds "
        "failed integrity verification."
    )


print("\nVERIFICATION PASSED")

print(
    "All candidate development backgrounds were constructed "
    "successfully using the fixed stage-balanced procedure."
)

print(
    "\nNo final background size has yet been selected."
)

CANDIDATE DEVELOPMENT BACKGROUND CONSTRUCTION

Candidate-background integrity summary:


,Background_N,Actual_N,Stages_Present,Min_Per_Stage,Max_Per_Stage,Max_Stage_Difference,All_Stages_Present,Frozen_Order,Missing_Values,Duplicate_Indices
0,100,100,17,5,6,1,True,True,0,0
1,250,250,17,14,15,1,True,True,0,0
2,500,500,17,29,30,1,True,True,0,0
3,1000,1000,17,58,59,1,True,True,0,0



----------------------------------------------------------------------
BACKGROUND N = 100 — STAGE ALLOCATION
----------------------------------------------------------------------


,operating_hour,sample_n
0,50,6
1,100,6
2,150,6
3,200,6
4,250,6
5,300,6
6,350,6
7,400,6
8,450,6
9,500,6



----------------------------------------------------------------------
BACKGROUND N = 250 — STAGE ALLOCATION
----------------------------------------------------------------------


...


----------------------------------------------------------------------
BACKGROUND N = 500 — STAGE ALLOCATION
----------------------------------------------------------------------


,operating_hour,sample_n
0,50,30
1,100,30
2,150,30
3,200,30
4,250,30
5,300,30
6,350,30
7,400,29
8,450,29
9,500,29



----------------------------------------------------------------------
BACKGROUND N = 1,000 — STAGE ALLOCATION
----------------------------------------------------------------------


,operating_hour,sample_n
0,50,59
1,100,59
2,150,59
3,200,59
4,250,59
5,300,59
6,350,59
7,400,59
8,450,59
9,500,59


CANDIDATE BACKGROUND INTEGRITY SUMMARY

All requested sample sizes correct : True
All 17 stages present             : True
Maximum stage imbalance <= 1      : True
Frozen predictor order preserved  : True
No missing predictor values       : True
No duplicate source indices       : True

VERIFICATION PASSED
All candidate development backgrounds were constructed successfully using the fixed stage-balanced procedure.

No final background size has yet been selected.


In [ ]:
### 13.6.3a Diagnose XGBoost Categorical Metadata

The first interventional TreeSHAP call raised a categorical-split compatibility error.

Before altering the SHAP formulation, the frozen XGBoost model is inspected to determine whether categorical predictors or categorical tree splits were actually used during model fitting.

This distinction is important because all predictors in the present PEMFC dataset are expected to be numerical. A categorical flag stored in the XGBoost sklearn wrapper does not necessarily demonstrate that categorical splits were used in the fitted trees.

No model parameter is changed in this diagnostic step.

In [16]:
# ============================================================
# 13.6.3a Diagnose XGBoost Categorical Metadata
# ============================================================

import json

print("=" * 70)
print("XGBOOST CATEGORICAL-METADATA DIAGNOSTIC")
print("=" * 70)


# ------------------------------------------------------------
# 1. Inspect predictor dtypes in reconstructed development data
# ------------------------------------------------------------

dtype_table = pd.DataFrame(
    {
        "Feature": FROZEN_FEATURES,
        "dtype": [
            str(X_development[feature].dtype)
            for feature in FROZEN_FEATURES
        ],
    }
)

dtype_table["Is_Categorical_Dtype"] = (
    dtype_table["dtype"]
    .str.lower()
    .str.contains("category|object|string")
)


print("\nPredictor dtypes:")
display(dtype_table)


# ------------------------------------------------------------
# 2. Inspect sklearn-wrapper parameters
# ------------------------------------------------------------

model_params = final_xgb_model.get_params()

enable_categorical_param = model_params.get(
    "enable_categorical",
    "<not present>"
)

print("\n" + "-" * 70)
print("SKLEARN XGBOOST WRAPPER")
print("-" * 70)

print(
    f"enable_categorical parameter : "
    f"{enable_categorical_param}"
)


# ------------------------------------------------------------
# 3. Inspect booster feature types
# ------------------------------------------------------------

booster = final_xgb_model.get_booster()

booster_feature_types = booster.feature_types

print("\n" + "-" * 70)
print("BOOSTER FEATURE TYPES")
print("-" * 70)

print(
    f"Booster feature_types available : "
    f"{booster_feature_types is not None}"
)

print(
    f"Booster feature_types            : "
    f"{booster_feature_types}"
)


# ------------------------------------------------------------
# 4. Inspect model configuration
# ------------------------------------------------------------

booster_config = json.loads(
    booster.save_config()
)

print("\n" + "-" * 70)
print("BOOSTER CONFIGURATION CHECK")
print("-" * 70)

# Print only relevant configuration fragments rather than
# dumping the entire JSON configuration.
config_text = json.dumps(
    booster_config,
    indent=2
)

categorical_mentions = [
    line.strip()
    for line in config_text.splitlines()
    if "categor" in line.lower()
]

if categorical_mentions:
    print(
        "\nConfiguration entries containing "
        "'categor':"
    )

    for line in categorical_mentions:
        print(line)

else:
    print(
        "\nNo configuration entries containing "
        "'categor' were found."
    )


# ------------------------------------------------------------
# 5. Inspect actual tree dump for categorical split syntax
# ------------------------------------------------------------

tree_dump_json = booster.get_dump(
    dump_format="json"
)

categorical_split_indicators = [
    '"split_condition": [',
    '"categories"',
    '"category"',
]

trees_with_possible_categorical_split = []

for tree_index, tree_text in enumerate(
    tree_dump_json
):
    lower_tree_text = tree_text.lower()

    if any(
        indicator.lower() in lower_tree_text
        for indicator in categorical_split_indicators
    ):
        trees_with_possible_categorical_split.append(
            tree_index
        )


# ------------------------------------------------------------
# 6. Consolidated diagnosis
# ------------------------------------------------------------

n_categorical_dtype_features = int(
    dtype_table[
        "Is_Categorical_Dtype"
    ].sum()
)

print("\n" + "=" * 70)
print("CATEGORICAL-METADATA SUMMARY")
print("=" * 70)

print(
    f"\nCategorical/object/string predictor dtypes : "
    f"{n_categorical_dtype_features}"
)

print(
    f"Model enable_categorical setting           : "
    f"{enable_categorical_param}"
)

print(
    f"Trees flagged by simple categorical scan   : "
    f"{len(trees_with_possible_categorical_split)}"
)

if trees_with_possible_categorical_split:
    print(
        f"First flagged tree indices                 : "
        f"{trees_with_possible_categorical_split[:10]}"
    )


print("\nDIAGNOSTIC COMPLETE")
print(
    "No model fitting, retraining, or parameter modification "
    "has been performed."
)

XGBOOST CATEGORICAL-METADATA DIAGNOSTIC

Predictor dtypes:


,Feature,dtype,Is_Categorical_Dtype
0,current,float64,False
1,pressure_anode_inlet,float64,False
2,pressure_anode_outlet,float64,False
3,pressure_cathode_inlet,float64,False
4,pressure_cathode_outlet,float64,False
5,temp_anode_endplate,float64,False
6,temp_anode_dewpoint_water,float64,False
7,temp_anode_inlet,float64,False
8,temp_anode_outlet,float64,False
9,temp_cathode_dewpoint_water,float64,False



----------------------------------------------------------------------
SKLEARN XGBOOST WRAPPER
----------------------------------------------------------------------
enable_categorical parameter : True

----------------------------------------------------------------------
BOOSTER FEATURE TYPES
----------------------------------------------------------------------
Booster feature_types available : True
Booster feature_types            : ['float', 'float', 'float', 'float', 'float', 'float', 'float', 'float', 'float', 'float', 'float', 'float', 'float', 'float', 'float', 'float', 'float', 'float', 'float', 'float']

----------------------------------------------------------------------
BOOSTER CONFIGURATION CHECK
----------------------------------------------------------------------

No configuration entries containing 'categor' were found.

CATEGORICAL-METADATA SUMMARY

Categorical/object/string predictor dtypes : 0
Model enable_categorical setting           : True
Trees flagged by si

In [ ]:
### 13.6.3b Verify Underlying XGBoost Booster Equivalence for SHAP

Diagnostic inspection showed that all 20 model predictors are numerical and that the fitted XGBoost Booster contains no detected categorical splits. However, the sklearn wrapper stores `enable_categorical=True`, which causes the installed SHAP version to invoke its unsupported-categorical safeguard during interventional TreeSHAP.

To avoid modifying the frozen fitted model or changing the planned SHAP perturbation formulation, the underlying fitted XGBoost Booster is used directly for TreeSHAP.

The Booster contains the same fitted tree ensemble used by the sklearn `XGBRegressor` wrapper. Before adopting it for explanation, numerical prediction equivalence between the wrapper and Booster is explicitly verified on the fixed diagnostic holdout sample.

This is a software-compatibility adaptation only. No retraining, hyperparameter modification, feature modification, or model reselection is performed.

In [17]:
# ============================================================
# 13.6.3b Verify Underlying XGBoost Booster Equivalence
# ============================================================

import xgboost as xgb


print("=" * 70)
print("XGBOOST WRAPPER–BOOSTER EQUIVALENCE CHECK")
print("=" * 70)


# ------------------------------------------------------------
# 1. Retrieve exact fitted Booster
# ------------------------------------------------------------

frozen_booster = final_xgb_model.get_booster()


print(
    f"\nSklearn model class : "
    f"{type(final_xgb_model)}"
)

print(
    f"Underlying Booster  : "
    f"{type(frozen_booster)}"
)


# ------------------------------------------------------------
# 2. Wrapper predictions on fixed diagnostic sample
# ------------------------------------------------------------

wrapper_predictions = final_xgb_model.predict(
    X_diagnostic_explanation
)


# ------------------------------------------------------------
# 3. Booster predictions on identical observations
# ------------------------------------------------------------

diagnostic_dmatrix = xgb.DMatrix(
    X_diagnostic_explanation,
    feature_names=FROZEN_FEATURES,
)

booster_predictions = frozen_booster.predict(
    diagnostic_dmatrix
)


# ------------------------------------------------------------
# 4. Numerical equivalence
# ------------------------------------------------------------

prediction_difference = (
    wrapper_predictions
    - booster_predictions
)

max_abs_difference = float(
    np.max(
        np.abs(prediction_difference)
    )
)

mean_abs_difference = float(
    np.mean(
        np.abs(prediction_difference)
    )
)

predictions_allclose = bool(
    np.allclose(
        wrapper_predictions,
        booster_predictions,
        rtol=1e-7,
        atol=1e-9,
    )
)


print("\nPrediction comparison:")

print(
    f"  Number of observations : "
    f"{len(wrapper_predictions):,}"
)

print(
    f"  Maximum absolute difference : "
    f"{max_abs_difference:.12f} V"
)

print(
    f"  Mean absolute difference    : "
    f"{mean_abs_difference:.12f} V"
)

print(
    f"  Predictions numerically equivalent : "
    f"{predictions_allclose}"
)


# ------------------------------------------------------------
# 5. Feature metadata
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("BOOSTER FEATURE VERIFICATION")
print("-" * 70)

print(
    f"Booster feature count : "
    f"{len(frozen_booster.feature_names)}"
)

print(
    f"Frozen feature count  : "
    f"{len(FROZEN_FEATURES)}"
)

booster_order_correct = (
    list(frozen_booster.feature_names)
    == list(FROZEN_FEATURES)
)

print(
    f"Exact feature order   : "
    f"{booster_order_correct}"
)


# ------------------------------------------------------------
# 6. Final safeguard
# ------------------------------------------------------------

if not (
    predictions_allclose
    and booster_order_correct
):
    raise ValueError(
        "Underlying XGBoost Booster did not reproduce the "
        "frozen sklearn model exactly enough for SHAP use."
    )


print("\n" + "=" * 70)
print("VERIFICATION PASSED")
print("=" * 70)

print(
    "\nThe underlying XGBoost Booster reproduces the frozen "
    "XGBRegressor predictions and predictor ordering."
)

print(
    "The Booster can therefore be used as the SHAP model "
    "representation without retraining or modifying the fitted trees."
)

XGBOOST WRAPPER–BOOSTER EQUIVALENCE CHECK

Sklearn model class : <class 'xgboost.sklearn.XGBRegressor'>
Underlying Booster  : <class 'xgboost.core.Booster'>

Prediction comparison:
  Number of observations : 900
  Maximum absolute difference : 0.000000000000 V
  Mean absolute difference    : 0.000000000000 V
  Predictions numerically equivalent : True

----------------------------------------------------------------------
BOOSTER FEATURE VERIFICATION
----------------------------------------------------------------------
Booster feature count : 20
Frozen feature count  : 20
Exact feature order   : True

VERIFICATION PASSED

The underlying XGBoost Booster reproduces the frozen XGBRegressor predictions and predictor ordering.
The Booster can therefore be used as the SHAP model representation without retraining or modifying the fitted trees.


In [ ]:
### 13.6.3 Compute Interventional TreeSHAP Across Candidate Background Sizes

Interventional TreeSHAP is now evaluated across the candidate development-background sizes while keeping the fitted model and explanation observations fixed.

The purpose of this diagnostic is to determine whether the TreeSHAP reference baseline and global attribution pattern remain sufficiently stable as the number of development-background observations increases.

The following elements are held constant across all comparisons:

- the frozen XGBoost model;
- the verified underlying XGBoost Booster used as the SHAP model representation;
- the 20 frozen predictors in their original training order;
- the fixed 900-observation diagnostic holdout explanation sample;
- the interventional TreeSHAP formulation;
- and the raw model-output scale.

Only the size of the development background is varied.

Because the installed SHAP version interprets the sklearn wrapper's `enable_categorical=True` metadata as categorical-model handling, the exact fitted underlying XGBoost Booster is supplied directly to TreeSHAP. Prediction equivalence between the sklearn wrapper and Booster was verified beforehand, confirming that this is a software-interface adaptation rather than a change to the fitted model.

An explicit `Independent` SHAP masker is constructed for each candidate background. Its `max_samples` parameter is set equal to the requested candidate size. This prevents SHAP's default masker behaviour from automatically reducing backgrounds larger than 100 observations to 100 samples.

For every candidate background, the effective number of observations retained by the masker is explicitly verified before SHAP values are computed.

For each background size, the analysis records:

- the effective background size used by SHAP;
- the TreeSHAP expected value;
- the SHAP value matrix for the same 900 diagnostic explanation observations;
- global mean absolute SHAP magnitude for each predictor;
- global feature ranking;
- and computation runtime.

No feature importance interpretation or final background-size decision is made at this stage. These outputs will be compared quantitatively in the following background-stability analysis.

In [20]:
# ============================================================
# 13.6.3 Compute Interventional TreeSHAP Across Background Sizes
# ============================================================

import shap
import time


# ------------------------------------------------------------
# Containers for diagnostic SHAP outputs
# ------------------------------------------------------------

background_shap_values = {}
background_expected_values = {}
background_mean_abs_shap = {}
background_feature_rankings = {}
background_effective_sizes = {}

background_runtime_rows = []


print("=" * 70)
print("INTERVENTIONAL TREESHAP — BACKGROUND SIZE DIAGNOSTIC")
print("=" * 70)

print(
    f"\nFixed diagnostic explanation observations: "
    f"{len(X_diagnostic_explanation):,}"
)

print(
    f"Candidate background sizes: "
    f"{BACKGROUND_CANDIDATE_SIZES}"
)

print(
    "SHAP model representation: "
    "verified underlying XGBoost Booster"
)

print(
    "Background handling: explicit Independent masker "
    "with max_samples equal to requested background size"
)


# ------------------------------------------------------------
# Loop through candidate background sizes
# ------------------------------------------------------------

for background_n in BACKGROUND_CANDIDATE_SIZES:

    print("\n" + "-" * 70)
    print(
        f"BACKGROUND N = {background_n:,}"
    )
    print("-" * 70)


    # --------------------------------------------------------
    # Prepare candidate background predictor matrix
    # --------------------------------------------------------

    X_background_candidate = (
        candidate_backgrounds[
            background_n
        ][FROZEN_FEATURES]
        .copy()
    )


    # --------------------------------------------------------
    # Construct explicit SHAP background masker
    # --------------------------------------------------------

    background_masker = shap.maskers.Independent(
        X_background_candidate,
        max_samples=background_n,
    )


    # --------------------------------------------------------
    # Verify effective background size
    # --------------------------------------------------------

    effective_background_n = len(
        background_masker.data
    )

    background_effective_sizes[
        background_n
    ] = effective_background_n


    print(
        f"Requested background size : "
        f"{background_n:,}"
    )

    print(
        f"Effective masker size     : "
        f"{effective_background_n:,}"
    )


    if effective_background_n != background_n:
        raise ValueError(
            f"Background-size mismatch for requested "
            f"n={background_n:,}. "
            f"SHAP retained only "
            f"{effective_background_n:,} observations."
        )


    # --------------------------------------------------------
    # Construct interventional TreeExplainer
    # --------------------------------------------------------

    start_time = time.perf_counter()

    explainer = shap.TreeExplainer(
        frozen_booster,
        data=background_masker,
        feature_perturbation="interventional",
        model_output="raw",
    )


    # --------------------------------------------------------
    # Compute SHAP values for fixed diagnostic explanation set
    # --------------------------------------------------------

    shap_output = explainer(
        X_diagnostic_explanation,
        check_additivity=True,
    )

    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    # --------------------------------------------------------
    # Standardise SHAP matrix extraction
    # --------------------------------------------------------

    if hasattr(shap_output, "values"):
        shap_matrix = np.asarray(
            shap_output.values
        )

    else:
        shap_matrix = np.asarray(
            shap_output
        )


    # --------------------------------------------------------
    # Verify SHAP matrix dimensions
    # --------------------------------------------------------

    if shap_matrix.ndim != 2:
        raise ValueError(
            f"Unexpected SHAP matrix dimensionality for "
            f"background n={background_n:,}: "
            f"{shap_matrix.shape}"
        )


    expected_shape = (
        len(X_diagnostic_explanation),
        len(FROZEN_FEATURES),
    )


    if shap_matrix.shape != expected_shape:
        raise ValueError(
            f"Unexpected SHAP matrix shape for "
            f"background n={background_n:,}: "
            f"{shap_matrix.shape}. "
            f"Expected {expected_shape}."
        )


    # --------------------------------------------------------
    # Extract TreeSHAP expected value
    # --------------------------------------------------------

    expected_value_array = np.asarray(
        explainer.expected_value
    ).reshape(-1)


    if len(expected_value_array) != 1:
        raise ValueError(
            "Expected a single TreeSHAP expected value "
            "for the regression model."
        )


    expected_value = float(
        expected_value_array[0]
    )


    # --------------------------------------------------------
    # Compute global mean absolute SHAP values
    # --------------------------------------------------------

    mean_abs_values = np.mean(
        np.abs(shap_matrix),
        axis=0,
    )


    mean_abs_series = pd.Series(
        mean_abs_values,
        index=FROZEN_FEATURES,
        name="Mean_Abs_SHAP_V",
    ).sort_values(
        ascending=False
    )


    # --------------------------------------------------------
    # Build feature-ranking table
    # --------------------------------------------------------

    ranking = pd.DataFrame(
        {
            "Feature": mean_abs_series.index,
            "Mean_Abs_SHAP_V": mean_abs_series.values,
        }
    )


    ranking["Mean_Abs_SHAP_mV"] = (
        ranking["Mean_Abs_SHAP_V"]
        * 1000
    )


    ranking["Rank"] = np.arange(
        1,
        len(ranking) + 1
    )


    # --------------------------------------------------------
    # Store diagnostic outputs
    # --------------------------------------------------------

    background_shap_values[
        background_n
    ] = shap_matrix


    background_expected_values[
        background_n
    ] = expected_value


    background_mean_abs_shap[
        background_n
    ] = mean_abs_series


    background_feature_rankings[
        background_n
    ] = ranking


    background_runtime_rows.append(
        {
            "Background_N": background_n,
            "Effective_Background_N": effective_background_n,
            "Expected_Value_V": expected_value,
            "Expected_Value_mV": expected_value * 1000,
            "Runtime_seconds": elapsed_seconds,
            "Runtime_minutes": elapsed_seconds / 60,
        }
    )


    # --------------------------------------------------------
    # Console summary for candidate
    # --------------------------------------------------------

    print(
        f"\nExpected value : "
        f"{expected_value:.6f} V"
    )


    print(
        f"Runtime        : "
        f"{elapsed_seconds:.2f} s "
        f"({elapsed_seconds / 60:.2f} min)"
    )


    print(
        f"SHAP matrix    : "
        f"{shap_matrix.shape}"
    )


    print(
        "\nTop 5 features by mean |SHAP|:"
    )


    display(
        ranking.head(5)
    )


# ------------------------------------------------------------
# Consolidated background-size diagnostic summary
# ------------------------------------------------------------

background_runtime_summary = pd.DataFrame(
    background_runtime_rows
)


print("\n" + "=" * 70)
print("BACKGROUND-SIZE SHAP COMPUTATION SUMMARY")
print("=" * 70)


display(
    background_runtime_summary
)


# ------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------

all_shap_outputs_present = (
    set(background_shap_values.keys())
    == set(BACKGROUND_CANDIDATE_SIZES)
)


all_expected_values_present = (
    set(background_expected_values.keys())
    == set(BACKGROUND_CANDIDATE_SIZES)
)


all_rankings_present = (
    set(background_feature_rankings.keys())
    == set(BACKGROUND_CANDIDATE_SIZES)
)


all_effective_sizes_correct = all(
    background_effective_sizes[
        background_n
    ] == background_n
    for background_n
    in BACKGROUND_CANDIDATE_SIZES
)


all_shap_shapes_correct = all(
    background_shap_values[
        background_n
    ].shape
    == (
        len(X_diagnostic_explanation),
        len(FROZEN_FEATURES),
    )
    for background_n
    in BACKGROUND_CANDIDATE_SIZES
)


print("\n" + "=" * 70)
print("BACKGROUND-SIZE DIAGNOSTIC VERIFICATION")
print("=" * 70)


print(
    f"\nAll candidate SHAP outputs present : "
    f"{all_shap_outputs_present}"
)

print(
    f"All expected values present        : "
    f"{all_expected_values_present}"
)

print(
    f"All ranking tables present         : "
    f"{all_rankings_present}"
)

print(
    f"All effective sizes correct        : "
    f"{all_effective_sizes_correct}"
)

print(
    f"All SHAP matrix shapes correct     : "
    f"{all_shap_shapes_correct}"
)


if not all(
    [
        all_shap_outputs_present,
        all_expected_values_present,
        all_rankings_present,
        all_effective_sizes_correct,
        all_shap_shapes_correct,
    ]
):
    raise ValueError(
        "One or more background-size diagnostic "
        "verification checks failed."
    )


print("\nVERIFICATION PASSED")

print(
    "Interventional TreeSHAP was computed successfully "
    "for every requested candidate background size."
)

print(
    "The explicit SHAP maskers retained the full requested "
    "background sizes, so the resulting outputs can now be "
    "used for quantitative background-stability assessment."
)

INTERVENTIONAL TREESHAP — BACKGROUND SIZE DIAGNOSTIC

Fixed diagnostic explanation observations: 900
Candidate background sizes: [100, 250, 500, 1000]
SHAP model representation: verified underlying XGBoost Booster
Background handling: explicit Independent masker with max_samples equal to requested background size

----------------------------------------------------------------------
BACKGROUND N = 100
----------------------------------------------------------------------
Requested background size : 100
Effective masker size     : 100


 97%|=================== | 869/900 [00:13<00:00]       


Expected value : 0.779151 V
Runtime        : 13.35 s (0.22 min)
SHAP matrix    : (900, 20)

Top 5 features by mean |SHAP|:


,Feature,Mean_Abs_SHAP_V,Mean_Abs_SHAP_mV,Rank
0,current,0.079637,79.636736,1
1,total_anode_stack_flow,0.009622,9.621663,2
2,temp_anode_endplate,0.006585,6.585376,3
3,cathode_pressure_diff,0.005816,5.815535,4
4,total_cathode_stack_flow,0.001586,1.586396,5



----------------------------------------------------------------------
BACKGROUND N = 250
----------------------------------------------------------------------
Requested background size : 250
Effective masker size     : 250


...


Expected value : 0.757106 V
Runtime        : 129.61 s (2.16 min)
SHAP matrix    : (900, 20)

Top 5 features by mean |SHAP|:


,Feature,Mean_Abs_SHAP_V,Mean_Abs_SHAP_mV,Rank
0,current,0.077805,77.805123,1
1,total_anode_stack_flow,0.009995,9.994985,2
2,cathode_pressure_diff,0.006053,6.052962,3
3,temp_anode_endplate,0.005788,5.787551,4
4,total_cathode_stack_flow,0.001727,1.726688,5



BACKGROUND-SIZE SHAP COMPUTATION SUMMARY


,Background_N,Effective_Background_N,Expected_Value_V,Expected_Value_mV,Runtime_seconds,Runtime_minutes
0,100,100,0.779151,779.150705,13.354074,0.222568
1,250,250,0.773633,773.632842,34.835676,0.580595
2,500,500,0.762257,762.257170,65.805240,1.096754
3,1000,1000,0.757106,757.106133,129.606366,2.160106



BACKGROUND-SIZE DIAGNOSTIC VERIFICATION

All candidate SHAP outputs present : True
All expected values present        : True
All ranking tables present         : True
All effective sizes correct        : True
All SHAP matrix shapes correct     : True

VERIFICATION PASSED
Interventional TreeSHAP was computed successfully for every requested candidate background size.
The explicit SHAP maskers retained the full requested background sizes, so the resulting outputs can now be used for quantitative background-stability assessment.


In [ ]:
### 13.6.4 Quantify Background-Size Stability

The candidate background sizes are now compared quantitatively to determine whether the TreeSHAP reference baseline and global attribution structure stabilise as the number of development-background observations increases.

The largest evaluated background, n = 1,000, is used as the diagnostic reference for comparison. This does not assume that n = 1,000 is automatically the final choice; it provides the most extensively sampled candidate against which smaller backgrounds can be assessed.

Background stability is evaluated using complementary measures:

- absolute difference in the TreeSHAP expected value;
- mean absolute difference in feature-level mean |SHAP| magnitudes;
- maximum absolute difference in feature-level mean |SHAP| magnitudes;
- Spearman rank correlation across all 20 feature-importance rankings;
- overlap among the top 5 features;
- overlap among the top 10 features.

Rank stability and magnitude stability are evaluated separately because a background may preserve the ordering of important features while still altering the numerical attribution magnitudes.

No universal numerical threshold is imposed. The smallest background that preserves the substantive global attribution conclusions with acceptable stability will be selected after reviewing the complete evidence.

In [21]:
# ============================================================
# 13.6.4 Quantify Background-Size Stability
# ============================================================

from scipy.stats import spearmanr


# ------------------------------------------------------------
# Reference background
# ------------------------------------------------------------

REFERENCE_BACKGROUND_N = max(
    BACKGROUND_CANDIDATE_SIZES
)

reference_expected_value = (
    background_expected_values[
        REFERENCE_BACKGROUND_N
    ]
)

reference_mean_abs = (
    background_mean_abs_shap[
        REFERENCE_BACKGROUND_N
    ]
    .reindex(FROZEN_FEATURES)
)

reference_ranking_table = (
    background_feature_rankings[
        REFERENCE_BACKGROUND_N
    ]
    .copy()
)

reference_rank_map = (
    reference_ranking_table
    .set_index("Feature")["Rank"]
    .reindex(FROZEN_FEATURES)
)


print("=" * 70)
print("BACKGROUND-SIZE STABILITY QUANTIFICATION")
print("=" * 70)

print(
    f"\nReference background size: "
    f"{REFERENCE_BACKGROUND_N:,}"
)


# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

stability_rows = []

feature_rank_comparison = pd.DataFrame(
    {
        "Feature": FROZEN_FEATURES
    }
)

feature_magnitude_comparison = pd.DataFrame(
    {
        "Feature": FROZEN_FEATURES
    }
)


# ------------------------------------------------------------
# Compare each candidate with n = 1000 reference
# ------------------------------------------------------------

for background_n in BACKGROUND_CANDIDATE_SIZES:

    candidate_expected_value = (
        background_expected_values[
            background_n
        ]
    )

    candidate_mean_abs = (
        background_mean_abs_shap[
            background_n
        ]
        .reindex(FROZEN_FEATURES)
    )

    candidate_ranking_table = (
        background_feature_rankings[
            background_n
        ]
        .copy()
    )

    candidate_rank_map = (
        candidate_ranking_table
        .set_index("Feature")["Rank"]
        .reindex(FROZEN_FEATURES)
    )


    # --------------------------------------------------------
    # Expected-value stability
    # --------------------------------------------------------

    expected_value_difference_v = abs(
        candidate_expected_value
        - reference_expected_value
    )

    expected_value_difference_mv = (
        expected_value_difference_v
        * 1000
    )


    # --------------------------------------------------------
    # Mean |SHAP| magnitude stability
    # --------------------------------------------------------

    shap_difference_v = abs(
        candidate_mean_abs
        - reference_mean_abs
    )

    shap_difference_mv = (
        shap_difference_v
        * 1000
    )

    mean_abs_shap_difference_mv = float(
        shap_difference_mv.mean()
    )

    max_abs_shap_difference_mv = float(
        shap_difference_mv.max()
    )


    # --------------------------------------------------------
    # Rank correlation across all 20 features
    # --------------------------------------------------------

    spearman_result = spearmanr(
        candidate_rank_map.values,
        reference_rank_map.values,
    )

    spearman_rho = float(
        spearman_result.statistic
    )


    # --------------------------------------------------------
    # Top-k overlap
    # --------------------------------------------------------

    candidate_top5 = set(
        candidate_ranking_table
        .nsmallest(
            5,
            "Rank"
        )["Feature"]
    )

    reference_top5 = set(
        reference_ranking_table
        .nsmallest(
            5,
            "Rank"
        )["Feature"]
    )

    candidate_top10 = set(
        candidate_ranking_table
        .nsmallest(
            10,
            "Rank"
        )["Feature"]
    )

    reference_top10 = set(
        reference_ranking_table
        .nsmallest(
            10,
            "Rank"
        )["Feature"]
    )


    top5_overlap_count = len(
        candidate_top5
        & reference_top5
    )

    top10_overlap_count = len(
        candidate_top10
        & reference_top10
    )


    top5_overlap_fraction = (
        top5_overlap_count / 5
    )

    top10_overlap_fraction = (
        top10_overlap_count / 10
    )


    # --------------------------------------------------------
    # Store candidate summary
    # --------------------------------------------------------

    stability_rows.append(
        {
            "Background_N": background_n,

            "Expected_Value_V":
                candidate_expected_value,

            "Expected_Value_Difference_mV":
                expected_value_difference_mv,

            "Mean_Abs_SHAP_Difference_mV":
                mean_abs_shap_difference_mv,

            "Max_Abs_SHAP_Difference_mV":
                max_abs_shap_difference_mv,

            "Spearman_Rank_Correlation":
                spearman_rho,

            "Top5_Overlap_Count":
                top5_overlap_count,

            "Top5_Overlap_Fraction":
                top5_overlap_fraction,

            "Top10_Overlap_Count":
                top10_overlap_count,

            "Top10_Overlap_Fraction":
                top10_overlap_fraction,
        }
    )


    # --------------------------------------------------------
    # Store feature-level comparisons
    # --------------------------------------------------------

    feature_rank_comparison[
        f"Rank_n{background_n}"
    ] = (
        candidate_rank_map.values
    )

    feature_magnitude_comparison[
        f"MeanAbsSHAP_mV_n{background_n}"
    ] = (
        candidate_mean_abs.values
        * 1000
    )


# ------------------------------------------------------------
# Consolidated stability summary
# ------------------------------------------------------------

background_stability_summary = pd.DataFrame(
    stability_rows
)


print(
    "\nBackground stability relative to "
    f"n={REFERENCE_BACKGROUND_N:,}:"
)

display(
    background_stability_summary
)


# ------------------------------------------------------------
# Rank comparison table
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("FEATURE-RANK COMPARISON")
print("-" * 70)

feature_rank_display = (
    feature_rank_comparison
    .sort_values(
        by=f"Rank_n{REFERENCE_BACKGROUND_N}"
    )
    .reset_index(drop=True)
)

display(
    feature_rank_display
)


# ------------------------------------------------------------
# Mean |SHAP| magnitude comparison
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("FEATURE-LEVEL MEAN |SHAP| COMPARISON")
print("-" * 70)

feature_magnitude_display = (
    feature_magnitude_comparison
    .copy()
)

feature_magnitude_display[
    "Reference_Rank"
] = (
    reference_rank_map.values
)

feature_magnitude_display = (
    feature_magnitude_display
    .sort_values(
        "Reference_Rank"
    )
    .reset_index(drop=True)
)

display(
    feature_magnitude_display
)


# ------------------------------------------------------------
# Adjacent-background expected-value changes
# ------------------------------------------------------------

adjacent_rows = []

sorted_background_sizes = sorted(
    BACKGROUND_CANDIDATE_SIZES
)

for previous_n, current_n in zip(
    sorted_background_sizes[:-1],
    sorted_background_sizes[1:],
):

    expected_change_mv = abs(
        background_expected_values[
            current_n
        ]
        - background_expected_values[
            previous_n
        ]
    ) * 1000

    previous_mean_abs = (
        background_mean_abs_shap[
            previous_n
        ]
        .reindex(FROZEN_FEATURES)
    )

    current_mean_abs = (
        background_mean_abs_shap[
            current_n
        ]
        .reindex(FROZEN_FEATURES)
    )

    mean_magnitude_change_mv = float(
        (
            abs(
                current_mean_abs
                - previous_mean_abs
            )
            * 1000
        ).mean()
    )

    adjacent_rows.append(
        {
            "Previous_Background_N":
                previous_n,

            "Current_Background_N":
                current_n,

            "Expected_Value_Change_mV":
                expected_change_mv,

            "Mean_Abs_SHAP_Change_mV":
                mean_magnitude_change_mv,
        }
    )


adjacent_background_changes = pd.DataFrame(
    adjacent_rows
)


print("\n" + "-" * 70)
print("ADJACENT BACKGROUND-SIZE CHANGES")
print("-" * 70)

display(
    adjacent_background_changes
)


print("\nVERIFICATION COMPLETE")

print(
    "Background-size stability has been quantified using "
    "expected-value, attribution-magnitude, feature-ranking, "
    "and top-k overlap measures."
)

print(
    "No final background size has yet been selected."
)

BACKGROUND-SIZE STABILITY QUANTIFICATION

Reference background size: 1,000

Background stability relative to n=1,000:


,Background_N,Expected_Value_V,Expected_Value_Difference_mV,Mean_Abs_SHAP_Difference_mV,Max_Abs_SHAP_Difference_mV,Spearman_Rank_Correlation,Top5_Overlap_Count,Top5_Overlap_Fraction,Top10_Overlap_Count,Top10_Overlap_Fraction
0,100,0.779151,22.044571,0.196058,1.831613,0.990977,5,1.000000,10,1.000000
1,250,0.773633,16.526709,0.154414,1.451567,0.986466,5,1.000000,10,1.000000
2,500,0.762257,5.151037,0.061789,0.518433,0.996992,5,1.000000,10,1.000000
3,1000,0.757106,0.000000,0.000000,0.000000,1.000000,5,1.000000,10,1.000000



----------------------------------------------------------------------
FEATURE-RANK COMPARISON
----------------------------------------------------------------------


,Feature,Rank_n100,Rank_n250,Rank_n500,Rank_n1000
0,current,1,1,1,1
1,total_anode_stack_flow,2,2,2,2
2,cathode_pressure_diff,4,4,4,3
3,temp_anode_endplate,3,3,3,4
4,total_cathode_stack_flow,5,5,5,5
5,pressure_cathode_outlet,6,7,7,6
6,temp_anode_dewpoint_water,8,6,6,7
7,anode_temp_diff,7,8,8,8
8,cathode_dewpoint_offset,9,9,9,9
9,temp_cathode_dewpoint_water,10,10,10,10



----------------------------------------------------------------------
FEATURE-LEVEL MEAN |SHAP| COMPARISON
----------------------------------------------------------------------


,Feature,MeanAbsSHAP_mV_n100,MeanAbsSHAP_mV_n250,MeanAbsSHAP_mV_n500,MeanAbsSHAP_mV_n1000,Reference_Rank
0,current,79.636736,79.256690,78.323556,77.805123,1
1,total_anode_stack_flow,9.621663,9.611563,9.828945,9.994985,2
2,cathode_pressure_diff,5.815535,5.785693,5.898352,6.052962,3
3,temp_anode_endplate,6.585376,6.240313,6.015600,5.787551,4
4,total_cathode_stack_flow,1.586396,1.719236,1.740042,1.726688,5
5,pressure_cathode_outlet,1.294194,1.142191,1.219925,1.289668,6
6,temp_anode_dewpoint_water,1.104946,1.224391,1.284046,1.289235,7
7,anode_temp_diff,1.131222,1.076573,1.123598,1.120163,8
8,cathode_dewpoint_offset,0.942512,0.894167,0.891995,0.892522,9
9,temp_cathode_dewpoint_water,0.741479,0.773543,0.810293,0.822385,10



----------------------------------------------------------------------
ADJACENT BACKGROUND-SIZE CHANGES
----------------------------------------------------------------------


,Previous_Background_N,Current_Background_N,Expected_Value_Change_mV,Mean_Abs_SHAP_Change_mV
0,100,250,5.517862,0.070612
1,250,500,11.375672,0.095995
2,500,1000,5.151037,0.061789



VERIFICATION COMPLETE
Background-size stability has been quantified using expected-value, attribution-magnitude, feature-ranking, and top-k overlap measures.
No final background size has yet been selected.


In [ ]:
### 13.6.5 Final Development-Background Selection

Background-size sensitivity was evaluated using candidate development backgrounds of 100, 250, 500 and 1,000 observations while holding the fitted XGBoost model and the 900-observation diagnostic explanation sample constant.

The 1,000-observation background was used as the diagnostic comparison reference rather than being assumed automatically to be the optimal choice.

The 500-observation background reproduced the substantive global attribution structure obtained with the 1,000-observation reference. Relative to n = 1,000, it achieved a Spearman feature-rank correlation of 0.997, preserved all five of the top-five predictors and all ten of the top-ten predictors, and produced a mean feature-level difference in mean absolute SHAP magnitude of approximately 0.062 mV.

The largest individual difference in mean absolute SHAP magnitude was approximately 0.518 mV. The principal feature hierarchy was therefore highly stable, with only a minor exchange in ordering between closely ranked predictors.

The TreeSHAP expected value remained somewhat sensitive to background size. The expected value for n = 500 differed from the n = 1,000 reference by approximately 5.15 mV. Accordingly, the analysis does not claim complete numerical convergence of the SHAP baseline. Instead, n = 500 is selected because the substantive global attribution conclusions had stabilised while computational cost remained materially lower than for n = 1,000.

The final primary interventional TreeSHAP background is therefore fixed at 500 stage-balanced observations sampled from the 50–850 h development regime using the previously defined deterministic sampling procedure.

This background represents the model's development-stage reference distribution for subsequent SHAP analysis. It should not be interpreted as a healthy-state or beginning-of-life voltage reference.

In [22]:
# ============================================================
# 13.6.5 Freeze Final Development Background
# ============================================================

FINAL_BACKGROUND_N = 500


print("=" * 70)
print("FINAL DEVELOPMENT BACKGROUND SELECTION")
print("=" * 70)


# ------------------------------------------------------------
# Retrieve already-validated 500-observation background
# ------------------------------------------------------------

final_background_data = (
    candidate_backgrounds[
        FINAL_BACKGROUND_N
    ]
    .copy()
)

X_final_background = (
    final_background_data[
        FROZEN_FEATURES
    ]
    .copy()
)


# ------------------------------------------------------------
# Retrieve allocation
# ------------------------------------------------------------

final_background_allocation = (
    candidate_background_allocations[
        FINAL_BACKGROUND_N
    ]
    .copy()
)


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

correct_total_n = (
    len(final_background_data)
    == FINAL_BACKGROUND_N
)

all_development_stages_present = (
    set(
        final_background_data[
            STAGE_COLUMN
        ].unique()
    )
    == set(DEVELOPMENT_STAGES)
)

frozen_predictor_order_correct = (
    list(X_final_background.columns)
    == list(FROZEN_FEATURES)
)

no_missing_predictors = (
    X_final_background
    .isna()
    .sum()
    .sum()
    == 0
)

no_duplicate_source_indices = (
    final_background_data
    .index
    .duplicated()
    .sum()
    == 0
)

stage_counts = (
    final_background_data[
        STAGE_COLUMN
    ]
    .value_counts()
)

maximum_stage_imbalance = (
    stage_counts.max()
    - stage_counts.min()
)

stage_balance_correct = (
    maximum_stage_imbalance <= 1
)


# ------------------------------------------------------------
# Display frozen background characteristics
# ------------------------------------------------------------

print(
    f"\nSelected background size       : "
    f"{FINAL_BACKGROUND_N:,}"
)

print(
    f"Development stages represented : "
    f"{len(stage_counts)}"
)

print(
    f"Minimum rows per stage         : "
    f"{stage_counts.min()}"
)

print(
    f"Maximum rows per stage         : "
    f"{stage_counts.max()}"
)

print(
    f"Maximum stage imbalance        : "
    f"{maximum_stage_imbalance}"
)


print("\nFinal stage allocation:")

display(
    final_background_allocation
)


# ------------------------------------------------------------
# Record stability evidence supporting the decision
# ------------------------------------------------------------

final_background_stability = (
    background_stability_summary[
        background_stability_summary[
            "Background_N"
        ]
        == FINAL_BACKGROUND_N
    ]
    .copy()
)


print("\nStability evidence relative to n=1,000:")

display(
    final_background_stability
)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("FINAL BACKGROUND INTEGRITY CHECKS")
print("-" * 70)

print(
    f"Correct total sample size       : "
    f"{correct_total_n}"
)

print(
    f"All development stages present  : "
    f"{all_development_stages_present}"
)

print(
    f"Stage imbalance <= 1            : "
    f"{stage_balance_correct}"
)

print(
    f"Frozen predictor order correct  : "
    f"{frozen_predictor_order_correct}"
)

print(
    f"No missing predictor values     : "
    f"{no_missing_predictors}"
)

print(
    f"No duplicate source indices     : "
    f"{no_duplicate_source_indices}"
)


if not all(
    [
        correct_total_n,
        all_development_stages_present,
        stage_balance_correct,
        frozen_predictor_order_correct,
        no_missing_predictors,
        no_duplicate_source_indices,
    ]
):
    raise ValueError(
        "Final development-background verification failed."
    )


print("\n" + "=" * 70)
print("FINAL BACKGROUND FROZEN")
print("=" * 70)

print(
    "\nThe primary interventional TreeSHAP background is now "
    "fixed at 500 stage-balanced development observations."
)

print(
    "No further background-size selection will be performed "
    "using subsequent SHAP results."
)

FINAL DEVELOPMENT BACKGROUND SELECTION

Selected background size       : 500
Development stages represented : 17
Minimum rows per stage         : 29
Maximum rows per stage         : 30
Maximum stage imbalance        : 1

Final stage allocation:


,operating_hour,sample_n
0,50,30
1,100,30
2,150,30
3,200,30
4,250,30
5,300,30
6,350,30
7,400,29
8,450,29
9,500,29



Stability evidence relative to n=1,000:


,Background_N,Expected_Value_V,Expected_Value_Difference_mV,Mean_Abs_SHAP_Difference_mV,Max_Abs_SHAP_Difference_mV,Spearman_Rank_Correlation,Top5_Overlap_Count,Top5_Overlap_Fraction,Top10_Overlap_Count,Top10_Overlap_Fraction
2,500,0.762257,5.151037,0.061789,0.518433,0.996992,5,1.000000,10,1.000000



----------------------------------------------------------------------
FINAL BACKGROUND INTEGRITY CHECKS
----------------------------------------------------------------------
Correct total sample size       : True
All development stages present  : True
Stage imbalance <= 1            : True
Frozen predictor order correct  : True
No missing predictor values     : True
No duplicate source indices     : True

FINAL BACKGROUND FROZEN

The primary interventional TreeSHAP background is now fixed at 500 stage-balanced development observations.
No further background-size selection will be performed using subsequent SHAP results.


In [ ]:
### 13.7 Holdout Explanation-Sample Design

With the development reference background fixed, the next methodological decision concerns the number of later-stage observations required for the primary SHAP analysis.

The complete later-stage holdout contains 580,560 observations from the previously untouched 900 h, 950 h and 1000 h durability stages. Computing and presenting SHAP explanations for every observation is not necessary if a smaller representative sample reproduces the substantive global attribution structure.

Candidate explanation-sample sizes of 3,000, 7,500, 15,000 and 30,000 observations are therefore evaluated empirically.

Each candidate sample is constructed using equal allocation across the three holdout durability stages. Equal stage weighting is a study-specific analytical choice intended to prevent the larger 1000 h stage from receiving greater influence solely because it contains more recorded observations.

Within each durability stage, observations are sampled randomly without replacement using a reproducible random seed. The naturally occurring within-stage distributions of Current and the remaining operational predictors are preserved. Current/load intervals are therefore not artificially balanced in the primary explanation sample.

The candidate explanation samples will subsequently be compared using global mean absolute SHAP magnitudes, feature-ranking stability and top-k feature overlap. Repeated-seed analysis will then assess whether the selected sample size is robust to the particular observations drawn.

The final explanation-sample size is not fixed a priori. The smallest evaluated sample that provides sufficiently stable substantive attribution conclusions will be retained for the primary holdout SHAP analysis.

In [23]:
# ============================================================
# 13.7.1 Construct Candidate Holdout Explanation Samples
# ============================================================

EXPLANATION_CANDIDATE_SIZES = [
    3000,
    7500,
    15000,
    30000,
]


candidate_explanation_samples = {}
candidate_explanation_allocations = {}

explanation_integrity_rows = []


print("=" * 70)
print("CANDIDATE HOLDOUT EXPLANATION-SAMPLE CONSTRUCTION")
print("=" * 70)

print(
    f"\nHoldout population size : "
    f"{len(holdout_data):,}"
)

print(
    f"Holdout stages          : "
    f"{HOLDOUT_STAGES}"
)

print(
    f"Candidate sample sizes  : "
    f"{EXPLANATION_CANDIDATE_SIZES}"
)


# ------------------------------------------------------------
# Construct each candidate explanation sample
# ------------------------------------------------------------

for explanation_n in EXPLANATION_CANDIDATE_SIZES:

    sampled_data, allocation = sample_stage_balanced(
        data=holdout_data,
        stages=HOLDOUT_STAGES,
        total_n=explanation_n,
        stage_column=STAGE_COLUMN,
        random_seed=RANDOM_SEED,
    )


    # --------------------------------------------------------
    # Preserve complete sampled rows
    # --------------------------------------------------------

    candidate_explanation_samples[
        explanation_n
    ] = sampled_data.copy()

    candidate_explanation_allocations[
        explanation_n
    ] = allocation.copy()


    # --------------------------------------------------------
    # Predictor matrix
    # --------------------------------------------------------

    X_candidate = sampled_data[
        FROZEN_FEATURES
    ]


    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    actual_n = len(
        sampled_data
    )

    stage_counts = (
        sampled_data[
            STAGE_COLUMN
        ]
        .value_counts()
        .sort_index()
    )

    stages_present = len(
        stage_counts
    )

    min_per_stage = int(
        stage_counts.min()
    )

    max_per_stage = int(
        stage_counts.max()
    )

    max_stage_difference = (
        max_per_stage
        - min_per_stage
    )

    all_stages_present = (
        set(stage_counts.index)
        == set(HOLDOUT_STAGES)
    )

    frozen_order_correct = (
        list(X_candidate.columns)
        == list(FROZEN_FEATURES)
    )

    missing_values = int(
        X_candidate
        .isna()
        .sum()
        .sum()
    )

    duplicate_indices = int(
        sampled_data
        .index
        .duplicated()
        .sum()
    )


    # --------------------------------------------------------
    # Store integrity results
    # --------------------------------------------------------

    explanation_integrity_rows.append(
        {
            "Explanation_N":
                explanation_n,

            "Actual_N":
                actual_n,

            "Stages_Present":
                stages_present,

            "Min_Per_Stage":
                min_per_stage,

            "Max_Per_Stage":
                max_per_stage,

            "Max_Stage_Difference":
                max_stage_difference,

            "All_Stages_Present":
                all_stages_present,

            "Frozen_Order":
                frozen_order_correct,

            "Missing_Values":
                missing_values,

            "Duplicate_Indices":
                duplicate_indices,
        }
    )


# ------------------------------------------------------------
# Consolidated integrity summary
# ------------------------------------------------------------

explanation_candidate_summary = pd.DataFrame(
    explanation_integrity_rows
)


print("\nCandidate explanation-sample integrity summary:")

display(
    explanation_candidate_summary
)


# ------------------------------------------------------------
# Display exact stage allocations
# ------------------------------------------------------------

for explanation_n in EXPLANATION_CANDIDATE_SIZES:

    print("\n" + "-" * 70)

    print(
        f"EXPLANATION N = {explanation_n:,} "
        "— STAGE ALLOCATION"
    )

    print("-" * 70)

    display(
        candidate_explanation_allocations[
            explanation_n
        ]
    )


# ------------------------------------------------------------
# Global verification
# ------------------------------------------------------------

all_requested_sizes_correct = all(
    explanation_candidate_summary[
        "Explanation_N"
    ]
    == explanation_candidate_summary[
        "Actual_N"
    ]
)

all_three_stages_present = bool(
    explanation_candidate_summary[
        "All_Stages_Present"
    ].all()
)

all_stage_allocations_equal = bool(
    (
        explanation_candidate_summary[
            "Max_Stage_Difference"
        ] == 0
    ).all()
)

all_frozen_orders_correct = bool(
    explanation_candidate_summary[
        "Frozen_Order"
    ].all()
)

no_missing_values = bool(
    (
        explanation_candidate_summary[
            "Missing_Values"
        ] == 0
    ).all()
)

no_duplicate_indices = bool(
    (
        explanation_candidate_summary[
            "Duplicate_Indices"
        ] == 0
    ).all()
)


print("\n" + "=" * 70)
print("CANDIDATE EXPLANATION-SAMPLE INTEGRITY SUMMARY")
print("=" * 70)

print(
    f"\nAll requested sample sizes correct : "
    f"{all_requested_sizes_correct}"
)

print(
    f"All 3 holdout stages present       : "
    f"{all_three_stages_present}"
)

print(
    f"Exact equal stage allocation       : "
    f"{all_stage_allocations_equal}"
)

print(
    f"Frozen predictor order preserved   : "
    f"{all_frozen_orders_correct}"
)

print(
    f"No missing predictor values        : "
    f"{no_missing_values}"
)

print(
    f"No duplicate source indices        : "
    f"{no_duplicate_indices}"
)


if not all(
    [
        all_requested_sizes_correct,
        all_three_stages_present,
        all_stage_allocations_equal,
        all_frozen_orders_correct,
        no_missing_values,
        no_duplicate_indices,
    ]
):
    raise ValueError(
        "One or more candidate explanation-sample "
        "integrity checks failed."
    )


print("\nVERIFICATION PASSED")

print(
    "All candidate holdout explanation samples were "
    "constructed successfully using equal durability-stage "
    "allocation and the fixed reproducible sampling procedure."
)

print(
    "\nNo final explanation-sample size has yet been selected."
)

CANDIDATE HOLDOUT EXPLANATION-SAMPLE CONSTRUCTION

Holdout population size : 580,560
Holdout stages          : [900, 950, 1000]
Candidate sample sizes  : [3000, 7500, 15000, 30000]

Candidate explanation-sample integrity summary:


,Explanation_N,Actual_N,Stages_Present,Min_Per_Stage,Max_Per_Stage,Max_Stage_Difference,All_Stages_Present,Frozen_Order,Missing_Values,Duplicate_Indices
0,3000,3000,3,1000,1000,0,True,True,0,0
1,7500,7500,3,2500,2500,0,True,True,0,0
2,15000,15000,3,5000,5000,0,True,True,0,0
3,30000,30000,3,10000,10000,0,True,True,0,0



----------------------------------------------------------------------
EXPLANATION N = 3,000 — STAGE ALLOCATION
----------------------------------------------------------------------


,operating_hour,sample_n
0,900,1000
1,950,1000
2,1000,1000



----------------------------------------------------------------------
EXPLANATION N = 7,500 — STAGE ALLOCATION
----------------------------------------------------------------------


...


----------------------------------------------------------------------
EXPLANATION N = 15,000 — STAGE ALLOCATION
----------------------------------------------------------------------


,operating_hour,sample_n
0,900,5000
1,950,5000
2,1000,5000



----------------------------------------------------------------------
EXPLANATION N = 30,000 — STAGE ALLOCATION
----------------------------------------------------------------------


,operating_hour,sample_n
0,900,10000
1,950,10000
2,1000,10000



CANDIDATE EXPLANATION-SAMPLE INTEGRITY SUMMARY

All requested sample sizes correct : True
All 3 holdout stages present       : True
Exact equal stage allocation       : True
Frozen predictor order preserved   : True
No missing predictor values        : True
No duplicate source indices        : True

VERIFICATION PASSED
All candidate holdout explanation samples were constructed successfully using equal durability-stage allocation and the fixed reproducible sampling procedure.

No final explanation-sample size has yet been selected.


In [ ]:
### 13.7.2 Representativeness Audit of Candidate Explanation Samples

Before evaluating SHAP convergence, the candidate holdout explanation samples are descriptively audited against their corresponding full holdout-stage populations.

The primary sampling strategy intentionally preserves the naturally occurring within-stage operational distribution rather than artificially balancing Current/load intervals. Therefore, the audit examines whether random stage-balanced sampling has retained representative coverage of the Current variable within each of the 900 h, 950 h and 1000 h durability stages.

Current is examined explicitly because it represents the applied electrical load and is expected to be an important determinant of instantaneous stack voltage.

For each durability stage and candidate explanation-sample size, the Current distribution is compared with the corresponding full-stage population using:

- mean;
- standard deviation;
- minimum and maximum;
- 25th percentile;
- median;
- 75th percentile;
- and additional distribution quantiles.

Absolute differences in the mean, median and selected quantiles are also calculated.

This audit is descriptive rather than based on null-hypothesis significance testing. Given the very large underlying dataset, conventional significance tests could identify very small distributional differences that are not practically important for the present explanation-sampling purpose.

The purpose is therefore to confirm broad coverage and practical similarity of the sampled Current distributions before proceeding to SHAP sample-size convergence analysis.

In [24]:
# ============================================================
# 13.7.2 Representativeness Audit of Explanation Samples
# ============================================================

CURRENT_FEATURE = "current"

CURRENT_QUANTILES = [
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
    0.99,
]


print("=" * 70)
print("EXPLANATION-SAMPLE REPRESENTATIVENESS AUDIT")
print("=" * 70)

print(
    "\nAudit variable : Current (applied electrical load)"
)

print(
    f"Holdout stages : {HOLDOUT_STAGES}"
)


# ------------------------------------------------------------
# Helper function for descriptive Current statistics
# ------------------------------------------------------------

def current_distribution_summary(
    data,
    stage,
    sample_label,
):
    stage_data = data.loc[
        data[STAGE_COLUMN] == stage,
        CURRENT_FEATURE,
    ]

    quantiles = stage_data.quantile(
        CURRENT_QUANTILES
    )

    return {
        "Sample": sample_label,
        "Stage_h": stage,
        "N": len(stage_data),

        "Mean_A":
            stage_data.mean(),

        "SD_A":
            stage_data.std(),

        "Min_A":
            stage_data.min(),

        "Q01_A":
            quantiles.loc[0.01],

        "Q05_A":
            quantiles.loc[0.05],

        "Q25_A":
            quantiles.loc[0.25],

        "Median_A":
            quantiles.loc[0.50],

        "Q75_A":
            quantiles.loc[0.75],

        "Q95_A":
            quantiles.loc[0.95],

        "Q99_A":
            quantiles.loc[0.99],

        "Max_A":
            stage_data.max(),
    }


# ------------------------------------------------------------
# Full holdout-stage population summaries
# ------------------------------------------------------------

population_rows = []

for stage in HOLDOUT_STAGES:

    population_rows.append(
        current_distribution_summary(
            data=holdout_data,
            stage=stage,
            sample_label="Full_Holdout",
        )
    )


population_current_summary = pd.DataFrame(
    population_rows
)


print("\n" + "-" * 70)
print("FULL HOLDOUT CURRENT DISTRIBUTIONS")
print("-" * 70)

display(
    population_current_summary
)


# ------------------------------------------------------------
# Candidate-sample summaries
# ------------------------------------------------------------

candidate_rows = []

for explanation_n in EXPLANATION_CANDIDATE_SIZES:

    candidate_data = (
        candidate_explanation_samples[
            explanation_n
        ]
    )

    for stage in HOLDOUT_STAGES:

        candidate_rows.append(
            current_distribution_summary(
                data=candidate_data,
                stage=stage,
                sample_label=f"n={explanation_n}",
            )
        )


candidate_current_summary = pd.DataFrame(
    candidate_rows
)


print("\n" + "-" * 70)
print("CANDIDATE EXPLANATION-SAMPLE CURRENT DISTRIBUTIONS")
print("-" * 70)

display(
    candidate_current_summary
)


# ------------------------------------------------------------
# Calculate deviations from corresponding full-stage population
# ------------------------------------------------------------

comparison_rows = []

comparison_columns = [
    "Mean_A",
    "SD_A",
    "Q01_A",
    "Q05_A",
    "Q25_A",
    "Median_A",
    "Q75_A",
    "Q95_A",
    "Q99_A",
]


for explanation_n in EXPLANATION_CANDIDATE_SIZES:

    candidate_data = (
        candidate_explanation_samples[
            explanation_n
        ]
    )

    for stage in HOLDOUT_STAGES:

        population_stage = (
            population_current_summary.loc[
                population_current_summary[
                    "Stage_h"
                ] == stage
            ]
            .iloc[0]
        )

        candidate_stage = (
            current_distribution_summary(
                data=candidate_data,
                stage=stage,
                sample_label=f"n={explanation_n}",
            )
        )


        row = {
            "Explanation_N":
                explanation_n,

            "Stage_h":
                stage,

            "Sample_N":
                candidate_stage["N"],
        }


        for column in comparison_columns:

            row[
                f"Abs_Diff_{column}"
            ] = abs(
                candidate_stage[column]
                - population_stage[column]
            )


        comparison_rows.append(
            row
        )


current_representativeness_comparison = (
    pd.DataFrame(
        comparison_rows
    )
)


print("\n" + "-" * 70)
print("ABSOLUTE DIFFERENCES FROM FULL-STAGE CURRENT DISTRIBUTION")
print("-" * 70)

display(
    current_representativeness_comparison
)


# ------------------------------------------------------------
# Summarise deviations across the three stages
# ------------------------------------------------------------

representativeness_summary_rows = []


for explanation_n in EXPLANATION_CANDIDATE_SIZES:

    subset = (
        current_representativeness_comparison.loc[
            current_representativeness_comparison[
                "Explanation_N"
            ] == explanation_n
        ]
    )


    representativeness_summary_rows.append(
        {
            "Explanation_N":
                explanation_n,

            "Max_Mean_Difference_A":
                subset[
                    "Abs_Diff_Mean_A"
                ].max(),

            "Max_SD_Difference_A":
                subset[
                    "Abs_Diff_SD_A"
                ].max(),

            "Max_Median_Difference_A":
                subset[
                    "Abs_Diff_Median_A"
                ].max(),

            "Max_Q05_Difference_A":
                subset[
                    "Abs_Diff_Q05_A"
                ].max(),

            "Max_Q95_Difference_A":
                subset[
                    "Abs_Diff_Q95_A"
                ].max(),

            "Max_Q01_Difference_A":
                subset[
                    "Abs_Diff_Q01_A"
                ].max(),

            "Max_Q99_Difference_A":
                subset[
                    "Abs_Diff_Q99_A"
                ].max(),
        }
    )


current_representativeness_summary = (
    pd.DataFrame(
        representativeness_summary_rows
    )
)


print("\n" + "=" * 70)
print("CURRENT REPRESENTATIVENESS SUMMARY")
print("=" * 70)

display(
    current_representativeness_summary
)


# ------------------------------------------------------------
# Basic coverage checks
# ------------------------------------------------------------

all_samples_have_three_stages = all(
    candidate_current_summary
    .groupby("Sample")[
        "Stage_h"
    ]
    .nunique()
    == len(HOLDOUT_STAGES)
)


all_samples_have_current_variation = all(
    candidate_current_summary[
        "SD_A"
    ] > 0
)


all_samples_cover_broad_current_range = all(
    candidate_current_summary[
        "Q95_A"
    ]
    > candidate_current_summary[
        "Q05_A"
    ]
)


print("\n" + "-" * 70)
print("REPRESENTATIVENESS AUDIT CHECKS")
print("-" * 70)

print(
    f"All candidates contain all 3 stages : "
    f"{all_samples_have_three_stages}"
)

print(
    f"Current varies within every sample  : "
    f"{all_samples_have_current_variation}"
)

print(
    f"Broad Current range retained        : "
    f"{all_samples_cover_broad_current_range}"
)


if not all(
    [
        all_samples_have_three_stages,
        all_samples_have_current_variation,
        all_samples_cover_broad_current_range,
    ]
):
    raise ValueError(
        "One or more explanation-sample "
        "representativeness checks failed."
    )


print("\nVERIFICATION PASSED")

print(
    "Candidate explanation samples retain within-stage "
    "Current variation and broad operating-range coverage."
)

print(
    "Detailed descriptive deviations should now be reviewed "
    "before SHAP sample-size convergence analysis."
)

EXPLANATION-SAMPLE REPRESENTATIVENESS AUDIT

Audit variable : Current (applied electrical load)
Holdout stages : [900, 950, 1000]

----------------------------------------------------------------------
FULL HOLDOUT CURRENT DISTRIBUTIONS
----------------------------------------------------------------------


,Sample,Stage_h,N,Mean_A,SD_A,Min_A,Q01_A,Q05_A,Q25_A,Median_A,Q75_A,Q95_A,Q99_A,Max_A
0,Full_Holdout,900,179360,9.986154,9.388818,-0.002500,0.000000,1.760400,1.760400,9.484800,14.813900,29.589400,35.534200,35.539000
1,Full_Holdout,950,179360,9.920678,9.395297,-0.002500,0.000000,1.760400,1.760400,9.484800,14.813900,29.589400,35.534200,35.539000
2,Full_Holdout,1000,221840,9.966173,9.390565,-0.002500,0.000000,1.760400,1.760400,9.484800,14.813900,29.589400,35.534200,35.539000



----------------------------------------------------------------------
CANDIDATE EXPLANATION-SAMPLE CURRENT DISTRIBUTIONS
----------------------------------------------------------------------


,Sample,Stage_h,N,Mean_A,SD_A,Min_A,Q01_A,Q05_A,Q25_A,Median_A,Q75_A,Q95_A,Q99_A,Max_A
0,n=3000,900,1000,9.765603,9.370123,0.000000,0.000000,1.760400,1.760400,9.484800,14.813900,29.589400,35.534200,35.539000
1,n=3000,950,1000,9.837336,9.463508,0.000000,0.000000,1.760400,1.760400,9.484800,14.813900,29.589520,35.534200,35.539000
2,n=3000,1000,1000,10.082306,9.228452,-0.002500,0.000000,1.760400,1.760400,9.484800,14.813900,29.589400,35.534200,35.539000
3,n=7500,900,2500,9.967513,9.281725,-0.002500,0.000000,1.760400,1.760400,9.484800,14.813900,29.589400,35.534200,35.539000
4,n=7500,950,2500,9.848136,9.320072,-0.002500,0.000000,1.760400,1.760400,9.484800,14.813900,29.589400,35.534200,35.539000
5,n=7500,1000,2500,9.975097,9.320114,-0.002500,0.000000,1.760400,1.760400,9.484800,14.813900,29.589400,35.534200,35.539000
6,n=15000,900,5000,9.991725,9.370147,-0.002500,0.000000,1.760400,1.760400,9.484800,14.813900,29.589400,35.534200,35.539000
7,n=15000,950,5000,9.894998,9.249579,-0.002500,0.000000,1.760400,1.760400,9.484800,14.813900,29.589400,35.534200,35.539000
8,n=15000,1000,5000,9.984521,9.338460,-0.002500,0.000000,1.760400,1.760400,9.484800,14.814500,29.589400,35.534200,35.539000
9,n=30000,900,10000,10.068337,9.406365,-0.002500,0.000000,1.760400,1.760400,9.484800,14.813900,29.589400,35.534200,35.539000



----------------------------------------------------------------------
ABSOLUTE DIFFERENCES FROM FULL-STAGE CURRENT DISTRIBUTION
----------------------------------------------------------------------


,Explanation_N,Stage_h,Sample_N,Abs_Diff_Mean_A,Abs_Diff_SD_A,Abs_Diff_Q01_A,Abs_Diff_Q05_A,Abs_Diff_Q25_A,Abs_Diff_Median_A,Abs_Diff_Q75_A,Abs_Diff_Q95_A,Abs_Diff_Q99_A
0,3000,900,1000,0.220552,0.018695,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,3000,950,1000,0.083343,0.068211,0.000000,0.000000,0.000000,0.000000,0.000000,0.000120,0.000000
2,3000,1000,1000,0.116133,0.162113,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,7500,900,2500,0.018642,0.107093,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,7500,950,2500,0.072542,0.075224,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,7500,1000,2500,0.008924,0.070451,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,15000,900,5000,0.005571,0.018671,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,15000,950,5000,0.025680,0.145718,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,15000,1000,5000,0.018348,0.052106,0.000000,0.000000,0.000000,0.000000,0.000600,0.000000,0.000000
9,30000,900,10000,0.082183,0.017546,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



CURRENT REPRESENTATIVENESS SUMMARY


,Explanation_N,Max_Mean_Difference_A,Max_SD_Difference_A,Max_Median_Difference_A,Max_Q05_Difference_A,Max_Q95_Difference_A,Max_Q01_Difference_A,Max_Q99_Difference_A
0,3000,0.220552,0.162113,0.000000,0.000000,0.000120,0.000000,0.000000
1,7500,0.072542,0.107093,0.000000,0.000000,0.000000,0.000000,0.000000
2,15000,0.025680,0.145718,0.000000,0.000000,0.000000,0.000000,0.000000
3,30000,0.092951,0.085905,0.000000,0.000000,0.000000,0.000000,0.000000



----------------------------------------------------------------------
REPRESENTATIVENESS AUDIT CHECKS
----------------------------------------------------------------------
All candidates contain all 3 stages : True
Current varies within every sample  : True
Broad Current range retained        : True

VERIFICATION PASSED
Candidate explanation samples retain within-stage Current variation and broad operating-range coverage.
Detailed descriptive deviations should now be reviewed before SHAP sample-size convergence analysis.


In [ ]:
### 13.7.3 Compute SHAP Across Candidate Explanation-Sample Sizes

The candidate explanation-sample sizes are now evaluated using the frozen primary interventional TreeSHAP configuration.

The fitted XGBoost Booster and the previously selected 500-observation development background remain fixed throughout this analysis. A single interventional TreeExplainer is constructed and then applied to the 3,000-, 7,500-, 15,000- and 30,000-observation holdout explanation samples.

Only the number of later-stage observations being explained changes across these computations. This isolates explanation-sample size as the principal experimental factor in the convergence assessment.

For each candidate explanation sample, the analysis records:

- the SHAP value matrix;
- global mean absolute SHAP magnitude for each predictor;
- global feature ranking;
- computation runtime;
- and an independent numerical additivity check comparing the SHAP-reconstructed prediction with the frozen XGBoost prediction.

The TreeSHAP expected value remains fixed because all candidate explanation samples use the same fitted model, interventional formulation and frozen 500-observation development background.

The resulting attribution structures will subsequently be compared quantitatively. The largest evaluated explanation sample, n = 30,000, will serve as the diagnostic comparison reference, without being assumed automatically to be the final explanation-sample size.

The final explanation-sample size will be selected only after evaluating attribution-magnitude stability, feature-ranking stability and top-k overlap. The objective is to retain the smallest evaluated sample that preserves the substantive global attribution conclusions.

In [26]:
# ============================================================
# 13.7.3 Compute SHAP Across Candidate Explanation Sizes
# ============================================================

import time


# ------------------------------------------------------------
# Build frozen primary interventional TreeExplainer
# ------------------------------------------------------------

print("=" * 70)
print("SHAP EXPLANATION-SAMPLE SIZE CONVERGENCE")
print("=" * 70)

print(
    f"\nFrozen development background : "
    f"{len(X_final_background):,} observations"
)

print(
    f"Candidate explanation sizes   : "
    f"{EXPLANATION_CANDIDATE_SIZES}"
)

print(
    "SHAP formulation              : "
    "Interventional TreeSHAP"
)

print(
    "Model representation          : "
    "Verified frozen XGBoost Booster"
)


# ------------------------------------------------------------
# Explicit background masker
# ------------------------------------------------------------

final_background_masker = shap.maskers.Independent(
    X_final_background,
    max_samples=FINAL_BACKGROUND_N,
)


effective_final_background_n = len(
    final_background_masker.data
)


print(
    f"Effective background size     : "
    f"{effective_final_background_n:,}"
)


if effective_final_background_n != FINAL_BACKGROUND_N:
    raise ValueError(
        "The final SHAP background masker did not retain "
        "the complete frozen background."
    )


# ------------------------------------------------------------
# Construct one fixed interventional TreeExplainer
# ------------------------------------------------------------

primary_interventional_explainer = shap.TreeExplainer(
    frozen_booster,
    data=final_background_masker,
    feature_perturbation="interventional",
    model_output="raw",
)


# ------------------------------------------------------------
# Freeze expected value
# ------------------------------------------------------------

expected_value_array = np.asarray(
    primary_interventional_explainer.expected_value
).reshape(-1)


if len(expected_value_array) != 1:
    raise ValueError(
        "Expected a single TreeSHAP expected value "
        "for the regression model."
    )


PRIMARY_EXPECTED_VALUE = float(
    expected_value_array[0]
)


print(
    f"TreeSHAP expected value       : "
    f"{PRIMARY_EXPECTED_VALUE:.6f} V"
)


# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

explanation_shap_values = {}

explanation_mean_abs_shap = {}

explanation_feature_rankings = {}

explanation_runtime_rows = []


# ------------------------------------------------------------
# Compute SHAP for every candidate explanation sample
# ------------------------------------------------------------

for explanation_n in EXPLANATION_CANDIDATE_SIZES:

    print("\n" + "-" * 70)

    print(
        f"EXPLANATION N = {explanation_n:,}"
    )

    print("-" * 70)


    # --------------------------------------------------------
    # Retrieve candidate explanation sample
    # --------------------------------------------------------

    candidate_data = (
        candidate_explanation_samples[
            explanation_n
        ]
    )


    X_candidate = (
        candidate_data[
            FROZEN_FEATURES
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Frozen XGBoost predictions
    # --------------------------------------------------------

    candidate_predictions = (
        final_xgb_model.predict(
            X_candidate
        )
    )


    # --------------------------------------------------------
    # Compute SHAP values
    # --------------------------------------------------------

    start_time = time.perf_counter()


    shap_output = (
        primary_interventional_explainer(
            X_candidate,
            check_additivity=True,
        )
    )


    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )


    # --------------------------------------------------------
    # Extract SHAP matrix
    # --------------------------------------------------------

    shap_matrix = np.asarray(
        shap_output.values
    )


    expected_shape = (
        explanation_n,
        len(FROZEN_FEATURES),
    )


    if shap_matrix.shape != expected_shape:
        raise ValueError(
            f"Unexpected SHAP matrix shape for "
            f"n={explanation_n:,}: "
            f"{shap_matrix.shape}. "
            f"Expected {expected_shape}."
        )


    # --------------------------------------------------------
    # Independent numerical additivity verification
    # --------------------------------------------------------

    reconstructed_predictions = (
        PRIMARY_EXPECTED_VALUE
        + shap_matrix.sum(axis=1)
    )


    additivity_abs_error = np.abs(
        reconstructed_predictions
        - candidate_predictions
    )


    max_additivity_error_v = float(
        additivity_abs_error.max()
    )


    mean_additivity_error_v = float(
        additivity_abs_error.mean()
    )


    # --------------------------------------------------------
    # Compute global mean |SHAP|
    # --------------------------------------------------------

    mean_abs_values = np.mean(
        np.abs(shap_matrix),
        axis=0,
    )


    mean_abs_series = pd.Series(
        mean_abs_values,
        index=FROZEN_FEATURES,
        name="Mean_Abs_SHAP_V",
    ).sort_values(
        ascending=False
    )


    # --------------------------------------------------------
    # Construct feature-ranking table
    # --------------------------------------------------------

    ranking = pd.DataFrame(
        {
            "Feature":
                mean_abs_series.index,

            "Mean_Abs_SHAP_V":
                mean_abs_series.values,
        }
    )


    ranking[
        "Mean_Abs_SHAP_mV"
    ] = (
        ranking[
            "Mean_Abs_SHAP_V"
        ]
        * 1000
    )


    ranking["Rank"] = np.arange(
        1,
        len(ranking) + 1
    )


    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    explanation_shap_values[
        explanation_n
    ] = shap_matrix


    explanation_mean_abs_shap[
        explanation_n
    ] = mean_abs_series


    explanation_feature_rankings[
        explanation_n
    ] = ranking


    explanation_runtime_rows.append(
        {
            "Explanation_N":
                explanation_n,

            "Background_N":
                FINAL_BACKGROUND_N,

            "Expected_Value_V":
                PRIMARY_EXPECTED_VALUE,

            "Runtime_seconds":
                elapsed_seconds,

            "Runtime_minutes":
                elapsed_seconds / 60,

            "Max_Additivity_Error_V":
                max_additivity_error_v,

            "Mean_Additivity_Error_V":
                mean_additivity_error_v,
        }
    )


    # --------------------------------------------------------
    # Candidate summary
    # --------------------------------------------------------

    print(
        f"\nRuntime                  : "
        f"{elapsed_seconds:.2f} s "
        f"({elapsed_seconds / 60:.2f} min)"
    )

    print(
        f"SHAP matrix              : "
        f"{shap_matrix.shape}"
    )

    print(
        f"Maximum additivity error : "
        f"{max_additivity_error_v:.10f} V"
    )

    print(
        f"Mean additivity error    : "
        f"{mean_additivity_error_v:.10f} V"
    )


    print(
        "\nTop 10 features by mean |SHAP|:"
    )


    display(
        ranking.head(10)
    )


# ------------------------------------------------------------
# Consolidated computation summary
# ------------------------------------------------------------

explanation_convergence_runtime = pd.DataFrame(
    explanation_runtime_rows
)


print("\n" + "=" * 70)
print("EXPLANATION-SAMPLE SHAP COMPUTATION SUMMARY")
print("=" * 70)


display(
    explanation_convergence_runtime
)


# ------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------

all_candidate_outputs_present = (
    set(
        explanation_shap_values.keys()
    )
    == set(
        EXPLANATION_CANDIDATE_SIZES
    )
)


all_matrix_shapes_correct = all(
    explanation_shap_values[
        explanation_n
    ].shape
    == (
        explanation_n,
        len(FROZEN_FEATURES),
    )

    for explanation_n
    in EXPLANATION_CANDIDATE_SIZES
)


all_rankings_present = (
    set(
        explanation_feature_rankings.keys()
    )
    == set(
        EXPLANATION_CANDIDATE_SIZES
    )
)


all_expected_values_identical = bool(
    np.allclose(
        explanation_convergence_runtime[
            "Expected_Value_V"
        ].values,
        PRIMARY_EXPECTED_VALUE,
    )
)


print("\n" + "=" * 70)
print("EXPLANATION-SAMPLE SHAP VERIFICATION")
print("=" * 70)


print(
    f"\nAll candidate outputs present  : "
    f"{all_candidate_outputs_present}"
)

print(
    f"All SHAP matrix shapes correct : "
    f"{all_matrix_shapes_correct}"
)

print(
    f"All ranking tables present     : "
    f"{all_rankings_present}"
)

print(
    f"Expected value fixed throughout: "
    f"{all_expected_values_identical}"
)


if not all(
    [
        all_candidate_outputs_present,
        all_matrix_shapes_correct,
        all_rankings_present,
        all_expected_values_identical,
    ]
):
    raise ValueError(
        "Explanation-sample SHAP convergence "
        "verification failed."
    )


print("\nVERIFICATION PASSED")

print(
    "Interventional TreeSHAP values were computed for all "
    "candidate explanation-sample sizes using the same frozen "
    "XGBoost model and 500-observation development background."
)

print(
    "The resulting attribution structures can now be compared "
    "quantitatively for explanation-sample convergence."
)

SHAP EXPLANATION-SAMPLE SIZE CONVERGENCE

Frozen development background : 500 observations
Candidate explanation sizes   : [3000, 7500, 15000, 30000]
SHAP formulation              : Interventional TreeSHAP
Model representation          : Verified frozen XGBoost Booster
Effective background size     : 500
TreeSHAP expected value       : 0.762257 V

----------------------------------------------------------------------
EXPLANATION N = 3,000
----------------------------------------------------------------------


100%|===================| 2997/3000 [07:35<00:00]        


Runtime                  : 455.63 s (7.59 min)
SHAP matrix              : (3000, 20)
Maximum additivity error : 0.0046564157 V
Mean additivity error    : 0.0000022654 V

Top 10 features by mean |SHAP|:


,Feature,Mean_Abs_SHAP_V,Mean_Abs_SHAP_mV,Rank
0,current,0.079195,79.195306,1
1,total_anode_stack_flow,0.009833,9.832620,2
2,temp_anode_endplate,0.006038,6.037608,3
3,cathode_pressure_diff,0.005859,5.858835,4
4,total_cathode_stack_flow,0.001701,1.701390,5
5,pressure_cathode_outlet,0.001240,1.239514,6
6,temp_anode_dewpoint_water,0.001238,1.237734,7
7,anode_temp_diff,0.001116,1.116236,8
8,cathode_dewpoint_offset,0.000895,0.894547,9
9,temp_cathode_dewpoint_water,0.000798,0.797782,10



----------------------------------------------------------------------
EXPLANATION N = 7,500
----------------------------------------------------------------------


...


Runtime                  : 4536.40 s (75.61 min)
SHAP matrix              : (30000, 20)
Maximum additivity error : 0.0050707597 V
Mean additivity error    : 0.0000009793 V

Top 10 features by mean |SHAP|:


,Feature,Mean_Abs_SHAP_V,Mean_Abs_SHAP_mV,Rank
0,current,0.079591,79.591108,1
1,total_anode_stack_flow,0.009865,9.864958,2
2,temp_anode_endplate,0.006077,6.077367,3
3,cathode_pressure_diff,0.005934,5.933798,4
4,total_cathode_stack_flow,0.001685,1.684626,5
5,temp_anode_dewpoint_water,0.001294,1.294194,6
6,pressure_cathode_outlet,0.001234,1.234395,7
7,anode_temp_diff,0.001119,1.118724,8
8,cathode_dewpoint_offset,0.000910,0.909569,9
9,temp_cathode_dewpoint_water,0.000831,0.830772,10



EXPLANATION-SAMPLE SHAP COMPUTATION SUMMARY


,Explanation_N,Background_N,Expected_Value_V,Runtime_seconds,Runtime_minutes,Max_Additivity_Error_V,Mean_Additivity_Error_V
0,3000,500,0.762257,455.633294,7.593888,0.004656,0.000002
1,7500,500,0.762257,1131.541614,18.859027,0.004656,0.000001
2,15000,500,0.762257,2135.140404,35.585673,0.005071,0.000001
3,30000,500,0.762257,4536.400836,75.606681,0.005071,0.000001



EXPLANATION-SAMPLE SHAP VERIFICATION

All candidate outputs present  : True
All SHAP matrix shapes correct : True
All ranking tables present     : True
Expected value fixed throughout: True

VERIFICATION PASSED
Interventional TreeSHAP values were computed for all candidate explanation-sample sizes using the same frozen XGBoost model and 500-observation development background.
The resulting attribution structures can now be compared quantitatively for explanation-sample convergence.


In [ ]:
### 13.7.4 Explanation-Sample Stability Quantification

The SHAP computations obtained in Section 13.7.3 are now compared quantitatively to determine whether the global attribution structure remains stable as the explanation-sample size increases.

The largest evaluated explanation sample (n = 30,000) is used as a diagnostic comparison reference rather than being treated as a ground-truth explanation. Each smaller candidate sample is compared with this reference using:

- mean absolute difference in mean |SHAP| across the 20 predictors;
- maximum feature-level difference in mean |SHAP|;
- Spearman rank correlation of the complete 20-feature importance ranking;
- overlap of the top-5 features;
- overlap of the top-10 features; and
- feature-by-feature rank differences.

The TreeSHAP expected value is not used as an explanation-sample convergence metric because the same frozen XGBoost model and the same 500-observation development background are used throughout; consequently, the expected value is fixed by design.

The candidate explanation samples were independently sampled rather than constructed as nested subsets. Therefore, differences between candidate sizes reflect both finite-sample composition and explanation-sample size. The analysis is consequently interpreted as an empirical stability assessment rather than a deterministic estimate of a pure sample-size effect.

The purpose is to identify the smallest evaluated explanation sample that preserves the substantive global attribution conclusions obtained from the largest evaluated sample. The final explanation-sample size will not be frozen until repeated-seed stability has subsequently been assessed.

In [28]:
# ============================================================
# 13.7.4 Explanation-Sample Stability Quantification
# ============================================================

from scipy.stats import spearmanr

REFERENCE_EXPLANATION_N = max(EXPLANATION_CANDIDATE_SIZES)

# 13.7.3 stores mean |SHAP| as a Series indexed by feature.
reference_importance = (
    explanation_mean_abs_shap[REFERENCE_EXPLANATION_N]
    .reindex(FROZEN_FEATURES)
    .copy()
)

# Ranking tables are DataFrames with Feature and Rank columns.
reference_ranking = (
    explanation_feature_rankings[REFERENCE_EXPLANATION_N]
    .set_index("Feature")
    .reindex(FROZEN_FEATURES)
    .copy()
)

reference_top5 = set(
    explanation_feature_rankings[REFERENCE_EXPLANATION_N]
    .sort_values("Rank")
    .head(5)["Feature"]
)

reference_top10 = set(
    explanation_feature_rankings[REFERENCE_EXPLANATION_N]
    .sort_values("Rank")
    .head(10)["Feature"]
)

stability_rows = []
feature_comparison_frames = []

print("=" * 78)
print("EXPLANATION-SAMPLE STABILITY QUANTIFICATION")
print("=" * 78)
print(f"Diagnostic reference explanation size : {REFERENCE_EXPLANATION_N:,}")
print(f"Frozen development background         : {FINAL_BACKGROUND_N:,}")
print(f"Number of predictors                  : {len(FROZEN_FEATURES)}")
print()

for explanation_n in EXPLANATION_CANDIDATE_SIZES:

    # --------------------------------------------------------
    # Retrieve candidate mean |SHAP| Series
    # --------------------------------------------------------

    candidate_importance = (
        explanation_mean_abs_shap[explanation_n]
        .reindex(FROZEN_FEATURES)
        .copy()
    )

    candidate_ranking = (
        explanation_feature_rankings[explanation_n]
        .set_index("Feature")
        .reindex(FROZEN_FEATURES)
        .copy()
    )

    # --------------------------------------------------------
    # Mean |SHAP| magnitude stability
    # --------------------------------------------------------

    # Values were stored in volts in 13.7.3.
    # Convert differences to mV for reporting.
    abs_difference_mV = (
        (candidate_importance - reference_importance).abs() * 1000
    )

    mean_abs_difference_mV = abs_difference_mV.mean()
    max_abs_difference_mV = abs_difference_mV.max()
    max_difference_feature = abs_difference_mV.idxmax()

    # --------------------------------------------------------
    # Ranking stability
    # --------------------------------------------------------

    candidate_ranks = candidate_ranking["Rank"]
    reference_ranks = reference_ranking["Rank"]

    spearman_rho, spearman_p = spearmanr(
        candidate_ranks,
        reference_ranks
    )

    candidate_top5 = set(
        explanation_feature_rankings[explanation_n]
        .sort_values("Rank")
        .head(5)["Feature"]
    )

    candidate_top10 = set(
        explanation_feature_rankings[explanation_n]
        .sort_values("Rank")
        .head(10)["Feature"]
    )

    top5_overlap = len(candidate_top5 & reference_top5)
    top10_overlap = len(candidate_top10 & reference_top10)

    # --------------------------------------------------------
    # Store summary
    # --------------------------------------------------------

    stability_rows.append({
        "Explanation_N": explanation_n,
        "Reference_N": REFERENCE_EXPLANATION_N,
        "Mean_Abs_SHAP_Difference_mV": mean_abs_difference_mV,
        "Max_Feature_Difference_mV": max_abs_difference_mV,
        "Max_Difference_Feature": max_difference_feature,
        "Spearman_Rho": spearman_rho,
        "Spearman_P_Value": spearman_p,
        "Top5_Overlap": top5_overlap,
        "Top5_Overlap_Fraction": top5_overlap / 5,
        "Top10_Overlap": top10_overlap,
        "Top10_Overlap_Fraction": top10_overlap / 10
    })

    # --------------------------------------------------------
    # Feature-by-feature comparison
    # --------------------------------------------------------

    feature_comparison = pd.DataFrame({
        "Feature": FROZEN_FEATURES,
        "Explanation_N": explanation_n,
        "Mean_Abs_SHAP_mV": candidate_importance.values * 1000,
        "Reference_Mean_Abs_SHAP_mV": reference_importance.values * 1000,
        "Absolute_Difference_mV": abs_difference_mV.values,
        "Rank": candidate_ranks.values,
        "Reference_Rank": reference_ranks.values
    })

    feature_comparison["Absolute_Rank_Difference"] = (
        feature_comparison["Rank"]
        - feature_comparison["Reference_Rank"]
    ).abs()

    feature_comparison_frames.append(feature_comparison)

    print("-" * 78)
    print(f"EXPLANATION N = {explanation_n:,}")
    print("-" * 78)
    print(
        f"Mean absolute mean |SHAP| difference : "
        f"{mean_abs_difference_mV:.6f} mV"
    )
    print(
        f"Maximum feature-level difference     : "
        f"{max_abs_difference_mV:.6f} mV"
    )
    print(
        f"Feature with maximum difference      : "
        f"{max_difference_feature}"
    )
    print(
        f"Spearman rank correlation            : "
        f"{spearman_rho:.6f}"
    )
    print(
        f"Top-5 overlap                        : "
        f"{top5_overlap}/5"
    )
    print(
        f"Top-10 overlap                       : "
        f"{top10_overlap}/10"
    )
    print()


# ============================================================
# Combined stability tables
# ============================================================

explanation_sample_stability = pd.DataFrame(stability_rows)

explanation_feature_stability = pd.concat(
    feature_comparison_frames,
    ignore_index=True
)

print("=" * 78)
print("EXPLANATION-SAMPLE STABILITY SUMMARY")
print("=" * 78)

display(
    explanation_sample_stability[
        [
            "Explanation_N",
            "Reference_N",
            "Mean_Abs_SHAP_Difference_mV",
            "Max_Feature_Difference_mV",
            "Max_Difference_Feature",
            "Spearman_Rho",
            "Top5_Overlap",
            "Top10_Overlap"
        ]
    ]
)

# ============================================================
# Feature-by-feature rank comparison
# ============================================================

rank_comparison = pd.DataFrame({
    "Feature": FROZEN_FEATURES
})

for explanation_n in EXPLANATION_CANDIDATE_SIZES:

    ranks = (
        explanation_feature_rankings[explanation_n]
        .set_index("Feature")
        .reindex(FROZEN_FEATURES)["Rank"]
    )

    rank_comparison[f"Rank_n{explanation_n}"] = ranks.values

rank_columns = [
    f"Rank_n{n}"
    for n in EXPLANATION_CANDIDATE_SIZES
]

rank_comparison["Maximum_Rank_Change"] = (
    rank_comparison[rank_columns].max(axis=1)
    - rank_comparison[rank_columns].min(axis=1)
)

rank_comparison = (
    rank_comparison
    .sort_values(f"Rank_n{REFERENCE_EXPLANATION_N}")
    .reset_index(drop=True)
)

print()
print("=" * 78)
print("FEATURE-BY-FEATURE RANK STABILITY")
print("=" * 78)

display(rank_comparison)

# ============================================================
# Integrity verification
# ============================================================

all_sizes_present = (
    set(explanation_sample_stability["Explanation_N"])
    == set(EXPLANATION_CANDIDATE_SIZES)
)

reference_row = explanation_sample_stability.loc[
    explanation_sample_stability["Explanation_N"]
    == REFERENCE_EXPLANATION_N
].iloc[0]

reference_is_zero = (
    np.isclose(
        reference_row["Mean_Abs_SHAP_Difference_mV"], 0.0
    )
    and np.isclose(
        reference_row["Max_Feature_Difference_mV"], 0.0
    )
    and np.isclose(
        reference_row["Spearman_Rho"], 1.0
    )
    and reference_row["Top5_Overlap"] == 5
    and reference_row["Top10_Overlap"] == 10
)

all_feature_comparisons_present = (
    len(explanation_feature_stability)
    == len(EXPLANATION_CANDIDATE_SIZES) * len(FROZEN_FEATURES)
)

no_missing_values = (
    not explanation_sample_stability.isna().any().any()
    and not explanation_feature_stability.isna().any().any()
    and not rank_comparison.isna().any().any()
)

print()
print("=" * 78)
print("STABILITY ANALYSIS VERIFICATION")
print("=" * 78)
print(f"All candidate sizes present       : {all_sizes_present}")
print(f"Reference self-comparison correct : {reference_is_zero}")
print(
    f"All feature comparisons present   : "
    f"{all_feature_comparisons_present}"
)
print(f"No missing stability values       : {no_missing_values}")

if not (
    all_sizes_present
    and reference_is_zero
    and all_feature_comparisons_present
    and no_missing_values
):
    raise ValueError(
        "Explanation-sample stability verification failed."
    )

print()
print("VERIFICATION PASSED")
print(
    "Explanation-sample stability has been quantified against "
    "the largest evaluated diagnostic reference."
)
print(
    "The results are ready for interpretation before "
    "repeated-seed robustness testing."
)

EXPLANATION-SAMPLE STABILITY QUANTIFICATION
Diagnostic reference explanation size : 30,000
Frozen development background         : 500
Number of predictors                  : 20

------------------------------------------------------------------------------
EXPLANATION N = 3,000
------------------------------------------------------------------------------
Mean absolute mean |SHAP| difference : 0.036135 mV
Maximum feature-level difference     : 0.395803 mV
Feature with maximum difference      : current
Spearman rank correlation            : 0.998496
Top-5 overlap                        : 5/5
Top-10 overlap                       : 10/10

------------------------------------------------------------------------------
EXPLANATION N = 7,500
------------------------------------------------------------------------------
Mean absolute mean |SHAP| difference : 0.029389 mV
Maximum feature-level difference     : 0.352407 mV
Feature with maximum difference      : current
Spearman rank correlation 

,Explanation_N,Reference_N,Mean_Abs_SHAP_Difference_mV,Max_Feature_Difference_mV,Max_Difference_Feature,Spearman_Rho,Top5_Overlap,Top10_Overlap
0,3000,30000,0.036135,0.395803,current,0.998496,5,10
1,7500,30000,0.029389,0.352407,current,1.000000,5,10
2,15000,30000,0.020666,0.295057,current,1.000000,5,10
3,30000,30000,0.000000,0.000000,current,1.000000,5,10



FEATURE-BY-FEATURE RANK STABILITY


,Feature,Rank_n3000,Rank_n7500,Rank_n15000,Rank_n30000,Maximum_Rank_Change
0,current,1,1,1,1,0
1,total_anode_stack_flow,2,2,2,2,0
2,temp_anode_endplate,3,3,3,3,0
3,cathode_pressure_diff,4,4,4,4,0
4,total_cathode_stack_flow,5,5,5,5,0
5,temp_anode_dewpoint_water,7,6,6,6,1
6,pressure_cathode_outlet,6,7,7,7,1
7,anode_temp_diff,8,8,8,8,0
8,cathode_dewpoint_offset,9,9,9,9,0
9,temp_cathode_dewpoint_water,10,10,10,10,0



STABILITY ANALYSIS VERIFICATION
All candidate sizes present       : True
Reference self-comparison correct : True
All feature comparisons present   : True
No missing stability values       : True

VERIFICATION PASSED
Explanation-sample stability has been quantified against the largest evaluated diagnostic reference.
The results are ready for interpretation before repeated-seed robustness testing.


In [ ]:
### 13.7.5 Repeated-Seed Stability of the Candidate Explanation Sample

The explanation-sample convergence analysis identified n = 7,500 as the smallest evaluated candidate that reproduced the complete 20-feature ranking of the n = 30,000 diagnostic reference, while also preserving identical top-5 and top-10 feature membership.

However, the candidate samples used in the convergence analysis were generated using a single random seed. A repeated-seed robustness assessment is therefore performed before freezing the final explanation-sample size.

Three reproducible random seeds are evaluated: 42, 123 and 2026. For each seed, an equal-stage explanation sample of 7,500 observations is constructed from the independent later-stage holdout, corresponding to 2,500 observations from each of the 900 h, 950 h and 1000 h durability stages. Random sampling occurs within each stage without replacement, thereby preserving the naturally occurring within-stage operational distribution rather than artificially balancing Current or other predictor values.

The frozen XGBoost model, 500-observation development background, interventional TreeSHAP formulation and predictor order remain unchanged. Consequently, only the composition of the explanation sample varies between seeds.

The SHAP values already calculated for seed 42 are reused rather than recomputed. New TreeSHAP calculations are required only for seeds 123 and 2026.

Repeated-seed stability is evaluated using:

- mean |SHAP| magnitude differences;
- maximum feature-level attribution differences;
- Spearman correlation of the complete 20-feature rankings;
- top-5 feature overlap;
- top-10 feature overlap; and
- feature-by-feature rank variation across seeds.

This analysis evaluates whether the substantive global attribution conclusions obtained at n = 7,500 are robust to random explanation-sample composition. The three-seed design is a pragmatic robustness assessment rather than a universal statistical requirement for SHAP analysis.

In [31]:
# ============================================================
# 13.7.5 Repeated-Seed Stability of Candidate Explanation Sample
# ============================================================

import time
from scipy.stats import spearmanr

CANDIDATE_FINAL_EXPLANATION_N = 7500
REPEATED_SEEDS = [42, 123, 2026]
REFERENCE_SEED = 42

# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

seed_shap_values = {}
seed_mean_abs_shap = {}
seed_feature_rankings = {}
seed_explanation_samples = {}
seed_runtime_rows = []

# ------------------------------------------------------------
# Reuse the already-computed seed-42 result from Section 13.7.3
# ------------------------------------------------------------

seed42_data = candidate_explanation_samples[
    CANDIDATE_FINAL_EXPLANATION_N
].copy()

seed42_shap_matrix = np.asarray(
    explanation_shap_values[
        CANDIDATE_FINAL_EXPLANATION_N
    ]
)

seed42_mean_abs = pd.Series(
    np.abs(seed42_shap_matrix).mean(axis=0),
    index=FROZEN_FEATURES,
    name="Mean_Abs_SHAP_V"
)

seed42_ranking = (
    pd.DataFrame({
        "Feature": FROZEN_FEATURES,
        "Mean_Abs_SHAP_V": seed42_mean_abs.values
    })
    .sort_values(
        "Mean_Abs_SHAP_V",
        ascending=False
    )
    .reset_index(drop=True)
)

seed42_ranking["Mean_Abs_SHAP_mV"] = (
    seed42_ranking["Mean_Abs_SHAP_V"] * 1000
)

seed42_ranking["Rank"] = (
    np.arange(1, len(seed42_ranking) + 1)
)

seed_explanation_samples[42] = seed42_data
seed_shap_values[42] = seed42_shap_matrix
seed_mean_abs_shap[42] = seed42_mean_abs
seed_feature_rankings[42] = seed42_ranking

seed_runtime_rows.append({
    "Seed": 42,
    "Explanation_N": CANDIDATE_FINAL_EXPLANATION_N,
    "Runtime_seconds": 0.0,
    "Runtime_minutes": 0.0,
    "Computation": "Reused from Section 13.7.3"
})

print("=" * 78)
print("REPEATED-SEED SHAP STABILITY")
print("=" * 78)
print(
    f"Candidate explanation size : "
    f"{CANDIDATE_FINAL_EXPLANATION_N:,}"
)
print(f"Seeds                      : {REPEATED_SEEDS}")
print(f"Reference seed             : {REFERENCE_SEED}")
print(f"Frozen background          : {FINAL_BACKGROUND_N:,}")
print("SHAP formulation           : Interventional TreeSHAP")
print()

print("-" * 78)
print("SEED = 42")
print("-" * 78)
print("Existing SHAP result reused from Section 13.7.3.")
print(
    "No additional TreeSHAP computation was performed "
    "for seed 42."
)
print()


# ============================================================
# Compute only the two new seeds
# ============================================================

for seed in [s for s in REPEATED_SEEDS if s != REFERENCE_SEED]:

    print("-" * 78)
    print(f"SEED = {seed}")
    print("-" * 78)

    # --------------------------------------------------------
    # Equal-stage holdout sample
    # --------------------------------------------------------

    seed_data, seed_allocation = sample_stage_balanced(
        data=holdout_data,
        stages=HOLDOUT_STAGES,
        total_n=CANDIDATE_FINAL_EXPLANATION_N,
        stage_column=STAGE_COLUMN,
        random_seed=seed
    )

    X_seed = seed_data[FROZEN_FEATURES].copy()

    # Verify equal-stage allocation
    stage_counts = (
        seed_data[STAGE_COLUMN]
        .value_counts()
        .sort_index()
    )

    expected_per_stage = (
        CANDIDATE_FINAL_EXPLANATION_N
        // len(HOLDOUT_STAGES)
    )

    equal_stage_allocation = bool(
        (stage_counts == expected_per_stage).all()
    )

    if not equal_stage_allocation:
        raise ValueError(
            f"Seed {seed}: equal-stage allocation failed."
        )

    if list(X_seed.columns) != list(FROZEN_FEATURES):
        raise ValueError(
            f"Seed {seed}: frozen predictor order changed."
        )

    if X_seed.isna().any().any():
        raise ValueError(
            f"Seed {seed}: missing predictor values detected."
        )

    # --------------------------------------------------------
    # Frozen-model predictions for independent additivity check
    # --------------------------------------------------------

    seed_predictions = final_xgb_model.predict(X_seed)

    # --------------------------------------------------------
    # Interventional TreeSHAP
    # --------------------------------------------------------

    start_time = time.perf_counter()

    seed_shap_explanation = primary_interventional_explainer(
        X_seed,
        check_additivity=True
    )

    runtime_seconds = time.perf_counter() - start_time
    runtime_minutes = runtime_seconds / 60

    shap_matrix = np.asarray(seed_shap_explanation.values)

    expected_shape = (
        CANDIDATE_FINAL_EXPLANATION_N,
        len(FROZEN_FEATURES)
    )

    if shap_matrix.shape != expected_shape:
        raise ValueError(
            f"Seed {seed}: unexpected SHAP matrix shape "
            f"{shap_matrix.shape}; expected {expected_shape}."
        )

    # --------------------------------------------------------
    # Independent additivity reconstruction
    # --------------------------------------------------------

    reconstructed_predictions = (
        PRIMARY_EXPECTED_VALUE
        + shap_matrix.sum(axis=1)
    )

    additivity_error = np.abs(
        reconstructed_predictions
        - seed_predictions
    )

    max_additivity_error = additivity_error.max()
    mean_additivity_error = additivity_error.mean()

    # --------------------------------------------------------
    # Mean |SHAP| and ranking
    # --------------------------------------------------------

    mean_abs = pd.Series(
        np.abs(shap_matrix).mean(axis=0),
        index=FROZEN_FEATURES,
        name="Mean_Abs_SHAP_V"
    )

    ranking = (
        pd.DataFrame({
            "Feature": FROZEN_FEATURES,
            "Mean_Abs_SHAP_V": mean_abs.values
        })
        .sort_values(
            "Mean_Abs_SHAP_V",
            ascending=False
        )
        .reset_index(drop=True)
    )

    ranking["Mean_Abs_SHAP_mV"] = (
        ranking["Mean_Abs_SHAP_V"] * 1000
    )

    ranking["Rank"] = (
        np.arange(1, len(ranking) + 1)
    )

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    seed_explanation_samples[seed] = seed_data
    seed_shap_values[seed] = shap_matrix
    seed_mean_abs_shap[seed] = mean_abs
    seed_feature_rankings[seed] = ranking

    seed_runtime_rows.append({
        "Seed": seed,
        "Explanation_N": CANDIDATE_FINAL_EXPLANATION_N,
        "Runtime_seconds": runtime_seconds,
        "Runtime_minutes": runtime_minutes,
        "Computation": "New TreeSHAP computation"
    })

    print(f"Stage allocation           : {stage_counts.to_dict()}")
    print(
        f"Runtime                    : "
        f"{runtime_seconds:.2f} s "
        f"({runtime_minutes:.2f} min)"
    )
    print(f"SHAP matrix                : {shap_matrix.shape}")
    print(
        f"Maximum additivity error   : "
        f"{max_additivity_error:.10f} V"
    )
    print(
        f"Mean additivity error      : "
        f"{mean_additivity_error:.10f} V"
    )
    print()
    print("Top 10 features by mean |SHAP|:")
    display(
        ranking[
            [
                "Feature",
                "Mean_Abs_SHAP_V",
                "Mean_Abs_SHAP_mV",
                "Rank"
            ]
        ].head(10)
    )


# ============================================================
# Pairwise stability against seed 42
# ============================================================

reference_importance = (
    seed_mean_abs_shap[REFERENCE_SEED]
    .reindex(FROZEN_FEATURES)
)

reference_ranking = (
    seed_feature_rankings[REFERENCE_SEED]
    .set_index("Feature")
    .reindex(FROZEN_FEATURES)
)

reference_top5 = set(
    seed_feature_rankings[REFERENCE_SEED]
    .sort_values("Rank")
    .head(5)["Feature"]
)

reference_top10 = set(
    seed_feature_rankings[REFERENCE_SEED]
    .sort_values("Rank")
    .head(10)["Feature"]
)

seed_stability_rows = []

for seed in REPEATED_SEEDS:

    candidate_importance = (
        seed_mean_abs_shap[seed]
        .reindex(FROZEN_FEATURES)
    )

    candidate_ranking = (
        seed_feature_rankings[seed]
        .set_index("Feature")
        .reindex(FROZEN_FEATURES)
    )

    # Attribution magnitude difference in mV
    abs_difference_mV = (
        (candidate_importance - reference_importance)
        .abs()
        * 1000
    )

    mean_difference_mV = abs_difference_mV.mean()
    max_difference_mV = abs_difference_mV.max()
    max_difference_feature = abs_difference_mV.idxmax()

    # Complete-ranking stability
    spearman_rho, spearman_p = spearmanr(
        candidate_ranking["Rank"],
        reference_ranking["Rank"]
    )

    candidate_top5 = set(
        seed_feature_rankings[seed]
        .sort_values("Rank")
        .head(5)["Feature"]
    )

    candidate_top10 = set(
        seed_feature_rankings[seed]
        .sort_values("Rank")
        .head(10)["Feature"]
    )

    top5_overlap = len(
        candidate_top5 & reference_top5
    )

    top10_overlap = len(
        candidate_top10 & reference_top10
    )

    seed_stability_rows.append({
        "Seed": seed,
        "Reference_Seed": REFERENCE_SEED,
        "Mean_Abs_SHAP_Difference_mV": mean_difference_mV,
        "Max_Feature_Difference_mV": max_difference_mV,
        "Max_Difference_Feature": max_difference_feature,
        "Spearman_Rho": spearman_rho,
        "Spearman_P_Value": spearman_p,
        "Top5_Overlap": top5_overlap,
        "Top10_Overlap": top10_overlap
    })


seed_stability_summary = pd.DataFrame(
    seed_stability_rows
)

seed_runtime_summary = pd.DataFrame(
    seed_runtime_rows
)


# ============================================================
# Feature-by-feature rank stability across all three seeds
# ============================================================

seed_rank_comparison = pd.DataFrame({
    "Feature": FROZEN_FEATURES
})

for seed in REPEATED_SEEDS:

    ranks = (
        seed_feature_rankings[seed]
        .set_index("Feature")
        .reindex(FROZEN_FEATURES)["Rank"]
    )

    seed_rank_comparison[
        f"Rank_seed_{seed}"
    ] = ranks.values


seed_rank_columns = [
    f"Rank_seed_{seed}"
    for seed in REPEATED_SEEDS
]

seed_rank_comparison["Minimum_Rank"] = (
    seed_rank_comparison[
        seed_rank_columns
    ].min(axis=1)
)

seed_rank_comparison["Maximum_Rank"] = (
    seed_rank_comparison[
        seed_rank_columns
    ].max(axis=1)
)

seed_rank_comparison["Maximum_Rank_Change"] = (
    seed_rank_comparison["Maximum_Rank"]
    - seed_rank_comparison["Minimum_Rank"]
)

seed_rank_comparison = (
    seed_rank_comparison
    .sort_values(f"Rank_seed_{REFERENCE_SEED}")
    .reset_index(drop=True)
)


# ============================================================
# Display summaries
# ============================================================

print()
print("=" * 78)
print("REPEATED-SEED STABILITY SUMMARY")
print("=" * 78)

display(
    seed_stability_summary[
        [
            "Seed",
            "Reference_Seed",
            "Mean_Abs_SHAP_Difference_mV",
            "Max_Feature_Difference_mV",
            "Max_Difference_Feature",
            "Spearman_Rho",
            "Top5_Overlap",
            "Top10_Overlap"
        ]
    ]
)

print()
print("=" * 78)
print("FEATURE-BY-FEATURE RANK STABILITY ACROSS SEEDS")
print("=" * 78)

display(seed_rank_comparison)

print()
print("=" * 78)
print("REPEATED-SEED COMPUTATION SUMMARY")
print("=" * 78)

display(seed_runtime_summary)


# ============================================================
# Integrity verification
# ============================================================

all_seeds_present = (
    set(seed_shap_values.keys())
    == set(REPEATED_SEEDS)
)

all_sample_sizes_correct = all(
    len(seed_explanation_samples[seed])
    == CANDIDATE_FINAL_EXPLANATION_N
    for seed in REPEATED_SEEDS
)

all_shapes_correct = all(
    seed_shap_values[seed].shape
    == (
        CANDIDATE_FINAL_EXPLANATION_N,
        len(FROZEN_FEATURES)
    )
    for seed in REPEATED_SEEDS
)

all_rankings_present = all(
    len(seed_feature_rankings[seed])
    == len(FROZEN_FEATURES)
    for seed in REPEATED_SEEDS
)

all_stage_allocations_equal = all(
    (
        seed_explanation_samples[seed][STAGE_COLUMN]
        .value_counts()
        .reindex(HOLDOUT_STAGES)
        == (
            CANDIDATE_FINAL_EXPLANATION_N
            // len(HOLDOUT_STAGES)
        )
    ).all()
    for seed in REPEATED_SEEDS
)

no_missing_seed_outputs = (
    not seed_stability_summary.isna().any().any()
    and not seed_rank_comparison.isna().any().any()
)

print()
print("=" * 78)
print("REPEATED-SEED VERIFICATION")
print("=" * 78)
print(f"All three seeds present          : {all_seeds_present}")
print(f"All sample sizes correct         : {all_sample_sizes_correct}")
print(f"All SHAP matrix shapes correct   : {all_shapes_correct}")
print(f"All ranking tables present       : {all_rankings_present}")
print(f"Equal stage allocation preserved : {all_stage_allocations_equal}")
print(f"No missing stability values      : {no_missing_seed_outputs}")

if not (
    all_seeds_present
    and all_sample_sizes_correct
    and all_shapes_correct
    and all_rankings_present
    and all_stage_allocations_equal
    and no_missing_seed_outputs
):
    raise ValueError(
        "Repeated-seed SHAP stability verification failed."
    )

print()
print("VERIFICATION PASSED")
print(
    "The n = 7,500 candidate explanation sample has been "
    "evaluated across three reproducible random seeds."
)
print(
    "The resulting attribution magnitudes and feature rankings "
    "are ready for final stability assessment."
)

REPEATED-SEED SHAP STABILITY
Candidate explanation size : 7,500
Seeds                      : [42, 123, 2026]
Reference seed             : 42
Frozen background          : 500
SHAP formulation           : Interventional TreeSHAP

------------------------------------------------------------------------------
SEED = 42
------------------------------------------------------------------------------
Existing SHAP result reused from Section 13.7.3.
No additional TreeSHAP computation was performed for seed 42.

------------------------------------------------------------------------------
SEED = 123
------------------------------------------------------------------------------


100%|===================| 7499/7500 [09:05<00:00]        

Stage allocation           : {900: 2500, 950: 2500, 1000: 2500}
Runtime                    : 544.95 s (9.08 min)
SHAP matrix                : (7500, 20)
Maximum additivity error   : 0.0014624879 V
Mean additivity error      : 0.0000007178 V

Top 10 features by mean |SHAP|:


,Feature,Mean_Abs_SHAP_V,Mean_Abs_SHAP_mV,Rank
0,current,0.079957,79.957043,1
1,total_anode_stack_flow,0.009845,9.844947,2
2,temp_anode_endplate,0.006085,6.084812,3
3,cathode_pressure_diff,0.005911,5.910567,4
4,total_cathode_stack_flow,0.001675,1.675027,5
5,temp_anode_dewpoint_water,0.001267,1.267446,6
6,pressure_cathode_outlet,0.001262,1.262210,7
7,anode_temp_diff,0.001111,1.110766,8
8,cathode_dewpoint_offset,0.000907,0.907165,9
9,temp_cathode_dewpoint_water,0.000807,0.806690,10


------------------------------------------------------------------------------
SEED = 2026
------------------------------------------------------------------------------


...


FEATURE-BY-FEATURE RANK STABILITY ACROSS SEEDS


,Feature,Rank_seed_42,Rank_seed_123,Rank_seed_2026,Minimum_Rank,Maximum_Rank,Maximum_Rank_Change
0,current,1,1,1,1,1,0
1,total_anode_stack_flow,2,2,2,2,2,0
2,temp_anode_endplate,3,3,3,3,3,0
3,cathode_pressure_diff,4,4,4,4,4,0
4,total_cathode_stack_flow,5,5,5,5,5,0
5,temp_anode_dewpoint_water,6,6,6,6,6,0
6,pressure_cathode_outlet,7,7,7,7,7,0
7,anode_temp_diff,8,8,8,8,8,0
8,cathode_dewpoint_offset,9,9,9,9,9,0
9,temp_cathode_dewpoint_water,10,10,10,10,10,0



REPEATED-SEED COMPUTATION SUMMARY


,Seed,Explanation_N,Runtime_seconds,Runtime_minutes,Computation
0,42,7500,0.000000,0.000000,Reused from Section 13.7.3
1,123,7500,544.951194,9.082520,New TreeSHAP computation
2,2026,7500,567.436029,9.457267,New TreeSHAP computation



REPEATED-SEED VERIFICATION
All three seeds present          : True
All sample sizes correct         : True
All SHAP matrix shapes correct   : True
All ranking tables present       : True
Equal stage allocation preserved : True
No missing stability values      : True

VERIFICATION PASSED
The n = 7,500 candidate explanation sample has been evaluated across three reproducible random seeds.
The resulting attribution magnitudes and feature rankings are ready for final stability assessment.


In [ ]:
### 13.7.6 Final Explanation-Sample Decision

The primary SHAP explanation sample is frozen at **n = 7,500 observations**, comprising 2,500 randomly sampled observations from each of the 900 h, 950 h and 1000 h later-stage holdout stages.

This decision was based on two complementary empirical assessments.

First, the explanation-sample convergence analysis compared candidate samples of 3,000, 7,500, 15,000 and 30,000 observations while holding the frozen XGBoost model, 500-observation development background and interventional TreeSHAP formulation constant. Relative to the largest evaluated diagnostic reference (n = 30,000), the n = 7,500 sample reproduced the complete 20-feature importance ranking (Spearman ρ = 1.000), with identical top-5 and top-10 feature membership. The mean absolute difference in feature-level mean |SHAP| was 0.0294 mV.

Second, repeated-seed analysis evaluated n = 7,500 using random seeds 42, 123 and 2026. Relative to seed 42, the alternative samples produced Spearman rank correlations of 0.9955 and 0.9970, while preserving identical top-5 and top-10 feature membership. The first 12 feature ranks were identical across all three samples, with only minor rank variation among lower-importance predictors.

These results indicate that n = 7,500 provides sufficiently stable global attribution structure for the interpretive objectives of this study. Increasing the explanation sample to 15,000 or 30,000 produced no substantive improvement in the principal feature-ranking conclusions while requiring substantially greater computation.

The final primary explanation sample therefore uses **seed 42**, with the seed-123 and seed-2026 analyses retained as robustness evidence. The three samples are not pooled. The final sample remains stage-balanced by analytical design, while the naturally occurring within-stage operational distribution is preserved.

This decision concerns the stability and computational efficiency of the SHAP explanation sample only. It does not imply that n = 7,500 is a universally optimal SHAP sample size or that the resulting feature attributions represent causal electrochemical effects.

In [32]:
# ============================================================
# 13.7.6 Freeze Final Primary Explanation Sample
# ============================================================

FINAL_EXPLANATION_N = 7500
FINAL_EXPLANATION_SEED = 42

# ------------------------------------------------------------
# Freeze the final stage-balanced explanation data
# ------------------------------------------------------------

final_explanation_data = (
    seed_explanation_samples[FINAL_EXPLANATION_SEED]
    .copy()
)

X_final_explanation = (
    final_explanation_data[FROZEN_FEATURES]
    .copy()
)

y_final_explanation = (
    final_explanation_data[TARGET]
    .copy()
)

# Reuse the already-computed primary interventional SHAP matrix.
final_shap_values = np.asarray(
    seed_shap_values[FINAL_EXPLANATION_SEED]
).copy()

final_mean_abs_shap = (
    seed_mean_abs_shap[FINAL_EXPLANATION_SEED]
    .copy()
)

final_feature_ranking = (
    seed_feature_rankings[FINAL_EXPLANATION_SEED]
    .copy()
)

FINAL_SHAP_EXPECTED_VALUE = float(
    PRIMARY_EXPECTED_VALUE
)

# ------------------------------------------------------------
# Final stage allocation
# ------------------------------------------------------------

final_explanation_allocation = (
    final_explanation_data[STAGE_COLUMN]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# Freeze corresponding model predictions
# ------------------------------------------------------------

final_explanation_predictions = (
    final_xgb_model.predict(X_final_explanation)
)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

correct_sample_size = (
    len(final_explanation_data)
    == FINAL_EXPLANATION_N
)

correct_shap_shape = (
    final_shap_values.shape
    == (
        FINAL_EXPLANATION_N,
        len(FROZEN_FEATURES)
    )
)

correct_feature_order = (
    list(X_final_explanation.columns)
    == list(FROZEN_FEATURES)
)

correct_stage_set = (
    set(final_explanation_allocation.index)
    == set(HOLDOUT_STAGES)
)

expected_per_stage = (
    FINAL_EXPLANATION_N
    // len(HOLDOUT_STAGES)
)

equal_stage_allocation = bool(
    (
        final_explanation_allocation
        == expected_per_stage
    ).all()
)

no_missing_predictors = (
    not X_final_explanation
    .isna()
    .any()
    .any()
)

prediction_length_correct = (
    len(final_explanation_predictions)
    == FINAL_EXPLANATION_N
)

ranking_complete = (
    len(final_feature_ranking)
    == len(FROZEN_FEATURES)
)

# ------------------------------------------------------------
# Display frozen design
# ------------------------------------------------------------

print("=" * 78)
print("FINAL PRIMARY SHAP EXPLANATION SAMPLE")
print("=" * 78)

print(
    f"Final explanation size      : "
    f"{FINAL_EXPLANATION_N:,}"
)
print(
    f"Final random seed           : "
    f"{FINAL_EXPLANATION_SEED}"
)
print(
    f"Frozen background size      : "
    f"{FINAL_BACKGROUND_N:,}"
)
print(
    f"SHAP formulation            : "
    f"Interventional TreeSHAP"
)
print(
    f"Expected value              : "
    f"{FINAL_SHAP_EXPECTED_VALUE:.6f} V"
)
print(
    f"SHAP matrix shape           : "
    f"{final_shap_values.shape}"
)

print()
print("Stage allocation:")
display(
    final_explanation_allocation
    .rename("Observations")
    .to_frame()
)

print()
print("Final global feature ranking:")
display(
    final_feature_ranking[
        [
            "Feature",
            "Mean_Abs_SHAP_V",
            "Mean_Abs_SHAP_mV",
            "Rank"
        ]
    ]
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print()
print("=" * 78)
print("FINAL EXPLANATION-SAMPLE VERIFICATION")
print("=" * 78)

print(
    f"Correct sample size          : "
    f"{correct_sample_size}"
)
print(
    f"Correct SHAP matrix shape    : "
    f"{correct_shap_shape}"
)
print(
    f"Frozen feature order retained: "
    f"{correct_feature_order}"
)
print(
    f"Correct holdout stages       : "
    f"{correct_stage_set}"
)
print(
    f"Equal stage allocation       : "
    f"{equal_stage_allocation}"
)
print(
    f"No missing predictor values  : "
    f"{no_missing_predictors}"
)
print(
    f"Prediction length correct    : "
    f"{prediction_length_correct}"
)
print(
    f"Complete feature ranking     : "
    f"{ranking_complete}"
)

all_checks_passed = all([
    correct_sample_size,
    correct_shap_shape,
    correct_feature_order,
    correct_stage_set,
    equal_stage_allocation,
    no_missing_predictors,
    prediction_length_correct,
    ranking_complete
])

if not all_checks_passed:
    raise ValueError(
        "Final SHAP explanation-sample verification failed."
    )

print()
print("VERIFICATION PASSED")
print(
    "The final primary SHAP explanation sample is now frozen."
)
print(
    "No further explanation-sample selection will be performed."
)

FINAL PRIMARY SHAP EXPLANATION SAMPLE
Final explanation size      : 7,500
Final random seed           : 42
Frozen background size      : 500
SHAP formulation            : Interventional TreeSHAP
Expected value              : 0.762257 V
SHAP matrix shape           : (7500, 20)

Stage allocation:


,Observations
operating_hour,
900,2500
950,2500
1000,2500



Final global feature ranking:


,Feature,Mean_Abs_SHAP_V,Mean_Abs_SHAP_mV,Rank
0,current,0.079239,79.238701,1
1,total_anode_stack_flow,0.009776,9.775657,2
2,temp_anode_endplate,0.006045,6.044815,3
3,cathode_pressure_diff,0.005904,5.903888,4
4,total_cathode_stack_flow,0.001697,1.696977,5
5,temp_anode_dewpoint_water,0.001299,1.299153,6
6,pressure_cathode_outlet,0.001230,1.229881,7
7,anode_temp_diff,0.001119,1.118551,8
8,cathode_dewpoint_offset,0.000905,0.904716,9
9,temp_cathode_dewpoint_water,0.000832,0.832284,10



FINAL EXPLANATION-SAMPLE VERIFICATION
Correct sample size          : True
Correct SHAP matrix shape    : True
Frozen feature order retained: True
Correct holdout stages       : True
Equal stage allocation       : True
No missing predictor values  : True
Prediction length correct    : True
Complete feature ranking     : True

VERIFICATION PASSED
The final primary SHAP explanation sample is now frozen.
No further explanation-sample selection will be performed.


In [ ]:
### 13.8 Final SHAP Additivity and Consistency Verification

Before interpreting the final SHAP attributions, the additive consistency of the frozen explanation is assessed.

For each observation, TreeSHAP represents the model prediction as the sum of the explainer expected value and the SHAP contributions assigned to the 20 predictors. Therefore, the XGBoost prediction can be reconstructed as:

Predicted voltage ≈ SHAP expected value + sum of the 20 feature SHAP values

The expected value is the model-reference output defined by the frozen 500-observation development background. It should not be interpreted as beginning-of-life voltage, healthy-cell voltage or a physical degradation reference.

The reconstructed SHAP predictions are compared with predictions obtained directly from the frozen XGBoost model for the final 7,500-observation explanation sample. The analysis reports the mean, median, 95th, 99th and maximum absolute reconstruction errors and identifies the observation associated with the largest discrepancy.

The purpose of this assessment is numerical verification of the explanation representation rather than predictive model evaluation. Predictive performance was evaluated previously on the complete independent later-stage holdout. Small numerical discrepancies may arise from implementation and floating-point behaviour and should not be interpreted as model prediction error or physical degradation.

Following this verification, the frozen SHAP values can be used for global, directional, stage-wise and local interpretation without further modification of the model, background or explanation sample.

In [33]:
# ============================================================
# 13.8 Final SHAP Additivity and Consistency Verification
# ============================================================

# ------------------------------------------------------------
# Reconstruct frozen XGBoost predictions from SHAP
# ------------------------------------------------------------

shap_reconstructed_predictions = (
    FINAL_SHAP_EXPECTED_VALUE
    + final_shap_values.sum(axis=1)
)

# Direct predictions were frozen in Section 13.7.6.
direct_predictions = np.asarray(
    final_explanation_predictions
)

# ------------------------------------------------------------
# Absolute and signed reconstruction differences
# ------------------------------------------------------------

shap_reconstruction_difference = (
    shap_reconstructed_predictions
    - direct_predictions
)

shap_absolute_reconstruction_error = np.abs(
    shap_reconstruction_difference
)

# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

additivity_summary = pd.DataFrame({
    "Metric": [
        "Mean absolute reconstruction error",
        "Median absolute reconstruction error",
        "95th percentile absolute reconstruction error",
        "99th percentile absolute reconstruction error",
        "Maximum absolute reconstruction error"
    ],
    "Error_V": [
        shap_absolute_reconstruction_error.mean(),
        np.median(shap_absolute_reconstruction_error),
        np.quantile(shap_absolute_reconstruction_error, 0.95),
        np.quantile(shap_absolute_reconstruction_error, 0.99),
        shap_absolute_reconstruction_error.max()
    ]
})

additivity_summary["Error_mV"] = (
    additivity_summary["Error_V"] * 1000
)

# ------------------------------------------------------------
# Locate maximum-error observation
# ------------------------------------------------------------

max_error_position = int(
    np.argmax(shap_absolute_reconstruction_error)
)

max_error_original_index = (
    final_explanation_data.index[max_error_position]
)

max_error_stage = (
    final_explanation_data.iloc[max_error_position][STAGE_COLUMN]
)

max_error_direct_prediction = (
    direct_predictions[max_error_position]
)

max_error_shap_prediction = (
    shap_reconstructed_predictions[max_error_position]
)

max_error_signed_difference = (
    shap_reconstruction_difference[max_error_position]
)

max_error_absolute_difference = (
    shap_absolute_reconstruction_error[max_error_position]
)

# ------------------------------------------------------------
# Additional consistency checks
# ------------------------------------------------------------

prediction_correlation = np.corrcoef(
    direct_predictions,
    shap_reconstructed_predictions
)[0, 1]

mean_direct_prediction = direct_predictions.mean()

mean_reconstructed_prediction = (
    shap_reconstructed_predictions.mean()
)

mean_prediction_difference = (
    mean_reconstructed_prediction
    - mean_direct_prediction
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 78)
print("FINAL SHAP ADDITIVITY AND CONSISTENCY VERIFICATION")
print("=" * 78)

print(
    f"Explanation observations       : "
    f"{FINAL_EXPLANATION_N:,}"
)
print(
    f"Predictors                     : "
    f"{len(FROZEN_FEATURES)}"
)
print(
    f"SHAP expected value            : "
    f"{FINAL_SHAP_EXPECTED_VALUE:.9f} V"
)
print(
    f"Mean direct prediction         : "
    f"{mean_direct_prediction:.9f} V"
)
print(
    f"Mean SHAP reconstruction       : "
    f"{mean_reconstructed_prediction:.9f} V"
)
print(
    f"Mean signed difference         : "
    f"{mean_prediction_difference:.10f} V"
)
print(
    f"Prediction correlation         : "
    f"{prediction_correlation:.12f}"
)

print()
print("Absolute reconstruction-error summary:")
display(additivity_summary)

print()
print("=" * 78)
print("MAXIMUM RECONSTRUCTION-ERROR OBSERVATION")
print("=" * 78)

print(
    f"Sample position                : "
    f"{max_error_position}"
)
print(
    f"Original dataframe index       : "
    f"{max_error_original_index}"
)
print(
    f"Operating stage                : "
    f"{max_error_stage}"
)
print(
    f"Direct XGBoost prediction      : "
    f"{max_error_direct_prediction:.9f} V"
)
print(
    f"SHAP reconstructed prediction  : "
    f"{max_error_shap_prediction:.9f} V"
)
print(
    f"Signed difference              : "
    f"{max_error_signed_difference:.10f} V"
)
print(
    f"Absolute difference            : "
    f"{max_error_absolute_difference:.10f} V "
    f"({max_error_absolute_difference * 1000:.6f} mV)"
)

# ------------------------------------------------------------
# Examine distribution of larger discrepancies
# ------------------------------------------------------------

thresholds_mV = [0.001, 0.01, 0.1, 0.5, 1.0]

threshold_rows = []

for threshold_mV in thresholds_mV:

    threshold_V = threshold_mV / 1000

    count_above = int(
        np.sum(
            shap_absolute_reconstruction_error
            > threshold_V
        )
    )

    threshold_rows.append({
        "Threshold_mV": threshold_mV,
        "Observations_Above_Threshold": count_above,
        "Percentage_Above_Threshold": (
            count_above
            / FINAL_EXPLANATION_N
            * 100
        )
    })

additivity_threshold_summary = pd.DataFrame(
    threshold_rows
)

print()
print("Reconstruction-error threshold audit:")
display(additivity_threshold_summary)

# ------------------------------------------------------------
# Final structural verification
# ------------------------------------------------------------

finite_shap_values = np.isfinite(
    final_shap_values
).all()

finite_reconstructed_predictions = np.isfinite(
    shap_reconstructed_predictions
).all()

correct_reconstruction_length = (
    len(shap_reconstructed_predictions)
    == FINAL_EXPLANATION_N
)

correct_difference_length = (
    len(shap_reconstruction_difference)
    == FINAL_EXPLANATION_N
)

correlation_valid = np.isfinite(
    prediction_correlation
)

print()
print("=" * 78)
print("ADDITIVITY VERIFICATION CHECKS")
print("=" * 78)

print(
    f"All final SHAP values finite       : "
    f"{finite_shap_values}"
)
print(
    f"All reconstructed predictions finite: "
    f"{finite_reconstructed_predictions}"
)
print(
    f"Reconstruction length correct      : "
    f"{correct_reconstruction_length}"
)
print(
    f"Difference vector length correct   : "
    f"{correct_difference_length}"
)
print(
    f"Prediction correlation finite      : "
    f"{correlation_valid}"
)

all_additivity_checks_passed = all([
    finite_shap_values,
    finite_reconstructed_predictions,
    correct_reconstruction_length,
    correct_difference_length,
    correlation_valid
])

if not all_additivity_checks_passed:
    raise ValueError(
        "Final SHAP additivity structural verification failed."
    )

print()
print("STRUCTURAL VERIFICATION PASSED")
print(
    "The final SHAP representation has been reconstructed "
    "against predictions from the frozen XGBoost model."
)
print(
    "The numerical error distribution should be interpreted "
    "before proceeding to global SHAP visualisation."
)

FINAL SHAP ADDITIVITY AND CONSISTENCY VERIFICATION
Explanation observations       : 7,500
Predictors                     : 20
SHAP expected value            : 0.762257170 V
Mean direct prediction         : 0.750223935 V
Mean SHAP reconstruction       : 0.750223490 V
Mean signed difference         : -0.0000004442 V
Prediction correlation         : 0.999999843153

Absolute reconstruction-error summary:


,Metric,Error_V,Error_mV
0,Mean absolute reconstruction error,0.000001,0.001263
1,Median absolute reconstruction error,0.000000,0.000194
2,95th percentile absolute reconstruction error,0.000001,0.000555
3,99th percentile absolute reconstruction error,0.000001,0.000713
4,Maximum absolute reconstruction error,0.004656,4.656416



MAXIMUM RECONSTRUCTION-ERROR OBSERVATION
Sample position                : 1283
Original dataframe index       : 3136785
Operating stage                : 900.0
Direct XGBoost prediction      : 0.695486665 V
SHAP reconstructed prediction  : 0.690830249 V
Signed difference              : -0.0046564157 V
Absolute difference            : 0.0046564157 V (4.656416 mV)

Reconstruction-error threshold audit:


,Threshold_mV,Observations_Above_Threshold,Percentage_Above_Threshold
0,0.001000,28,0.373333
1,0.010000,27,0.360000
2,0.100000,15,0.200000
3,0.500000,1,0.013333
4,1.000000,1,0.013333



ADDITIVITY VERIFICATION CHECKS
All final SHAP values finite       : True
All reconstructed predictions finite: True
Reconstruction length correct      : True
Difference vector length correct   : True
Prediction correlation finite      : True

STRUCTURAL VERIFICATION PASSED
The final SHAP representation has been reconstructed against predictions from the frozen XGBoost model.
The numerical error distribution should be interpreted before proceeding to global SHAP visualisation.


In [ ]:
### 13.9 Global SHAP Feature Importance

Global feature importance is evaluated using the mean absolute SHAP value, mean |SHAP|, across the frozen 7,500-observation later-stage explanation sample.

For each predictor, the absolute SHAP contribution is calculated for every explained observation and then averaged across the complete explanation sample. The resulting quantity represents the average magnitude by which that predictor contributes to the XGBoost model output relative to the fixed SHAP reference.

SHAP magnitudes are reported in millivolts (mV) to provide an interpretable voltage scale. Larger mean |SHAP| values indicate that a predictor contributes more strongly, on average, to variation in the model's voltage predictions within the explained 900–1000 h holdout regime.

Mean |SHAP| measures attribution magnitude only. It does not indicate whether high or low feature values increase or decrease predicted voltage, and it does not establish causal or electrochemical importance. Directional behaviour is examined subsequently using the SHAP beeswarm and dependence analyses.

The global importance ranking is based exclusively on the frozen primary SHAP design: the frozen XGBoost model, 500-observation development background, interventional TreeSHAP formulation and stage-balanced 7,500-observation later-stage explanation sample.

In [34]:
# ============================================================
# 13.9 Global SHAP Feature Importance — Mean |SHAP|
# ============================================================

# ------------------------------------------------------------
# Recalculate directly from the frozen final SHAP matrix
# ------------------------------------------------------------

global_mean_abs_shap_V = np.abs(
    final_shap_values
).mean(axis=0)

global_shap_importance = pd.DataFrame({
    "Feature": FROZEN_FEATURES,
    "Mean_Abs_SHAP_V": global_mean_abs_shap_V
})

global_shap_importance["Mean_Abs_SHAP_mV"] = (
    global_shap_importance["Mean_Abs_SHAP_V"] * 1000
)

global_shap_importance = (
    global_shap_importance
    .sort_values(
        "Mean_Abs_SHAP_V",
        ascending=False
    )
    .reset_index(drop=True)
)

global_shap_importance["Rank"] = (
    np.arange(
        1,
        len(global_shap_importance) + 1
    )
)

# ------------------------------------------------------------
# Consistency check against frozen ranking from Section 13.7.6
# ------------------------------------------------------------

frozen_ranking_check = (
    final_feature_ranking[
        ["Feature", "Mean_Abs_SHAP_V", "Rank"]
    ]
    .sort_values("Rank")
    .reset_index(drop=True)
)

same_feature_order = (
    global_shap_importance["Feature"].tolist()
    == frozen_ranking_check["Feature"].tolist()
)

same_ranks = np.array_equal(
    global_shap_importance["Rank"].to_numpy(),
    frozen_ranking_check["Rank"].to_numpy()
)

same_magnitudes = np.allclose(
    global_shap_importance["Mean_Abs_SHAP_V"].to_numpy(),
    frozen_ranking_check["Mean_Abs_SHAP_V"].to_numpy(),
    rtol=1e-10,
    atol=1e-12
)

# ------------------------------------------------------------
# Relative contribution for descriptive context
# ------------------------------------------------------------

total_mean_abs_shap = (
    global_shap_importance[
        "Mean_Abs_SHAP_mV"
    ].sum()
)

global_shap_importance[
    "Relative_Mean_Abs_SHAP_Percent"
] = (
    global_shap_importance[
        "Mean_Abs_SHAP_mV"
    ]
    / total_mean_abs_shap
    * 100
)

# ------------------------------------------------------------
# Display numerical ranking
# ------------------------------------------------------------

print("=" * 78)
print("GLOBAL SHAP FEATURE IMPORTANCE")
print("=" * 78)

print(
    f"Explanation observations : "
    f"{FINAL_EXPLANATION_N:,}"
)
print(
    f"Background observations  : "
    f"{FINAL_BACKGROUND_N:,}"
)
print(
    f"SHAP formulation         : "
    f"Interventional TreeSHAP"
)
print(
    f"Number of predictors     : "
    f"{len(FROZEN_FEATURES)}"
)

print()
print("Global ranking by mean |SHAP|:")

display(
    global_shap_importance[
        [
            "Rank",
            "Feature",
            "Mean_Abs_SHAP_V",
            "Mean_Abs_SHAP_mV",
            "Relative_Mean_Abs_SHAP_Percent"
        ]
    ]
)

# ------------------------------------------------------------
# Dissertation-ready horizontal bar figure
# ------------------------------------------------------------

plot_data = (
    global_shap_importance
    .sort_values(
        "Mean_Abs_SHAP_mV",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(10, 8))

bars = ax.barh(
    plot_data["Feature"],
    plot_data["Mean_Abs_SHAP_mV"]
)

ax.set_xlabel(
    "Mean absolute SHAP value (mV)"
)

ax.set_ylabel(
    "Predictor"
)

ax.set_title(
    "Global SHAP Feature Importance for Later-Stage "
    "PEMFC Voltage Prediction"
)

# Add numerical labels to bars
for bar in bars:
    width = bar.get_width()

    ax.text(
        width,
        bar.get_y() + bar.get_height() / 2,
        f" {width:.2f}",
        va="center",
        ha="left",
        fontsize=8
    )

ax.grid(
    axis="x",
    alpha=0.25
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

all_importance_values_finite = np.isfinite(
    global_shap_importance[
        "Mean_Abs_SHAP_V"
    ]
).all()

all_importance_values_nonnegative = (
    global_shap_importance[
        "Mean_Abs_SHAP_V"
    ] >= 0
).all()

complete_feature_set = (
    set(global_shap_importance["Feature"])
    == set(FROZEN_FEATURES)
)

correct_feature_count = (
    len(global_shap_importance)
    == len(FROZEN_FEATURES)
)

print()
print("=" * 78)
print("GLOBAL IMPORTANCE VERIFICATION")
print("=" * 78)

print(
    f"Matches frozen feature order : "
    f"{same_feature_order}"
)
print(
    f"Matches frozen ranks         : "
    f"{same_ranks}"
)
print(
    f"Matches frozen magnitudes    : "
    f"{same_magnitudes}"
)
print(
    f"All importance values finite : "
    f"{all_importance_values_finite}"
)
print(
    f"All importance values >= 0   : "
    f"{all_importance_values_nonnegative}"
)
print(
    f"Complete frozen feature set  : "
    f"{complete_feature_set}"
)
print(
    f"Correct predictor count      : "
    f"{correct_feature_count}"
)

all_global_checks_passed = all([
    same_feature_order,
    same_ranks,
    same_magnitudes,
    all_importance_values_finite,
    all_importance_values_nonnegative,
    complete_feature_set,
    correct_feature_count
])

if not all_global_checks_passed:
    raise ValueError(
        "Global SHAP importance verification failed."
    )

print()
print("VERIFICATION PASSED")
print(
    "The final global SHAP importance ranking has been "
    "reproduced directly from the frozen SHAP matrix."
)

GLOBAL SHAP FEATURE IMPORTANCE
Explanation observations : 7,500
Background observations  : 500
SHAP formulation         : Interventional TreeSHAP
Number of predictors     : 20

Global ranking by mean |SHAP|:


,Rank,Feature,Mean_Abs_SHAP_V,Mean_Abs_SHAP_mV,Relative_Mean_Abs_SHAP_Percent
0,1,current,0.079239,79.238701,71.260017
1,2,total_anode_stack_flow,0.009776,9.775657,8.791328
2,3,temp_anode_endplate,0.006045,6.044815,5.436152
3,4,cathode_pressure_diff,0.005904,5.903888,5.309416
4,5,total_cathode_stack_flow,0.001697,1.696977,1.526105
5,6,temp_anode_dewpoint_water,0.001299,1.299153,1.168339
6,7,pressure_cathode_outlet,0.001230,1.229881,1.106042
7,8,anode_temp_diff,0.001119,1.118551,1.005922
8,9,cathode_dewpoint_offset,0.000905,0.904716,0.813619
9,10,temp_cathode_dewpoint_water,0.000832,0.832284,0.748479


<Figure size 1000x800 with 1 Axes>


GLOBAL IMPORTANCE VERIFICATION
Matches frozen feature order : True
Matches frozen ranks         : True
Matches frozen magnitudes    : True
All importance values finite : True
All importance values >= 0   : True
Complete frozen feature set  : True
Correct predictor count      : True

VERIFICATION PASSED
The final global SHAP importance ranking has been reproduced directly from the frozen SHAP matrix.


In [ ]:
### 13.10 SHAP Beeswarm: Global Attribution Magnitude and Direction

The global mean |SHAP| analysis identifies which predictors contribute most strongly to the XGBoost model output on average, but absolute SHAP values do not preserve the direction of those contributions.

A SHAP beeswarm plot is therefore used to examine both attribution magnitude and direction across the frozen 7,500-observation later-stage explanation sample. Each point represents one explained observation for one predictor. Its horizontal position represents the SHAP contribution to predicted stack voltage relative to the fixed SHAP expected value, while colour represents the corresponding observed predictor value.

Positive SHAP values indicate contributions that push the model prediction above its reference output, whereas negative SHAP values indicate contributions that push it below the reference output. The horizontal spread additionally shows heterogeneity in model attribution across observations.

Predictors are ordered by global mean absolute SHAP magnitude so that the beeswarm remains directly comparable with the global importance ranking in Section 13.9.

The beeswarm describes the behaviour of the fitted XGBoost model rather than a causal physical relationship. In particular, correlated operational predictors and engineered variables may share or redistribute attribution. Therefore, directional SHAP patterns are interpreted as model-attribution evidence and are subsequently considered alongside PEMFC operating principles, previous exploratory evidence and polarization-based degradation measurements.

In [35]:
# ============================================================
# 13.10 SHAP Beeswarm — Global Magnitude and Direction
# ============================================================

# ------------------------------------------------------------
# Construct a SHAP Explanation from the already-computed
# frozen SHAP matrix.
#
# IMPORTANT:
# No TreeSHAP values are recalculated in this section.
# ------------------------------------------------------------

final_shap_explanation = shap.Explanation(
    values=final_shap_values,
    base_values=np.full(
        FINAL_EXPLANATION_N,
        FINAL_SHAP_EXPECTED_VALUE
    ),
    data=X_final_explanation.to_numpy(),
    feature_names=FROZEN_FEATURES
)

# ------------------------------------------------------------
# Verify ordering before plotting
# ------------------------------------------------------------

beeswarm_mean_abs = np.abs(
    final_shap_explanation.values
).mean(axis=0)

beeswarm_ranking = pd.DataFrame({
    "Feature": FROZEN_FEATURES,
    "Mean_Abs_SHAP_V": beeswarm_mean_abs
}).sort_values(
    "Mean_Abs_SHAP_V",
    ascending=False
).reset_index(drop=True)

beeswarm_order_matches_global = (
    beeswarm_ranking["Feature"].tolist()
    == global_shap_importance["Feature"].tolist()
)

beeswarm_values_match_global = np.allclose(
    beeswarm_ranking["Mean_Abs_SHAP_V"].to_numpy(),
    global_shap_importance["Mean_Abs_SHAP_V"].to_numpy(),
    rtol=1e-10,
    atol=1e-12
)

print("=" * 78)
print("SHAP BEESWARM PRE-PLOT VERIFICATION")
print("=" * 78)

print(
    f"Explanation matrix shape       : "
    f"{final_shap_explanation.values.shape}"
)

print(
    f"Feature order matches global   : "
    f"{beeswarm_order_matches_global}"
)

print(
    f"Importance values match global : "
    f"{beeswarm_values_match_global}"
)

if not (
    beeswarm_order_matches_global
    and beeswarm_values_match_global
):
    raise ValueError(
        "Beeswarm/global importance consistency check failed."
    )

# ------------------------------------------------------------
# Global beeswarm — all 20 predictors
# ------------------------------------------------------------

plt.figure(figsize=(11, 9))

shap.plots.beeswarm(
    final_shap_explanation,
    max_display=len(FROZEN_FEATURES),
    order=shap.Explanation.abs.mean(0),
    show=False
)

ax = plt.gca()

ax.set_xlabel(
    "SHAP value for predicted stack voltage (V)"
)

ax.set_title(
    "SHAP Attribution Distribution for Later-Stage "
    "PEMFC Voltage Prediction",
    pad=15
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Directional numerical summary
# ------------------------------------------------------------

directional_rows = []

for feature_position, feature in enumerate(FROZEN_FEATURES):

    feature_shap = final_shap_values[:, feature_position]

    directional_rows.append({
        "Feature": feature,
        "Mean_SHAP_mV": (
            np.mean(feature_shap) * 1000
        ),
        "Median_SHAP_mV": (
            np.median(feature_shap) * 1000
        ),
        "Min_SHAP_mV": (
            np.min(feature_shap) * 1000
        ),
        "Max_SHAP_mV": (
            np.max(feature_shap) * 1000
        ),
        "Positive_SHAP_Percent": (
            np.mean(feature_shap > 0) * 100
        ),
        "Negative_SHAP_Percent": (
            np.mean(feature_shap < 0) * 100
        )
    })

directional_shap_summary = pd.DataFrame(
    directional_rows
)

# Preserve global importance order.
directional_shap_summary = (
    global_shap_importance[
        ["Rank", "Feature"]
    ]
    .merge(
        directional_shap_summary,
        on="Feature",
        how="left"
    )
    .sort_values("Rank")
    .reset_index(drop=True)
)

print()
print("=" * 78)
print("GLOBAL SHAP DIRECTIONAL SUMMARY")
print("=" * 78)

display(directional_shap_summary)

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------

correct_beeswarm_shape = (
    final_shap_explanation.values.shape
    == (FINAL_EXPLANATION_N, len(FROZEN_FEATURES))
)

finite_beeswarm_values = np.isfinite(
    final_shap_explanation.values
).all()

complete_directional_summary = (
    len(directional_shap_summary)
    == len(FROZEN_FEATURES)
)

print()
print("=" * 78)
print("BEESWARM VERIFICATION")
print("=" * 78)

print(
    f"Correct SHAP matrix shape      : "
    f"{correct_beeswarm_shape}"
)

print(
    f"All plotted SHAP values finite : "
    f"{finite_beeswarm_values}"
)

print(
    f"All 20 predictors summarised   : "
    f"{complete_directional_summary}"
)

if not all([
    correct_beeswarm_shape,
    finite_beeswarm_values,
    complete_directional_summary
]):
    raise ValueError(
        "SHAP beeswarm verification failed."
    )

print()
print("VERIFICATION PASSED")
print(
    "The global beeswarm uses the frozen final SHAP matrix "
    "without recalculating TreeSHAP values."
)

SHAP BEESWARM PRE-PLOT VERIFICATION
Explanation matrix shape       : (7500, 20)
Feature order matches global   : True
Importance values match global : True


<Figure size 800x950 with 2 Axes>


GLOBAL SHAP DIRECTIONAL SUMMARY


,Rank,Feature,Mean_SHAP_mV,Median_SHAP_mV,Min_SHAP_mV,Max_SHAP_mV,Positive_SHAP_Percent,Negative_SHAP_Percent
0,1,current,-6.389181,-22.939179,-214.713677,154.229092,46.093333,53.906667
1,2,total_anode_stack_flow,-1.309827,3.132555,-35.130926,21.461835,59.360000,40.640000
2,3,temp_anode_endplate,-5.367449,-4.952429,-18.072814,19.272208,7.493333,92.506667
3,4,cathode_pressure_diff,0.018139,-3.654579,-9.757008,24.014418,33.093333,66.906667
4,5,total_cathode_stack_flow,-0.298355,-0.586510,-9.688421,10.509342,38.680000,61.320000
5,6,temp_anode_dewpoint_water,0.221387,-0.496946,-4.582220,13.892784,26.960000,73.040000
6,7,pressure_cathode_outlet,-0.057076,-0.507378,-6.572731,12.821061,15.520000,84.480000
7,8,anode_temp_diff,0.622663,0.786786,-6.332842,3.842331,84.986667,15.013333
8,9,cathode_dewpoint_offset,0.064493,-0.145008,-2.377295,2.816600,48.493333,51.506667
9,10,temp_cathode_dewpoint_water,0.163965,-0.109443,-2.153031,7.844247,46.120000,53.880000



BEESWARM VERIFICATION
Correct SHAP matrix shape      : True
All plotted SHAP values finite : True
All 20 predictors summarised   : True

VERIFICATION PASSED
The global beeswarm uses the frozen final SHAP matrix without recalculating TreeSHAP values.


In [ ]:
### 13.11 SHAP Dependence Analysis: Selection of Predictors

The global importance and beeswarm analyses establish the overall magnitude and directional distribution of model attribution, but they do not show in detail how SHAP contributions vary across the observed range of individual predictors. SHAP dependence analysis is therefore used to examine selected high-value predictors more closely.

Six predictors are selected using two complementary criteria: global SHAP importance and coverage of distinct PEMFC operational domains. This avoids mechanically selecting only the highest-ranked variables when some predictors contain strongly overlapping operational information.

The selected predictors are:

- `current` — Rank 1; applied electrical load/current.
- `total_anode_stack_flow` — Rank 2; anode reactant-flow behaviour.
- `temp_anode_endplate` — Rank 3; thermal operating behaviour.
- `cathode_pressure_diff` — Rank 4; cathode pressure-gradient behaviour.
- `temp_anode_dewpoint_water` — Rank 6; anode humidification/water-management conditions.
- `cathode_dewpoint_offset` — Rank 9; engineered cathode humidification/water-management indicator.

The purpose of this selection is interpretive coverage rather than additional feature selection. Predictors excluded from the dependence figures remain part of the frozen XGBoost model and the complete global SHAP analysis.

Dependence plots are interpreted as descriptions of how the fitted XGBoost model attributes voltage predictions across observed predictor values. They do not establish causal relationships or independently identify electrochemical degradation mechanisms, particularly because several operational and engineered predictors are correlated.

In [36]:
# ============================================================
# 13.11 SHAP Dependence Analysis — Predictor Selection
# ============================================================

dependence_features = [
    "current",
    "total_anode_stack_flow",
    "temp_anode_endplate",
    "cathode_pressure_diff",
    "temp_anode_dewpoint_water",
    "cathode_dewpoint_offset"
]

dependence_selection = (
    global_shap_importance[
        [
            "Rank",
            "Feature",
            "Mean_Abs_SHAP_mV",
            "Relative_Mean_Abs_SHAP_Percent"
        ]
    ]
    .loc[
        lambda df:
        df["Feature"].isin(dependence_features)
    ]
    .sort_values("Rank")
    .reset_index(drop=True)
)

print("=" * 78)
print("SHAP DEPENDENCE-PLOT PREDICTOR SELECTION")
print("=" * 78)

display(dependence_selection)

print()
print(f"Selected predictors : {len(dependence_features)}")

for feature in dependence_features:
    print(f"  - {feature}")

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

all_features_available = all(
    feature in FROZEN_FEATURES
    for feature in dependence_features
)

all_features_ranked = (
    len(dependence_selection)
    == len(dependence_features)
)

unique_features = (
    len(set(dependence_features))
    == len(dependence_features)
)

print()
print("=" * 78)
print("DEPENDENCE-SELECTION VERIFICATION")
print("=" * 78)

print(
    f"All predictors in frozen model : "
    f"{all_features_available}"
)

print(
    f"All predictors globally ranked : "
    f"{all_features_ranked}"
)

print(
    f"All selected predictors unique : "
    f"{unique_features}"
)

if not all([
    all_features_available,
    all_features_ranked,
    unique_features
]):
    raise ValueError(
        "Dependence-feature selection verification failed."
    )

print()
print("VERIFICATION PASSED")
print(
    "Six predictors are retained for detailed SHAP "
    "dependence analysis."
)

SHAP DEPENDENCE-PLOT PREDICTOR SELECTION


,Rank,Feature,Mean_Abs_SHAP_mV,Relative_Mean_Abs_SHAP_Percent
0,1,current,79.238701,71.260017
1,2,total_anode_stack_flow,9.775657,8.791328
2,3,temp_anode_endplate,6.044815,5.436152
3,4,cathode_pressure_diff,5.903888,5.309416
4,6,temp_anode_dewpoint_water,1.299153,1.168339
5,9,cathode_dewpoint_offset,0.904716,0.813619



Selected predictors : 6
  - current
  - total_anode_stack_flow
  - temp_anode_endplate
  - cathode_pressure_diff
  - temp_anode_dewpoint_water
  - cathode_dewpoint_offset

DEPENDENCE-SELECTION VERIFICATION
All predictors in frozen model : True
All predictors globally ranked : True
All selected predictors unique : True

VERIFICATION PASSED
Six predictors are retained for detailed SHAP dependence analysis.


In [ ]:
### 13.12 SHAP Dependence Analysis

SHAP dependence plots are used to examine how the contribution assigned to each selected predictor changes across its observed values in the frozen later-stage explanation sample.

For each plot, the x-axis represents the observed predictor value and the y-axis represents the corresponding SHAP contribution to predicted stack voltage in millivolts (mV). A horizontal reference line at SHAP = 0 mV separates contributions that push the model prediction above the fixed SHAP reference output from contributions that push it below that reference.

Six predictors are examined individually: current, total anode stack flow, anode endplate temperature, cathode pressure difference, anode dewpoint-water temperature and cathode dewpoint offset.

The plots are intentionally generated separately because the predictors have different physical units, ranges and attribution magnitudes. This also allows nonlinearities, thresholds, clusters and attribution heterogeneity to be inspected without compressing the individual relationships.

No additional SHAP values are calculated in this section. The plots use the frozen 7,500 × 20 interventional TreeSHAP matrix established previously.

The dependence patterns describe relationships learned by the fitted XGBoost model. They should not be interpreted as isolated causal effects because correlated operational predictors, engineered variables and interactions can redistribute attribution among features.

In [39]:
# ============================================================
# 13.12 SHAP Dependence Plots
# ============================================================

import scipy

# Human-readable labels and physical units
dependence_axis_labels = {
    "current": "Current (A)",
    "total_anode_stack_flow": "Total anode stack flow (NLPM)",
    "temp_anode_endplate": "Anode endplate temperature (°C)",
    "cathode_pressure_diff": "Cathode pressure difference (kPag)",
    "temp_anode_dewpoint_water": "Anode dewpoint-water temperature (°C)",
    "cathode_dewpoint_offset": "Cathode dewpoint offset (°C)"
}

# ------------------------------------------------------------
# Create one dependence figure per selected predictor
# ------------------------------------------------------------

for feature in dependence_features:

    feature_position = FROZEN_FEATURES.index(feature)

    feature_values = (
        X_final_explanation[feature]
        .to_numpy()
    )

    feature_shap_mV = (
        final_shap_values[:, feature_position]
        * 1000
    )

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    ax.scatter(
        feature_values,
        feature_shap_mV,
        s=10,
        alpha=0.35
    )

    ax.axhline(
        y=0,
        linewidth=1,
        linestyle="--"
    )

    ax.set_xlabel(
        dependence_axis_labels[feature]
    )

    ax.set_ylabel(
        "SHAP contribution to predicted stack voltage (mV)"
    )

    ax.set_title(
        f"SHAP Dependence: {feature}"
    )

    ax.grid(
        alpha=0.20
    )

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# Numerical dependence summary
# ------------------------------------------------------------

dependence_summary_rows = []

for feature in dependence_features:

    feature_position = FROZEN_FEATURES.index(feature)

    feature_values = (
        X_final_explanation[feature]
        .to_numpy()
    )

    feature_shap_mV = (
        final_shap_values[:, feature_position]
        * 1000
    )

    # Spearman is used here only as a descriptive summary
    # of monotonic association between observed feature
    # values and their SHAP attributions.
    spearman_result = scipy.stats.spearmanr(
        feature_values,
        feature_shap_mV
    )

    dependence_summary_rows.append({
        "Feature": feature,
        "Feature_Min": np.min(feature_values),
        "Feature_Median": np.median(feature_values),
        "Feature_Max": np.max(feature_values),
        "SHAP_Min_mV": np.min(feature_shap_mV),
        "SHAP_Median_mV": np.median(feature_shap_mV),
        "SHAP_Max_mV": np.max(feature_shap_mV),
        "Spearman_Feature_vs_SHAP": (
            spearman_result.statistic
        )
    })

dependence_numeric_summary = pd.DataFrame(
    dependence_summary_rows
)

# Preserve selected importance order
dependence_numeric_summary = (
    dependence_selection[
        ["Rank", "Feature"]
    ]
    .merge(
        dependence_numeric_summary,
        on="Feature",
        how="left"
    )
    .sort_values("Rank")
    .reset_index(drop=True)
)

print("=" * 78)
print("SHAP DEPENDENCE NUMERICAL SUMMARY")
print("=" * 78)

display(dependence_numeric_summary)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

correct_dependence_feature_count = (
    len(dependence_numeric_summary)
    == len(dependence_features)
)

all_dependence_values_finite = (
    dependence_numeric_summary
    .drop(columns=["Feature"])
    .select_dtypes(include=[np.number])
    .apply(np.isfinite)
    .all()
    .all()
)

all_selected_features_present = (
    set(dependence_numeric_summary["Feature"])
    == set(dependence_features)
)

print()
print("=" * 78)
print("DEPENDENCE ANALYSIS VERIFICATION")
print("=" * 78)

print(
    f"Correct number of predictors : "
    f"{correct_dependence_feature_count}"
)

print(
    f"All numerical values finite  : "
    f"{all_dependence_values_finite}"
)

print(
    f"All selected features present: "
    f"{all_selected_features_present}"
)

if not all([
    correct_dependence_feature_count,
    all_dependence_values_finite,
    all_selected_features_present
]):
    raise ValueError(
        "SHAP dependence-analysis verification failed."
    )

print()
print("VERIFICATION PASSED")
print(
    "Dependence analysis uses the frozen final SHAP "
    "matrix without recalculating TreeSHAP values."
)

<Figure size 900x600 with 1 Axes>

<Figure size 900x600 with 1 Axes>

<Figure size 900x600 with 1 Axes>

<Figure size 900x600 with 1 Axes>

<Figure size 900x600 with 1 Axes>

<Figure size 900x600 with 1 Axes>

SHAP DEPENDENCE NUMERICAL SUMMARY


,Rank,Feature,Feature_Min,Feature_Median,Feature_Max,SHAP_Min_mV,SHAP_Median_mV,SHAP_Max_mV,Spearman_Feature_vs_SHAP
0,1,current,-0.002500,9.484800,35.539000,-214.713677,-22.939179,154.229092,-0.937324
1,2,total_anode_stack_flow,0.083000,0.133000,0.497000,-35.130926,3.132555,21.461835,-0.939948
2,3,temp_anode_endplate,83.066971,83.609184,84.827179,-18.072814,-4.952429,19.272208,-0.889261
3,4,cathode_pressure_diff,0.096268,1.412519,9.627284,-9.757008,-3.654579,24.014418,0.948716
4,6,temp_anode_dewpoint_water,52.708138,54.989353,55.642582,-4.582220,-0.496946,13.892784,-0.816262
5,9,cathode_dewpoint_offset,3.496681,4.809444,10.771855,-2.377295,-0.145008,2.816600,0.922054



DEPENDENCE ANALYSIS VERIFICATION
Correct number of predictors : True
All numerical values finite  : True
All selected features present: True

VERIFICATION PASSED
Dependence analysis uses the frozen final SHAP matrix without recalculating TreeSHAP values.


In [ ]:
### 13.13 Stage-Wise SHAP Attribution Analysis

The preceding SHAP analyses aggregate the complete 900–1000 h explanation sample. To determine whether the model's attribution structure remains similar across individual later durability stages, SHAP magnitude is now evaluated separately at 900, 950 and 1000 h.

The frozen explanation design contains exactly 2,500 observations from each stage. Consequently, stage-wise mean absolute SHAP values can be compared without unequal explanation-sample sizes influencing the descriptive comparison.

For every predictor and durability stage, mean |SHAP| is calculated in millivolts. These values describe the average magnitude of each predictor's contribution to the XGBoost voltage predictions within that stage.

A heatmap is used to compare the ten globally highest-ranked predictors across the three stages, while a complete numerical table retains results for all 20 predictors. Additional rank-based comparisons assess whether the overall attribution hierarchy remains stable across the later-stage regime.

Stage-wise differences in SHAP attribution should not automatically be interpreted as physical degradation effects. They may reflect changes in operational distributions, model behaviour, predictor dependence or interactions across stages. The analysis therefore provides evidence about temporal variation in model attribution rather than direct evidence of an electrochemical degradation mechanism.

In [42]:
# ============================================================
# 13.13 Stage-Wise SHAP Attribution Analysis
# ============================================================

from matplotlib.colors import LogNorm

STAGE_SHAP_STAGES = [900, 950, 1000]

stage_shap_rows = []

# ------------------------------------------------------------
# Calculate mean |SHAP| separately within each stage
# ------------------------------------------------------------

for stage in STAGE_SHAP_STAGES:

    stage_mask = (
        final_explanation_data[STAGE_COLUMN]
        .to_numpy()
        == stage
    )

    stage_count = int(stage_mask.sum())

    stage_shap_matrix = (
        final_shap_values[stage_mask]
    )

    stage_mean_abs_shap_mV = (
        np.abs(stage_shap_matrix)
        .mean(axis=0)
        * 1000
    )

    for feature_position, feature in enumerate(FROZEN_FEATURES):

        stage_shap_rows.append({
            "Operating_Hour": stage,
            "Feature": feature,
            "Mean_Abs_SHAP_mV":
                stage_mean_abs_shap_mV[feature_position]
        })

stage_shap_long = pd.DataFrame(
    stage_shap_rows
)

# ------------------------------------------------------------
# Wide table: features × stages
# ------------------------------------------------------------

stage_shap_wide = (
    stage_shap_long
    .pivot(
        index="Feature",
        columns="Operating_Hour",
        values="Mean_Abs_SHAP_mV"
    )
    .reset_index()
)

stage_shap_wide.columns.name = None

stage_shap_wide = (
    global_shap_importance[
        ["Rank", "Feature"]
    ]
    .merge(
        stage_shap_wide,
        on="Feature",
        how="left"
    )
    .sort_values("Rank")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Stage-specific ranks
# ------------------------------------------------------------

for stage in STAGE_SHAP_STAGES:

    stage_shap_wide[
        f"Rank_{stage}h"
    ] = (
        stage_shap_wide[stage]
        .rank(
            method="min",
            ascending=False
        )
        .astype(int)
    )

# ------------------------------------------------------------
# Numerical comparison
# ------------------------------------------------------------

print("=" * 78)
print("STAGE-WISE SHAP IMPORTANCE")
print("=" * 78)

for stage in STAGE_SHAP_STAGES:

    count = int(
        (
            final_explanation_data[STAGE_COLUMN]
            == stage
        ).sum()
    )

    print(
        f"{stage} h explanation observations : "
        f"{count:,}"
    )

print()
print("All 20 predictors:")
display(stage_shap_wide)

# ------------------------------------------------------------
# Rank stability between stages
# ------------------------------------------------------------

stage_rank_correlations = []

for i, stage_a in enumerate(STAGE_SHAP_STAGES):

    for stage_b in STAGE_SHAP_STAGES[i + 1:]:

        rho = scipy.stats.spearmanr(
            stage_shap_wide[stage_a],
            stage_shap_wide[stage_b]
        ).statistic

        top5_a = set(
            stage_shap_wide
            .nlargest(5, stage_a)["Feature"]
        )

        top5_b = set(
            stage_shap_wide
            .nlargest(5, stage_b)["Feature"]
        )

        top10_a = set(
            stage_shap_wide
            .nlargest(10, stage_a)["Feature"]
        )

        top10_b = set(
            stage_shap_wide
            .nlargest(10, stage_b)["Feature"]
        )

        stage_rank_correlations.append({
            "Stage_A": stage_a,
            "Stage_B": stage_b,
            "Spearman_Rho": rho,
            "Top5_Overlap": len(
                top5_a.intersection(top5_b)
            ),
            "Top10_Overlap": len(
                top10_a.intersection(top10_b)
            )
        })

stage_rank_stability = pd.DataFrame(
    stage_rank_correlations
)

print()
print("Stage-wise attribution stability:")
display(stage_rank_stability)

# ------------------------------------------------------------
# Heatmap — globally top 10 predictors
# ------------------------------------------------------------

top10_features = (
    global_shap_importance
    .head(10)["Feature"]
    .tolist()
)

heatmap_data = (
    stage_shap_wide
    .set_index("Feature")
    .loc[top10_features, STAGE_SHAP_STAGES]
)

heatmap_values = heatmap_data.to_numpy()

# ------------------------------------------------------------
# Professional heatmap
#
# Logarithmic colour normalization is used because current
# (~79 mV) is much larger than the remaining predictors.
# The displayed numerical values remain the original mean
# |SHAP| magnitudes in mV.
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10.5, 7.5)
)

vmin = heatmap_values.min()
vmax = heatmap_values.max()

im = ax.imshow(
    heatmap_values,
    aspect="auto",
    cmap="Blues",
    norm=LogNorm(
        vmin=vmin,
        vmax=vmax
    )
)

# ------------------------------------------------------------
# Axis labels
# ------------------------------------------------------------

ax.set_xticks(
    np.arange(len(STAGE_SHAP_STAGES))
)

ax.set_xticklabels(
    [f"{stage} h" for stage in STAGE_SHAP_STAGES],
    fontsize=11
)

ax.set_yticks(
    np.arange(len(top10_features))
)

ax.set_yticklabels(
    top10_features,
    fontsize=10
)

ax.set_xlabel(
    "Later durability stage",
    fontsize=11
)

ax.set_ylabel(
    "Predictor",
    fontsize=11
)

ax.set_title(
    "Stage-Wise Mean Absolute SHAP Attribution",
    fontsize=13,
    pad=14
)

# ------------------------------------------------------------
# Colour bar
# ------------------------------------------------------------

cbar = fig.colorbar(
    im,
    ax=ax,
    pad=0.03
)

cbar.set_label(
    "Mean |SHAP| (mV) — logarithmic colour scale",
    fontsize=10
)

cbar.ax.tick_params(
    labelsize=9
)

# ------------------------------------------------------------
# Numerical annotations with adaptive text contrast
# ------------------------------------------------------------

log_vmin = np.log10(vmin)
log_vmax = np.log10(vmax)

for row in range(
    heatmap_data.shape[0]
):

    for column in range(
        heatmap_data.shape[1]
    ):

        value = heatmap_data.iloc[
            row,
            column
        ]

        # Position of the value within the logarithmic
        # colour scale, used only to choose readable text.
        normalized_log_value = (
            (np.log10(value) - log_vmin)
            / (log_vmax - log_vmin)
        )

        if normalized_log_value > 0.58:
            annotation_colour = "white"
        else:
            annotation_colour = "black"

        ax.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=9,
            fontweight="normal",
            color=annotation_colour
        )

# ------------------------------------------------------------
# Clean dissertation-ready appearance
# ------------------------------------------------------------

ax.tick_params(
    axis="both",
    which="both",
    length=0
)

for spine in ax.spines.values():
    spine.set_visible(False)

# Light separation between heatmap cells
ax.set_xticks(
    np.arange(-0.5, len(STAGE_SHAP_STAGES), 1),
    minor=True
)

ax.set_yticks(
    np.arange(-0.5, len(top10_features), 1),
    minor=True
)

ax.grid(
    which="minor",
    linewidth=0.8,
    alpha=0.35
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)

# Extra left margin prevents long predictor names
# from being clipped.
plt.subplots_adjust(
    left=0.31,
    right=0.88,
    top=0.90,
    bottom=0.12
)

plt.show()

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

stage_counts_correct = all(
    int(
        (
            final_explanation_data[STAGE_COLUMN]
            == stage
        ).sum()
    ) == 2500
    for stage in STAGE_SHAP_STAGES
)

complete_stage_table = (
    len(stage_shap_wide)
    == len(FROZEN_FEATURES)
)

finite_stage_values = np.isfinite(
    stage_shap_wide[
        STAGE_SHAP_STAGES
    ].to_numpy()
).all()

all_stage_values_nonnegative = (
    stage_shap_wide[
        STAGE_SHAP_STAGES
    ].to_numpy()
    >= 0
).all()

positive_heatmap_values = (
    heatmap_values > 0
).all()

print()
print("=" * 78)
print("STAGE-WISE SHAP VERIFICATION")
print("=" * 78)

print(
    f"2,500 observations per stage : "
    f"{stage_counts_correct}"
)

print(
    f"All 20 predictors retained   : "
    f"{complete_stage_table}"
)

print(
    f"All stage values finite      : "
    f"{finite_stage_values}"
)

print(
    f"All mean |SHAP| values >= 0  : "
    f"{all_stage_values_nonnegative}"
)

print(
    f"Heatmap values > 0 for LogNorm: "
    f"{positive_heatmap_values}"
)

if not all([
    stage_counts_correct,
    complete_stage_table,
    finite_stage_values,
    all_stage_values_nonnegative,
    positive_heatmap_values
]):
    raise ValueError(
        "Stage-wise SHAP verification failed."
    )

print()
print("VERIFICATION PASSED")
print(
    "Stage-wise attribution was calculated from the "
    "frozen primary SHAP matrix without recomputing TreeSHAP."
)
print(
    "The heatmap uses logarithmic colour normalization "
    "for visualisation only; displayed values remain the "
    "original mean |SHAP| magnitudes in mV."
)

STAGE-WISE SHAP IMPORTANCE
900 h explanation observations : 2,500
950 h explanation observations : 2,500
1000 h explanation observations : 2,500

All 20 predictors:


,Rank,Feature,900,950,1000,Rank_900h,Rank_950h,Rank_1000h
0,1,current,78.532501,79.639306,79.544298,1,1,1
1,2,total_anode_stack_flow,9.773832,9.874530,9.678608,2,2,2
2,3,temp_anode_endplate,6.103328,6.155641,5.875476,3,3,4
3,4,cathode_pressure_diff,5.867098,5.751346,6.093221,4,4,3
4,5,total_cathode_stack_flow,1.681444,1.681828,1.727659,5,5,5
5,6,temp_anode_dewpoint_water,1.067165,1.595391,1.234903,7,6,7
6,7,pressure_cathode_outlet,1.279701,1.230907,1.179034,6,7,8
7,8,anode_temp_diff,0.953730,1.050319,1.351604,8,8,6
8,9,cathode_dewpoint_offset,0.918108,0.872601,0.923441,9,10,9
9,10,temp_cathode_dewpoint_water,0.722970,0.991737,0.782143,10,9,10



Stage-wise attribution stability:


,Stage_A,Stage_B,Spearman_Rho,Top5_Overlap,Top10_Overlap
0,900,950,0.990977,5,10
1,900,1000,0.981955,5,10
2,950,1000,0.971429,5,10


<Figure size 1050x750 with 2 Axes>


STAGE-WISE SHAP VERIFICATION
2,500 observations per stage : True
All 20 predictors retained   : True
All stage values finite      : True
All mean |SHAP| values >= 0  : True
Heatmap values > 0 for LogNorm: True

VERIFICATION PASSED
Stage-wise attribution was calculated from the frozen primary SHAP matrix without recomputing TreeSHAP.
The heatmap uses logarithmic colour normalization for visualisation only; displayed values remain the original mean |SHAP| magnitudes in mV.


In [ ]:
### 13.14 Representative Local SHAP Explanations

Global and stage-wise SHAP analyses describe attribution patterns across groups of observations, but they do not show how individual model predictions are constructed. Representative local explanations are therefore examined for one observation from each later durability stage: 900, 950 and 1000 h.

To avoid subjective or visually convenient case selection, the representative observation for each stage is selected using a predefined prediction-error criterion. Within each stage, the absolute XGBoost prediction error is calculated for every observation in the frozen explanation sample. The observation whose absolute prediction error is closest to the stage-specific median absolute prediction error is retained.

This approach selects a case with approximately typical predictive error rather than an unusually accurate or inaccurate prediction.

For each representative observation, the local SHAP explanation decomposes the frozen XGBoost prediction into the fixed SHAP expected value and the contributions assigned to the 20 predictors. Positive SHAP contributions push the prediction above the reference output, whereas negative contributions push it below the reference output.

The resulting waterfall plots are local model explanations only. They illustrate how the XGBoost model constructed three representative predictions and should not be interpreted as causal decompositions of PEMFC voltage or degradation.

In [43]:
# ============================================================
# 13.14 Representative Local SHAP Explanations
# ============================================================

LOCAL_EXPLANATION_STAGES = [900, 950, 1000]

# ------------------------------------------------------------
# Prediction error for every observation in the frozen
# explanation sample
# ------------------------------------------------------------

final_actual_voltage = (
    y_final_explanation
    .to_numpy()
)

final_prediction_error = (
    final_explanation_predictions
    - final_actual_voltage
)

final_absolute_prediction_error = np.abs(
    final_prediction_error
)

# ------------------------------------------------------------
# Select observation nearest to median absolute prediction
# error within each stage
# ------------------------------------------------------------

representative_rows = []
representative_positions = {}

stage_array = (
    final_explanation_data[STAGE_COLUMN]
    .to_numpy()
)

for stage in LOCAL_EXPLANATION_STAGES:

    stage_positions = np.where(
        stage_array == stage
    )[0]

    stage_abs_errors = (
        final_absolute_prediction_error[
            stage_positions
        ]
    )

    stage_median_abs_error = float(
        np.median(stage_abs_errors)
    )

    distance_from_median = np.abs(
        stage_abs_errors
        - stage_median_abs_error
    )

    local_position = int(
        np.argmin(distance_from_median)
    )

    selected_position = int(
        stage_positions[local_position]
    )

    representative_positions[stage] = (
        selected_position
    )

    representative_rows.append({
        "Operating_Hour": stage,
        "Sample_Position": selected_position,
        "Original_Index":
            final_explanation_data.index[
                selected_position
            ],
        "Actual_Voltage_V":
            final_actual_voltage[
                selected_position
            ],
        "Predicted_Voltage_V":
            final_explanation_predictions[
                selected_position
            ],
        "Signed_Error_mV":
            final_prediction_error[
                selected_position
            ] * 1000,
        "Absolute_Error_mV":
            final_absolute_prediction_error[
                selected_position
            ] * 1000,
        "Stage_Median_Absolute_Error_mV":
            stage_median_abs_error * 1000,
        "Distance_From_Stage_Median_mV":
            distance_from_median[
                local_position
            ] * 1000
    })

representative_local_cases = pd.DataFrame(
    representative_rows
)

print("=" * 78)
print("REPRESENTATIVE LOCAL SHAP CASE SELECTION")
print("=" * 78)

display(representative_local_cases)

# ------------------------------------------------------------
# Construct individual SHAP Explanation objects
# ------------------------------------------------------------

representative_explanations = {}

for stage in LOCAL_EXPLANATION_STAGES:

    position = representative_positions[
        stage
    ]

    representative_explanations[stage] = (
        shap.Explanation(
            values=final_shap_values[
                position
            ],
            base_values=FINAL_SHAP_EXPECTED_VALUE,
            data=X_final_explanation
                .iloc[position]
                .to_numpy(),
            feature_names=FROZEN_FEATURES
        )
    )

# ------------------------------------------------------------
# Local additivity information
# ------------------------------------------------------------

local_additivity_rows = []

for stage in LOCAL_EXPLANATION_STAGES:

    position = representative_positions[
        stage
    ]

    explanation = (
        representative_explanations[
            stage
        ]
    )

    reconstructed_prediction = (
        FINAL_SHAP_EXPECTED_VALUE
        + explanation.values.sum()
    )

    direct_prediction = (
        final_explanation_predictions[
            position
        ]
    )

    local_additivity_rows.append({
        "Operating_Hour": stage,
        "Direct_Prediction_V":
            direct_prediction,
        "SHAP_Reconstruction_V":
            reconstructed_prediction,
        "Reconstruction_Difference_mV":
            (
                reconstructed_prediction
                - direct_prediction
            ) * 1000
    })

representative_additivity = pd.DataFrame(
    local_additivity_rows
)

print()
print("Local SHAP reconstruction check:")
display(representative_additivity)

# ------------------------------------------------------------
# Verification before plotting
# ------------------------------------------------------------

correct_number_cases = (
    len(representative_local_cases)
    == 3
)

correct_stages_selected = (
    set(
        representative_local_cases[
            "Operating_Hour"
        ]
    )
    == set(LOCAL_EXPLANATION_STAGES)
)

unique_positions = (
    len(
        set(
            representative_positions.values()
        )
    )
    == 3
)

all_cases_finite = (
    representative_local_cases
    .select_dtypes(include=[np.number])
    .apply(np.isfinite)
    .all()
    .all()
)

print()
print("=" * 78)
print("LOCAL CASE-SELECTION VERIFICATION")
print("=" * 78)

print(
    f"Exactly three cases selected : "
    f"{correct_number_cases}"
)

print(
    f"900/950/1000 h represented   : "
    f"{correct_stages_selected}"
)

print(
    f"Selected positions unique    : "
    f"{unique_positions}"
)

print(
    f"All case statistics finite   : "
    f"{all_cases_finite}"
)

if not all([
    correct_number_cases,
    correct_stages_selected,
    unique_positions,
    all_cases_finite
]):
    raise ValueError(
        "Representative local-case selection failed."
    )

print()
print("VERIFICATION PASSED")
print(
    "One representative observation per later durability "
    "stage has been selected objectively using the "
    "stage-specific median absolute prediction error."
)

REPRESENTATIVE LOCAL SHAP CASE SELECTION


,Operating_Hour,Sample_Position,Original_Index,Actual_Voltage_V,Predicted_Voltage_V,Signed_Error_mV,Absolute_Error_mV,Stage_Median_Absolute_Error_mV,Distance_From_Stage_Median_mV
0,900,811,3201119,0.685200,0.681279,-3.921294,3.921294,3.921509,0.000215
1,950,318,3354208,0.726200,0.730543,4.342719,4.342719,4.341529,0.001191
2,1000,2691,3423797,0.847700,0.841734,-5.966127,5.966127,5.964555,0.001573



Local SHAP reconstruction check:


,Operating_Hour,Direct_Prediction_V,SHAP_Reconstruction_V,Reconstruction_Difference_mV
0,900,0.681279,0.681279,-0.000097
1,950,0.730543,0.730543,0.000442
2,1000,0.841734,0.841734,0.000085



LOCAL CASE-SELECTION VERIFICATION
Exactly three cases selected : True
900/950/1000 h represented   : True
Selected positions unique    : True
All case statistics finite   : True

VERIFICATION PASSED
One representative observation per later durability stage has been selected objectively using the stage-specific median absolute prediction error.


In [ ]:
### 13.15 Representative Local SHAP Waterfall Explanations

Waterfall plots are used to visualise how individual feature attributions combine to form the XGBoost prediction for the three representative later-stage observations selected in Section 13.14.

One representative observation is examined at each of 900, 950 and 1000 h. These observations were selected objectively as those with absolute prediction errors closest to the median absolute prediction error within their respective stages.

Each waterfall begins from the fixed SHAP expected value defined by the development-background reference. Individual feature contributions then move the model output upward or downward until the final predicted stack voltage is reached.

For readability, the ten largest local SHAP contributions are displayed individually in each figure, while the remaining lower-magnitude contributions are aggregated by the SHAP plotting routine. All 20 predictors remain included in the underlying explanation.

These local explanations illustrate how the fitted XGBoost model constructed specific representative predictions. They do not represent causal decompositions of stack voltage and should not be interpreted as identifying physical degradation mechanisms.

In [47]:
# ============================================================
# 13.15 Representative Local SHAP Waterfall Explanations
# Final dissertation-ready version
# ============================================================

LOCAL_WATERFALL_MAX_DISPLAY = 10

# ------------------------------------------------------------
# Professional colour palette
# ------------------------------------------------------------

POSITIVE_COLOUR = "#A64B4B"      # muted red
NEGATIVE_COLOUR = "#3F73A8"      # muted blue
OTHER_COLOUR = "#777777"         # neutral grey
REFERENCE_COLOUR = "#666666"     # SHAP reference line
PREDICTION_COLOUR = "#222222"    # model prediction line


# ============================================================
# A. GENERATE REPRESENTATIVE LOCAL WATERFALL PLOTS
# ============================================================

for stage in LOCAL_EXPLANATION_STAGES:

    # --------------------------------------------------------
    # Representative observation identifiers
    # --------------------------------------------------------

    position = int(
        representative_positions[stage]
    )

    original_index = int(
        final_explanation_data.index[
            position
        ]
    )

    # --------------------------------------------------------
    # Observed and predicted voltage
    # --------------------------------------------------------

    actual_voltage = float(
        final_actual_voltage[
            position
        ]
    )

    predicted_voltage = float(
        final_explanation_predictions[
            position
        ]
    )

    absolute_error_mV = float(
        final_absolute_prediction_error[
            position
        ] * 1000
    )

    # --------------------------------------------------------
    # Frozen local SHAP values
    # --------------------------------------------------------

    local_shap_mV = (
        final_shap_values[
            position
        ] * 1000
    )

    local_feature_values = (
        X_final_explanation
        .iloc[position]
        .to_numpy()
    )

    # --------------------------------------------------------
    # Select the 9 largest individual SHAP contributions.
    #
    # max_display = 10 therefore gives:
    #   9 individual predictors
    # + 1 aggregated "other features" contribution.
    #
    # All 20 predictors remain represented in the explanation.
    # --------------------------------------------------------

    abs_order = np.argsort(
        np.abs(local_shap_mV)
    )[::-1]

    individual_positions = (
        abs_order[
            :LOCAL_WATERFALL_MAX_DISPLAY - 1
        ]
    )

    remaining_positions = (
        abs_order[
            LOCAL_WATERFALL_MAX_DISPLAY - 1:
        ]
    )

    other_shap_mV = float(
        local_shap_mV[
            remaining_positions
        ].sum()
    )

    # --------------------------------------------------------
    # Build display information
    # --------------------------------------------------------

    display_rows = []

    for feature_position in individual_positions:

        feature = (
            FROZEN_FEATURES[
                feature_position
            ]
        )

        feature_value = float(
            local_feature_values[
                feature_position
            ]
        )

        contribution_mV = float(
            local_shap_mV[
                feature_position
            ]
        )

        display_rows.append({
            "Label":
                f"{feature} = {feature_value:.3f}",
            "SHAP_mV":
                contribution_mV,
            "Type":
                "Feature"
        })

    number_other_features = len(
        remaining_positions
    )

    display_rows.append({
        "Label":
            f"{number_other_features} other features",
        "SHAP_mV":
            other_shap_mV,
        "Type":
            "Other"
    })

    # --------------------------------------------------------
    # Construct additive waterfall path in mV
    # --------------------------------------------------------

    baseline_mV = float(
        FINAL_SHAP_EXPECTED_VALUE
        * 1000
    )

    running_value = baseline_mV

    waterfall_segments = []

    # Smaller displayed effects are plotted first and the
    # largest effects last so that the additive path remains
    # visually interpretable.
    plotting_rows = list(
        reversed(display_rows)
    )

    for row in plotting_rows:

        contribution_mV = float(
            row["SHAP_mV"]
        )

        start_value = float(
            running_value
        )

        end_value = float(
            running_value
            + contribution_mV
        )

        waterfall_segments.append({
            **row,
            "Start_mV":
                start_value,
            "End_mV":
                end_value
        })

        running_value = (
            end_value
        )

    # --------------------------------------------------------
    # Determine plotting limits before drawing annotations
    # --------------------------------------------------------

    all_path_values = [
        baseline_mV,
        predicted_voltage * 1000
    ]

    for segment in waterfall_segments:

        all_path_values.extend([
            segment["Start_mV"],
            segment["End_mV"]
        ])

    path_min = float(
        min(all_path_values)
    )

    path_max = float(
        max(all_path_values)
    )

    path_range = float(
        path_max - path_min
    )

    if path_range == 0:
        path_range = 1.0

    # Reserve whitespace for external contribution labels.
    x_padding = (
        path_range * 0.18
    )

    x_min = (
        path_min - x_padding
    )

    x_max = (
        path_max + x_padding
    )

    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(12.5, 8.0)
    )

    y_positions = np.arange(
        len(waterfall_segments)
    )

    bar_height = 0.58

    # Large contributions are labelled inside the bar.
    # Smaller contributions are labelled outside.
    inside_label_threshold = (
        path_range * 0.085
    )

    outside_offset = (
        path_range * 0.018
    )

    # --------------------------------------------------------
    # Draw waterfall segments
    # --------------------------------------------------------

    for y, segment in zip(
        y_positions,
        waterfall_segments
    ):

        start = float(
            segment["Start_mV"]
        )

        end = float(
            segment["End_mV"]
        )

        contribution = float(
            segment["SHAP_mV"]
        )

        left = min(
            start,
            end
        )

        width = abs(
            contribution
        )

        # ----------------------------------------------------
        # Contribution colour
        # ----------------------------------------------------

        if segment["Type"] == "Other":

            colour = OTHER_COLOUR

        elif contribution >= 0:

            colour = POSITIVE_COLOUR

        else:

            colour = NEGATIVE_COLOUR

        # ----------------------------------------------------
        # Contribution bar
        # ----------------------------------------------------

        ax.barh(
            y,
            width,
            left=left,
            height=bar_height,
            color=colour,
            edgecolor="none",
            zorder=3
        )

        # ----------------------------------------------------
        # Connector between successive additive steps
        # ----------------------------------------------------

        if y < (
            len(waterfall_segments) - 1
        ):

            ax.plot(
                [end, end],
                [
                    y + bar_height / 2,
                    y + 1 - bar_height / 2
                ],
                linestyle=":",
                linewidth=0.8,
                color="0.65",
                zorder=2
            )

        # ----------------------------------------------------
        # SHAP contribution annotation
        # ----------------------------------------------------

        if width >= inside_label_threshold:

            # Large bar:
            # place contribution inside the bar.
            text_x = (
                left + width / 2
            )

            ax.text(
                text_x,
                y,
                f"{contribution:+.2f} mV",
                ha="center",
                va="center",
                fontsize=9,
                color="white",
                zorder=4
            )

        else:

            # Small bar:
            # place contribution outside the bar.
            if contribution >= 0:

                text_x = (
                    end
                    + outside_offset
                )

                horizontal_alignment = (
                    "left"
                )

            else:

                text_x = (
                    end
                    - outside_offset
                )

                horizontal_alignment = (
                    "right"
                )

            ax.text(
                text_x,
                y,
                f"{contribution:+.2f} mV",
                ha=horizontal_alignment,
                va="center",
                fontsize=9,
                color=colour,
                zorder=4
            )

    # --------------------------------------------------------
    # SHAP reference output
    # --------------------------------------------------------

    ax.axvline(
        baseline_mV,
        linestyle="--",
        linewidth=1.2,
        color=REFERENCE_COLOUR,
        zorder=1,
        label=(
            "SHAP reference "
            f"({FINAL_SHAP_EXPECTED_VALUE:.4f} V)"
        )
    )

    # --------------------------------------------------------
    # Final XGBoost prediction
    # --------------------------------------------------------

    ax.axvline(
        predicted_voltage * 1000,
        linestyle="-",
        linewidth=1.4,
        color=PREDICTION_COLOUR,
        zorder=1,
        label=(
            "Model prediction "
            f"({predicted_voltage:.4f} V)"
        )
    )

    # --------------------------------------------------------
    # Predictor labels
    # --------------------------------------------------------

    ax.set_yticks(
        y_positions
    )

    ax.set_yticklabels(
        [
            segment["Label"]
            for segment
            in waterfall_segments
        ],
        fontsize=10
    )

    # --------------------------------------------------------
    # Main figure title
    # --------------------------------------------------------

    fig.suptitle(
        (
            "Representative Local SHAP Explanation "
            f"— {stage} h"
        ),
        fontsize=14,
        y=0.985
    )

    # --------------------------------------------------------
    # Explicit representative-observation information
    #
    # This makes clear that the figure explains ONE
    # observation rather than the complete durability stage.
    # --------------------------------------------------------

    ax.set_title(
        (
            "Single representative observation\n"
            f"Explanation-sample position: {position:,}   |   "
            f"Original dataframe index: {original_index:,}\n"
            f"Actual voltage: {actual_voltage:.4f} V   |   "
            f"Predicted voltage: {predicted_voltage:.4f} V   |   "
            f"Absolute error: {absolute_error_mV:.3f} mV"
        ),
        fontsize=10,
        pad=16,
        linespacing=1.45
    )

    # --------------------------------------------------------
    # Axis labels and limits
    # --------------------------------------------------------

    ax.set_xlabel(
        "Model output / cumulative SHAP contribution (mV)",
        fontsize=11
    )

    ax.set_ylabel(
        "Predictor value for representative observation",
        fontsize=11
    )

    ax.set_xlim(
        x_min,
        x_max
    )

    # --------------------------------------------------------
    # Grid and styling
    # --------------------------------------------------------

    ax.grid(
        axis="x",
        alpha=0.16,
        linewidth=0.8
    )

    ax.set_axisbelow(
        True
    )

    for spine in [
        "top",
        "right",
        "left"
    ]:

        ax.spines[
            spine
        ].set_visible(
            False
        )

    ax.tick_params(
        axis="y",
        length=0,
        pad=6
    )

    # --------------------------------------------------------
    # Legend outside plotting region
    # --------------------------------------------------------

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(
            0.5,
            -0.13
        ),
        ncol=2,
        frameon=False,
        fontsize=9
    )

    # --------------------------------------------------------
    # Reserve independent space for:
    # - long predictor labels
    # - observation metadata
    # - bottom legend
    # --------------------------------------------------------

    plt.subplots_adjust(
        left=0.32,
        right=0.96,
        top=0.81,
        bottom=0.22
    )

    plt.show()


# ============================================================
# B. COMPLETE NUMERICAL LOCAL-ATTRIBUTION TABLE
# ============================================================

local_attribution_rows = []

for stage in LOCAL_EXPLANATION_STAGES:

    position = int(
        representative_positions[
            stage
        ]
    )

    original_index = int(
        final_explanation_data.index[
            position
        ]
    )

    local_shap_mV = (
        final_shap_values[
            position
        ]
        * 1000
    )

    for feature_position, feature in enumerate(
        FROZEN_FEATURES
    ):

        local_attribution_rows.append({

            "Operating_Hour":
                stage,

            "Explanation_Sample_Position":
                position,

            "Original_Dataframe_Index":
                original_index,

            "Feature":
                feature,

            "Feature_Value":
                X_final_explanation.iloc[
                    position,
                    feature_position
                ],

            "SHAP_mV":
                local_shap_mV[
                    feature_position
                ],

            "Abs_SHAP_mV":
                abs(
                    local_shap_mV[
                        feature_position
                    ]
                )
        })

local_attribution_table = pd.DataFrame(
    local_attribution_rows
)


# ============================================================
# C. TOP 10 LOCAL CONTRIBUTIONS PER REPRESENTATIVE OBSERVATION
# ============================================================

top_local_attributions = (
    local_attribution_table
    .sort_values(
        [
            "Operating_Hour",
            "Abs_SHAP_mV"
        ],
        ascending=[
            True,
            False
        ]
    )
    .groupby(
        "Operating_Hour",
        group_keys=False
    )
    .head(
        LOCAL_WATERFALL_MAX_DISPLAY
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# D. REPRESENTATIVE-OBSERVATION SUMMARY TABLE
# ============================================================

representative_observation_summary = (
    representative_local_cases[
        [
            "Operating_Hour",
            "Sample_Position",
            "Original_Index",
            "Actual_Voltage_V",
            "Predicted_Voltage_V",
            "Signed_Error_mV",
            "Absolute_Error_mV",
            "Stage_Median_Absolute_Error_mV",
            "Distance_From_Stage_Median_mV"
        ]
    ]
    .copy()
)

representative_observation_summary = (
    representative_observation_summary
    .rename(
        columns={
            "Sample_Position":
                "Explanation_Sample_Position",

            "Original_Index":
                "Original_Dataframe_Index"
        }
    )
)

representative_observation_summary[
    "Selection_Criterion"
] = (
    "Nearest to stage-specific median absolute prediction error"
)

print("=" * 78)
print("REPRESENTATIVE LOCAL EXPLANATION OBSERVATIONS")
print("=" * 78)

display(
    representative_observation_summary
)


# ============================================================
# E. DISPLAY TOP LOCAL SHAP CONTRIBUTIONS
# ============================================================

print()
print("=" * 78)
print("TOP LOCAL SHAP CONTRIBUTIONS")
print("=" * 78)

for stage in LOCAL_EXPLANATION_STAGES:

    position = int(
        representative_positions[
            stage
        ]
    )

    original_index = int(
        final_explanation_data.index[
            position
        ]
    )

    actual_voltage = float(
        final_actual_voltage[
            position
        ]
    )

    predicted_voltage = float(
        final_explanation_predictions[
            position
        ]
    )

    absolute_error_mV = float(
        final_absolute_prediction_error[
            position
        ]
        * 1000
    )

    stage_median_error_mV = float(
        representative_local_cases.loc[
            representative_local_cases[
                "Operating_Hour"
            ] == stage,
            "Stage_Median_Absolute_Error_mV"
        ].iloc[0]
    )

    distance_from_median_mV = float(
        representative_local_cases.loc[
            representative_local_cases[
                "Operating_Hour"
            ] == stage,
            "Distance_From_Stage_Median_mV"
        ].iloc[0]
    )

    print()
    print("-" * 78)
    print(
        f"REPRESENTATIVE OBSERVATION — {stage} h"
    )
    print("-" * 78)

    print(
        f"Explanation-sample position : "
        f"{position:,}"
    )

    print(
        f"Original dataframe index    : "
        f"{original_index:,}"
    )

    print(
        f"Actual voltage              : "
        f"{actual_voltage:.6f} V"
    )

    print(
        f"Predicted voltage           : "
        f"{predicted_voltage:.6f} V"
    )

    print(
        f"Absolute prediction error   : "
        f"{absolute_error_mV:.3f} mV"
    )

    print(
        f"Stage median absolute error : "
        f"{stage_median_error_mV:.3f} mV"
    )

    print(
        f"Distance from stage median  : "
        f"{distance_from_median_mV:.6f} mV"
    )

    print(
        "Selection criterion         : "
        "Observation nearest to the stage-specific "
        "median absolute prediction error"
    )

    print()
    print(
        "Ten largest local SHAP contributions:"
    )

    display(
        top_local_attributions[
            top_local_attributions[
                "Operating_Hour"
            ] == stage
        ][
            [
                "Feature",
                "Feature_Value",
                "SHAP_mV",
                "Abs_SHAP_mV"
            ]
        ]
        .reset_index(
            drop=True
        )
    )


# ============================================================
# F. VERIFICATION
# ============================================================

complete_local_table = (
    len(local_attribution_table)
    ==
    (
        len(
            LOCAL_EXPLANATION_STAGES
        )
        * len(
            FROZEN_FEATURES
        )
    )
)

correct_top_local_count = all(

    len(
        top_local_attributions[
            top_local_attributions[
                "Operating_Hour"
            ] == stage
        ]
    )
    == LOCAL_WATERFALL_MAX_DISPLAY

    for stage
    in LOCAL_EXPLANATION_STAGES
)

finite_local_shap = np.isfinite(
    local_attribution_table[
        "SHAP_mV"
    ]
).all()

all_local_stages_present = (
    set(
        local_attribution_table[
            "Operating_Hour"
        ]
    )
    ==
    set(
        LOCAL_EXPLANATION_STAGES
    )
)

correct_sample_positions = all(

    int(
        local_attribution_table.loc[
            local_attribution_table[
                "Operating_Hour"
            ] == stage,
            "Explanation_Sample_Position"
        ].iloc[0]
    )
    ==
    int(
        representative_positions[
            stage
        ]
    )

    for stage
    in LOCAL_EXPLANATION_STAGES
)

correct_original_indices = all(

    int(
        local_attribution_table.loc[
            local_attribution_table[
                "Operating_Hour"
            ] == stage,
            "Original_Dataframe_Index"
        ].iloc[0]
    )
    ==
    int(
        final_explanation_data.index[
            representative_positions[
                stage
            ]
        ]
    )

    for stage
    in LOCAL_EXPLANATION_STAGES
)

correct_summary_count = (
    len(
        representative_observation_summary
    )
    == len(
        LOCAL_EXPLANATION_STAGES
    )
)

print()
print("=" * 78)
print("LOCAL WATERFALL VERIFICATION")
print("=" * 78)

print(
    f"Complete 3 × 20 attribution table : "
    f"{complete_local_table}"
)

print(
    f"Top 10 retained for every stage   : "
    f"{correct_top_local_count}"
)

print(
    f"All local SHAP values finite      : "
    f"{finite_local_shap}"
)

print(
    f"All three stages represented      : "
    f"{all_local_stages_present}"
)

print(
    f"Sample positions verified         : "
    f"{correct_sample_positions}"
)

print(
    f"Original dataframe indices verified: "
    f"{correct_original_indices}"
)

print(
    f"Three observation summaries present: "
    f"{correct_summary_count}"
)

if not all([
    complete_local_table,
    correct_top_local_count,
    finite_local_shap,
    all_local_stages_present,
    correct_sample_positions,
    correct_original_indices,
    correct_summary_count
]):

    raise ValueError(
        "Local waterfall verification failed."
    )

print()
print("VERIFICATION PASSED")

print(
    "Each waterfall represents one objectively selected "
    "representative observation, not the complete durability stage."
)

print(
    "Explanation-sample positions and original dataframe indices "
    "have been retained explicitly for traceability."
)

print(
    "The plots and tables use the frozen SHAP values without "
    "recalculating TreeSHAP."
)

<Figure size 1250x800 with 1 Axes>

<Figure size 1250x800 with 1 Axes>

<Figure size 1250x800 with 1 Axes>

REPRESENTATIVE LOCAL EXPLANATION OBSERVATIONS


,Operating_Hour,Explanation_Sample_Position,Original_Dataframe_Index,Actual_Voltage_V,Predicted_Voltage_V,Signed_Error_mV,Absolute_Error_mV,Stage_Median_Absolute_Error_mV,Distance_From_Stage_Median_mV,Selection_Criterion
0,900,811,3201119,0.685200,0.681279,-3.921294,3.921294,3.921509,0.000215,Nearest to stage-specific median absolute pred...
1,950,318,3354208,0.726200,0.730543,4.342719,4.342719,4.341529,0.001191,Nearest to stage-specific median absolute pred...
2,1000,2691,3423797,0.847700,0.841734,-5.966127,5.966127,5.964555,0.001573,Nearest to stage-specific median absolute pred...


...


------------------------------------------------------------------------------
REPRESENTATIVE OBSERVATION — 950 h
------------------------------------------------------------------------------
Explanation-sample position : 318
Original dataframe index    : 3,354,208
Actual voltage              : 0.726200 V
Predicted voltage           : 0.730543 V
Absolute prediction error   : 4.343 mV
Stage median absolute error : 4.342 mV
Distance from stage median  : 0.001191 mV
Selection criterion         : Observation nearest to the stage-specific median absolute prediction error

Ten largest local SHAP contributions:


,Feature,Feature_Value,SHAP_mV,Abs_SHAP_mV
0,current,9.484800,-22.761152,22.761152
1,temp_anode_endplate,83.622147,-7.293118,7.293118
2,cathode_pressure_diff,1.412714,-3.079316,3.079316
3,total_anode_stack_flow,0.132000,2.715101,2.715101
4,anode_temp_diff,-30.533252,1.127648,1.127648
5,temp_anode_dewpoint_water,54.971870,-0.598042,0.598042
6,temp_cathode_outlet,54.821934,-0.568876,0.568876
7,pressure_cathode_outlet,108.792114,-0.502227,0.502227
8,cathode_dewpoint_offset,4.645675,-0.458447,0.458447
9,pressure_anode_outlet,110.428352,-0.388217,0.388217



------------------------------------------------------------------------------
REPRESENTATIVE OBSERVATION — 1000 h
------------------------------------------------------------------------------
Explanation-sample position : 2,691
Original dataframe index    : 3,423,797
Actual voltage              : 0.847700 V
Predicted voltage           : 0.841734 V
Absolute prediction error   : 5.966 mV
Stage median absolute error : 5.965 mV
Distance from stage median  : 0.001573 mV
Selection criterion         : Observation nearest to the stage-specific median absolute prediction error

Ten largest local SHAP contributions:


,Feature,Feature_Value,SHAP_mV,Abs_SHAP_mV
0,current,1.762800,80.447750,80.447750
1,total_anode_stack_flow,0.084000,8.172594,8.172594
2,cathode_pressure_diff,1.008014,-4.813012,4.813012
3,temp_anode_endplate,83.547455,-4.122486,4.122486
4,total_cathode_stack_flow,0.349000,-2.096212,2.096212
5,temp_anode_dewpoint_water,54.890625,1.499157,1.499157
6,temp_cathode_dewpoint_water,64.874649,0.947310,0.947310
7,cathode_dewpoint_offset,4.613319,-0.919863,0.919863
8,anode_temp_diff,-29.198421,0.913934,0.913934
9,pressure_cathode_outlet,108.792114,-0.537933,0.537933



LOCAL WATERFALL VERIFICATION
Complete 3 × 20 attribution table : True
Top 10 retained for every stage   : True
All local SHAP values finite      : True
All three stages represented      : True
Sample positions verified         : True
Original dataframe indices verified: True
Three observation summaries present: True

VERIFICATION PASSED
Each waterfall represents one objectively selected representative observation, not the complete durability stage.
Explanation-sample positions and original dataframe indices have been retained explicitly for traceability.
The plots and tables use the frozen SHAP values without recalculating TreeSHAP.


In [ ]:
## 13.16 TreeSHAP Feature-Dependence Sensitivity Analysis

The primary explainability analysis used interventional TreeSHAP with an explicit 500-observation development-stage background dataset. However, several predictors in the PEMFC dataset are correlated or structurally related, including current, reactant-flow variables, pressure variables, and engineered differential features. SHAP attribution can therefore depend partly on the assumptions used to handle feature dependence.

A secondary tree-path-dependent TreeSHAP analysis is consequently performed as a sensitivity assessment. In this formulation, the fitted tree structure and training-path information are used to represent the model's internal feature distribution rather than the explicit interventional background adopted for the primary analysis.

The purpose of this analysis is not to select whichever SHAP formulation produces the most convenient interpretation. Instead, it assesses whether the principal global attribution conclusions remain broadly stable under an alternative TreeSHAP dependence assumption.

The frozen XGBoost model, exact 20-feature order, and frozen 7,500-observation explanation sample are retained unchanged. The resulting tree-path-dependent attributions will subsequently be compared with the primary interventional results using attribution magnitudes, feature ranks, rank correlation and top-feature overlap.

The interventional formulation remains the primary dissertation analysis. Tree-path-dependent results are treated as a robustness/sensitivity analysis and do not feed back into model development or feature selection.

In [48]:
# ============================================================
# 13.16.1 Tree-Path-Dependent TreeSHAP Computation
# ============================================================

import time

print("=" * 78)
print("TREE-PATH-DEPENDENT SHAP SENSITIVITY ANALYSIS")
print("=" * 78)

print(f"Frozen explanation observations : {len(X_final_explanation):,}")
print(f"Frozen predictors               : {len(FROZEN_FEATURES)}")
print("Primary formulation             : Interventional TreeSHAP")
print("Sensitivity formulation         : Tree-path-dependent TreeSHAP")
print("Model                           : Frozen XGBoost Booster")
print()

# ------------------------------------------------------------
# 1. Construct secondary TreeSHAP explainer
#
# IMPORTANT:
# No explicit background dataset is supplied here.
# Tree-path-dependent TreeSHAP uses information encoded in the
# fitted tree ensemble.
# ------------------------------------------------------------

tree_path_explainer = shap.TreeExplainer(
    frozen_booster,
    feature_perturbation="tree_path_dependent",
    model_output="raw"
)

tree_path_expected_value = float(
    np.asarray(
        tree_path_explainer.expected_value
    ).reshape(-1)[0]
)

print(
    f"Tree-path-dependent expected value : "
    f"{tree_path_expected_value:.9f} V"
)

print(
    f"Primary interventional expected value: "
    f"{FINAL_SHAP_EXPECTED_VALUE:.9f} V"
)

print()

# ------------------------------------------------------------
# 2. Compute SHAP values on EXACTLY the same frozen
#    7,500-observation explanation sample
# ------------------------------------------------------------

tree_path_start_time = time.perf_counter()

tree_path_explanation = tree_path_explainer(
    X_final_explanation,
    check_additivity=True
)

tree_path_runtime_seconds = (
    time.perf_counter()
    - tree_path_start_time
)

tree_path_shap_values = np.asarray(
    tree_path_explanation.values
)

print(
    f"Tree-path-dependent SHAP shape      : "
    f"{tree_path_shap_values.shape}"
)

print(
    f"Runtime                             : "
    f"{tree_path_runtime_seconds:.2f} s "
    f"({tree_path_runtime_seconds / 60:.2f} min)"
)

# ------------------------------------------------------------
# 3. Reconstruct model predictions
# ------------------------------------------------------------

tree_path_reconstructed_predictions = (
    tree_path_expected_value
    + tree_path_shap_values.sum(axis=1)
)

tree_path_reconstruction_difference = (
    tree_path_reconstructed_predictions
    - final_explanation_predictions
)

tree_path_absolute_reconstruction_error = np.abs(
    tree_path_reconstruction_difference
)

# ------------------------------------------------------------
# 4. Additivity summary
# ------------------------------------------------------------

tree_path_additivity_summary = pd.DataFrame({
    "Metric": [
        "Mean absolute reconstruction error",
        "Median absolute reconstruction error",
        "95th percentile absolute reconstruction error",
        "99th percentile absolute reconstruction error",
        "Maximum absolute reconstruction error"
    ],

    "Error_V": [
        np.mean(
            tree_path_absolute_reconstruction_error
        ),

        np.median(
            tree_path_absolute_reconstruction_error
        ),

        np.percentile(
            tree_path_absolute_reconstruction_error,
            95
        ),

        np.percentile(
            tree_path_absolute_reconstruction_error,
            99
        ),

        np.max(
            tree_path_absolute_reconstruction_error
        )
    ]
})

tree_path_additivity_summary[
    "Error_mV"
] = (
    tree_path_additivity_summary[
        "Error_V"
    ] * 1000
)

# ------------------------------------------------------------
# 5. Calculate global mean |SHAP|
# ------------------------------------------------------------

tree_path_mean_abs_shap = pd.Series(
    np.mean(
        np.abs(
            tree_path_shap_values
        ),
        axis=0
    ),
    index=FROZEN_FEATURES,
    name="Mean_Abs_SHAP_V"
)

tree_path_feature_ranking = (
    tree_path_mean_abs_shap
    .sort_values(
        ascending=False
    )
    .reset_index()
    .rename(
        columns={
            "index":
                "Feature"
        }
    )
)

tree_path_feature_ranking[
    "Mean_Abs_SHAP_mV"
] = (
    tree_path_feature_ranking[
        "Mean_Abs_SHAP_V"
    ] * 1000
)

tree_path_feature_ranking[
    "Rank"
] = np.arange(
    1,
    len(
        tree_path_feature_ranking
    ) + 1
)

# ------------------------------------------------------------
# 6. Structural verification
# ------------------------------------------------------------

correct_tree_path_shape = (
    tree_path_shap_values.shape
    ==
    final_shap_values.shape
)

all_tree_path_values_finite = np.isfinite(
    tree_path_shap_values
).all()

tree_path_expected_value_finite = np.isfinite(
    tree_path_expected_value
)

tree_path_reconstruction_finite = np.isfinite(
    tree_path_reconstructed_predictions
).all()

complete_tree_path_ranking = (
    len(
        tree_path_feature_ranking
    )
    ==
    len(
        FROZEN_FEATURES
    )
)

all_tree_path_features_present = (
    set(
        tree_path_feature_ranking[
            "Feature"
        ]
    )
    ==
    set(
        FROZEN_FEATURES
    )
)

# ------------------------------------------------------------
# 7. Output
# ------------------------------------------------------------

print()
print("=" * 78)
print("TREE-PATH-DEPENDENT ADDITIVITY SUMMARY")
print("=" * 78)

display(
    tree_path_additivity_summary
)

print()
print(
    "Tree-path-dependent global feature ranking:"
)

display(
    tree_path_feature_ranking[
        [
            "Rank",
            "Feature",
            "Mean_Abs_SHAP_V",
            "Mean_Abs_SHAP_mV"
        ]
    ]
)

print()
print("=" * 78)
print("TREE-PATH-DEPENDENT STRUCTURAL VERIFICATION")
print("=" * 78)

print(
    f"Correct SHAP matrix shape       : "
    f"{correct_tree_path_shape}"
)

print(
    f"All SHAP values finite          : "
    f"{all_tree_path_values_finite}"
)

print(
    f"Expected value finite           : "
    f"{tree_path_expected_value_finite}"
)

print(
    f"Reconstructions finite          : "
    f"{tree_path_reconstruction_finite}"
)

print(
    f"Complete 20-feature ranking     : "
    f"{complete_tree_path_ranking}"
)

print(
    f"All frozen predictors retained  : "
    f"{all_tree_path_features_present}"
)

if not all([
    correct_tree_path_shape,
    all_tree_path_values_finite,
    tree_path_expected_value_finite,
    tree_path_reconstruction_finite,
    complete_tree_path_ranking,
    all_tree_path_features_present
]):

    raise ValueError(
        "Tree-path-dependent SHAP verification failed."
    )

print()
print("VERIFICATION PASSED")

print(
    "Tree-path-dependent SHAP values have been calculated "
    "for the same frozen 7,500-observation explanation sample."
)

print(
    "No model fitting, feature selection, explanation-sample "
    "selection or primary interventional SHAP calculation "
    "has been changed."
)

TREE-PATH-DEPENDENT SHAP SENSITIVITY ANALYSIS
Frozen explanation observations : 7,500
Frozen predictors               : 20
Primary formulation             : Interventional TreeSHAP
Sensitivity formulation         : Tree-path-dependent TreeSHAP
Model                           : Frozen XGBoost Booster

Tree-path-dependent expected value : 0.757276650 V
Primary interventional expected value: 0.762257170 V

Tree-path-dependent SHAP shape      : (7500, 20)
Runtime                             : 9.14 s (0.15 min)

TREE-PATH-DEPENDENT ADDITIVITY SUMMARY


,Metric,Error_V,Error_mV
0,Mean absolute reconstruction error,0.000002,0.001875
1,Median absolute reconstruction error,0.000002,0.001907
2,95th percentile absolute reconstruction error,0.000002,0.002265
3,99th percentile absolute reconstruction error,0.000002,0.002444
4,Maximum absolute reconstruction error,0.000003,0.002623



Tree-path-dependent global feature ranking:


,Rank,Feature,Mean_Abs_SHAP_V,Mean_Abs_SHAP_mV
0,1,current,0.072049,72.049477
1,2,total_anode_stack_flow,0.012546,12.546186
2,3,temp_anode_endplate,0.005312,5.312314
3,4,cathode_pressure_diff,0.003546,3.545962
4,5,total_cathode_stack_flow,0.001463,1.462942
5,6,temp_anode_dewpoint_water,0.001377,1.377229
6,7,anode_temp_diff,0.001095,1.094522
7,8,pressure_cathode_outlet,0.001073,1.073323
8,9,temp_cathode_dewpoint_water,0.000883,0.882769
9,10,cathode_dewpoint_offset,0.000633,0.632892



TREE-PATH-DEPENDENT STRUCTURAL VERIFICATION
Correct SHAP matrix shape       : True
All SHAP values finite          : True
Expected value finite           : True
Reconstructions finite          : True
Complete 20-feature ranking     : True
All frozen predictors retained  : True

VERIFICATION PASSED
Tree-path-dependent SHAP values have been calculated for the same frozen 7,500-observation explanation sample.
No model fitting, feature selection, explanation-sample selection or primary interventional SHAP calculation has been changed.


In [ ]:
### 13.16.2 Comparison of Interventional and Tree-Path-Dependent Attributions

The secondary tree-path-dependent SHAP results are compared directly with the frozen primary interventional SHAP results using the same 7,500 later-stage observations and the same 20 predictors.

Robustness is assessed at two complementary levels. First, attribution magnitude is compared using mean absolute SHAP values, allowing identification of predictors whose assigned contribution changes under the alternative feature-dependence assumption. Second, attribution structure is compared using feature ranks, Spearman rank correlation, and top-5 and top-10 feature overlap.

Differences between the two formulations are expected because they use different reference and feature-dependence assumptions. Therefore, exact equality is not required for robustness. Particular attention is given to whether the dominant predictors and broader ranking structure remain stable, while magnitude or rank changes among correlated and engineered predictors are treated as evidence that attribution can be redistributed according to the SHAP formulation.

The interventional formulation remains the primary analysis. The tree-path-dependent formulation is used only as a sensitivity analysis and does not replace or modify the frozen primary SHAP results.

In [49]:
# ============================================================
# 13.16.2 Interventional vs Tree-Path-Dependent Comparison
# ============================================================

from scipy.stats import spearmanr

print("=" * 78)
print("INTERVENTIONAL VS TREE-PATH-DEPENDENT SHAP COMPARISON")
print("=" * 78)

# ------------------------------------------------------------
# 1. Primary interventional mean |SHAP|
# ------------------------------------------------------------

interventional_mean_abs = pd.Series(
    np.mean(
        np.abs(final_shap_values),
        axis=0
    ),
    index=FROZEN_FEATURES,
    name="Interventional_Mean_Abs_SHAP_V"
)

# ------------------------------------------------------------
# 2. Secondary tree-path-dependent mean |SHAP|
# ------------------------------------------------------------

tree_path_mean_abs = pd.Series(
    np.mean(
        np.abs(tree_path_shap_values),
        axis=0
    ),
    index=FROZEN_FEATURES,
    name="TreePath_Mean_Abs_SHAP_V"
)

# ------------------------------------------------------------
# 3. Create comparison table
# ------------------------------------------------------------

shap_formulation_comparison = pd.concat(
    [
        interventional_mean_abs,
        tree_path_mean_abs
    ],
    axis=1
)

shap_formulation_comparison[
    "Interventional_Mean_Abs_SHAP_mV"
] = (
    shap_formulation_comparison[
        "Interventional_Mean_Abs_SHAP_V"
    ] * 1000
)

shap_formulation_comparison[
    "TreePath_Mean_Abs_SHAP_mV"
] = (
    shap_formulation_comparison[
        "TreePath_Mean_Abs_SHAP_V"
    ] * 1000
)

# ------------------------------------------------------------
# 4. Calculate ranks independently
# ------------------------------------------------------------

shap_formulation_comparison[
    "Interventional_Rank"
] = (
    shap_formulation_comparison[
        "Interventional_Mean_Abs_SHAP_V"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

shap_formulation_comparison[
    "TreePath_Rank"
] = (
    shap_formulation_comparison[
        "TreePath_Mean_Abs_SHAP_V"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

# ------------------------------------------------------------
# 5. Magnitude and rank differences
# ------------------------------------------------------------

shap_formulation_comparison[
    "Magnitude_Difference_mV"
] = (
    shap_formulation_comparison[
        "TreePath_Mean_Abs_SHAP_mV"
    ]
    -
    shap_formulation_comparison[
        "Interventional_Mean_Abs_SHAP_mV"
    ]
)

shap_formulation_comparison[
    "Absolute_Magnitude_Difference_mV"
] = np.abs(
    shap_formulation_comparison[
        "Magnitude_Difference_mV"
    ]
)

shap_formulation_comparison[
    "Relative_Magnitude_Change_Percent"
] = (
    shap_formulation_comparison[
        "Magnitude_Difference_mV"
    ]
    /
    shap_formulation_comparison[
        "Interventional_Mean_Abs_SHAP_mV"
    ]
    * 100
)

shap_formulation_comparison[
    "Rank_Change_TreePath_Minus_Interventional"
] = (
    shap_formulation_comparison[
        "TreePath_Rank"
    ]
    -
    shap_formulation_comparison[
        "Interventional_Rank"
    ]
)

shap_formulation_comparison[
    "Absolute_Rank_Change"
] = np.abs(
    shap_formulation_comparison[
        "Rank_Change_TreePath_Minus_Interventional"
    ]
)

# ------------------------------------------------------------
# 6. Sort using the PRIMARY interventional ranking
# ------------------------------------------------------------

shap_formulation_comparison = (
    shap_formulation_comparison
    .reset_index()
    .rename(
        columns={
            "index": "Feature"
        }
    )
    .sort_values(
        "Interventional_Rank"
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7. Rank correlation
# ------------------------------------------------------------

spearman_rho, spearman_p = spearmanr(
    shap_formulation_comparison[
        "Interventional_Rank"
    ],
    shap_formulation_comparison[
        "TreePath_Rank"
    ]
)

# ------------------------------------------------------------
# 8. Top-k overlap
# ------------------------------------------------------------

interventional_top5 = set(
    shap_formulation_comparison
    .nsmallest(
        5,
        "Interventional_Rank"
    )["Feature"]
)

tree_path_top5 = set(
    shap_formulation_comparison
    .nsmallest(
        5,
        "TreePath_Rank"
    )["Feature"]
)

interventional_top10 = set(
    shap_formulation_comparison
    .nsmallest(
        10,
        "Interventional_Rank"
    )["Feature"]
)

tree_path_top10 = set(
    shap_formulation_comparison
    .nsmallest(
        10,
        "TreePath_Rank"
    )["Feature"]
)

top5_overlap = len(
    interventional_top5
    & tree_path_top5
)

top10_overlap = len(
    interventional_top10
    & tree_path_top10
)

# ------------------------------------------------------------
# 9. Summary magnitude statistics
# ------------------------------------------------------------

mean_absolute_magnitude_difference = float(
    shap_formulation_comparison[
        "Absolute_Magnitude_Difference_mV"
    ].mean()
)

median_absolute_magnitude_difference = float(
    shap_formulation_comparison[
        "Absolute_Magnitude_Difference_mV"
    ].median()
)

max_difference_row = (
    shap_formulation_comparison
    .loc[
        shap_formulation_comparison[
            "Absolute_Magnitude_Difference_mV"
        ].idxmax()
    ]
)

largest_absolute_rank_change = int(
    shap_formulation_comparison[
        "Absolute_Rank_Change"
    ].max()
)

# ------------------------------------------------------------
# 10. Expected-value difference
# ------------------------------------------------------------

expected_value_difference_mV = (
    (
        tree_path_expected_value
        - FINAL_SHAP_EXPECTED_VALUE
    )
    * 1000
)

# ------------------------------------------------------------
# 11. Compact robustness summary
# ------------------------------------------------------------

formulation_robustness_summary = pd.DataFrame({
    "Metric": [
        "Interventional expected value (V)",
        "Tree-path-dependent expected value (V)",
        "Expected-value difference (mV)",
        "Spearman rank correlation",
        "Top-5 overlap",
        "Top-10 overlap",
        "Mean absolute mean-|SHAP| difference (mV)",
        "Median absolute mean-|SHAP| difference (mV)",
        "Largest absolute rank change"
    ],

    "Value": [
        FINAL_SHAP_EXPECTED_VALUE,
        tree_path_expected_value,
        expected_value_difference_mV,
        spearman_rho,
        f"{top5_overlap}/5",
        f"{top10_overlap}/10",
        mean_absolute_magnitude_difference,
        median_absolute_magnitude_difference,
        largest_absolute_rank_change
    ]
})

# ------------------------------------------------------------
# 12. Output
# ------------------------------------------------------------

print()
print("Global formulation-comparison table:")
print()

display(
    shap_formulation_comparison[
        [
            "Feature",
            "Interventional_Mean_Abs_SHAP_mV",
            "TreePath_Mean_Abs_SHAP_mV",
            "Magnitude_Difference_mV",
            "Relative_Magnitude_Change_Percent",
            "Interventional_Rank",
            "TreePath_Rank",
            "Rank_Change_TreePath_Minus_Interventional"
        ]
    ]
)

print()
print("=" * 78)
print("FORMULATION ROBUSTNESS SUMMARY")
print("=" * 78)

display(
    formulation_robustness_summary
)

print()
print(
    f"Feature with largest absolute magnitude difference : "
    f"{max_difference_row['Feature']}"
)

print(
    f"Interventional mean |SHAP|                         : "
    f"{max_difference_row['Interventional_Mean_Abs_SHAP_mV']:.6f} mV"
)

print(
    f"Tree-path-dependent mean |SHAP|                    : "
    f"{max_difference_row['TreePath_Mean_Abs_SHAP_mV']:.6f} mV"
)

print(
    f"Absolute magnitude difference                      : "
    f"{max_difference_row['Absolute_Magnitude_Difference_mV']:.6f} mV"
)

# ------------------------------------------------------------
# 13. Top-feature membership
# ------------------------------------------------------------

print()
print("=" * 78)
print("TOP-FEATURE MEMBERSHIP")
print("=" * 78)

print(
    "Interventional top 5 : "
    + ", ".join(
        shap_formulation_comparison
        .nsmallest(
            5,
            "Interventional_Rank"
        )["Feature"]
        .tolist()
    )
)

print(
    "Tree-path top 5      : "
    + ", ".join(
        shap_formulation_comparison
        .nsmallest(
            5,
            "TreePath_Rank"
        )["Feature"]
        .tolist()
    )
)

print()

print(
    "Interventional top 10: "
    + ", ".join(
        shap_formulation_comparison
        .nsmallest(
            10,
            "Interventional_Rank"
        )["Feature"]
        .tolist()
    )
)

print(
    "Tree-path top 10     : "
    + ", ".join(
        shap_formulation_comparison
        .nsmallest(
            10,
            "TreePath_Rank"
        )["Feature"]
        .tolist()
    )
)

# ------------------------------------------------------------
# 14. Structural verification
# ------------------------------------------------------------

complete_comparison = (
    len(
        shap_formulation_comparison
    )
    ==
    len(
        FROZEN_FEATURES
    )
)

all_features_present = (
    set(
        shap_formulation_comparison[
            "Feature"
        ]
    )
    ==
    set(
        FROZEN_FEATURES
    )
)

all_comparison_values_finite = np.isfinite(
    shap_formulation_comparison[
        [
            "Interventional_Mean_Abs_SHAP_mV",
            "TreePath_Mean_Abs_SHAP_mV",
            "Magnitude_Difference_mV",
            "Relative_Magnitude_Change_Percent",
            "Interventional_Rank",
            "TreePath_Rank"
        ]
    ].to_numpy()
).all()

spearman_valid = np.isfinite(
    spearman_rho
)

top_overlap_valid = (
    0 <= top5_overlap <= 5
    and
    0 <= top10_overlap <= 10
)

print()
print("=" * 78)
print("FORMULATION-COMPARISON VERIFICATION")
print("=" * 78)

print(
    f"Complete 20-feature comparison : "
    f"{complete_comparison}"
)

print(
    f"All frozen predictors present  : "
    f"{all_features_present}"
)

print(
    f"All comparison values finite   : "
    f"{all_comparison_values_finite}"
)

print(
    f"Spearman correlation valid     : "
    f"{spearman_valid}"
)

print(
    f"Top-k overlaps valid           : "
    f"{top_overlap_valid}"
)

if not all([
    complete_comparison,
    all_features_present,
    all_comparison_values_finite,
    spearman_valid,
    top_overlap_valid
]):

    raise ValueError(
        "SHAP formulation comparison verification failed."
    )

print()
print("VERIFICATION PASSED")

print(
    "Interventional and tree-path-dependent SHAP formulations "
    "have been compared using the identical frozen model, "
    "feature set and 7,500-observation explanation sample."
)

INTERVENTIONAL VS TREE-PATH-DEPENDENT SHAP COMPARISON

Global formulation-comparison table:



,Feature,Interventional_Mean_Abs_SHAP_mV,TreePath_Mean_Abs_SHAP_mV,Magnitude_Difference_mV,Relative_Magnitude_Change_Percent,Interventional_Rank,TreePath_Rank,Rank_Change_TreePath_Minus_Interventional
0,current,79.238701,72.049477,-7.189225,-9.072871,1,1,0
1,total_anode_stack_flow,9.775657,12.546186,2.770530,28.341113,2,2,0
2,temp_anode_endplate,6.044815,5.312314,-0.732502,-12.117850,3,3,0
3,cathode_pressure_diff,5.903888,3.545962,-2.357927,-39.938540,4,4,0
4,total_cathode_stack_flow,1.696977,1.462942,-0.234035,-13.791314,5,5,0
5,temp_anode_dewpoint_water,1.299153,1.377229,0.078076,6.009739,6,6,0
6,pressure_cathode_outlet,1.229881,1.073323,-0.156558,-12.729504,7,8,1
7,anode_temp_diff,1.118551,1.094522,-0.024029,-2.148226,8,7,-1
8,cathode_dewpoint_offset,0.904716,0.632892,-0.271825,-30.045287,9,10,1
9,temp_cathode_dewpoint_water,0.832284,0.882769,0.050486,6.065930,10,9,-1



FORMULATION ROBUSTNESS SUMMARY


,Metric,Value
0,Interventional expected value (V),0.762257
1,Tree-path-dependent expected value (V),0.757277
2,Expected-value difference (mV),-4.980520
3,Spearman rank correlation,0.975940
4,Top-5 overlap,5/5
5,Top-10 overlap,10/10
6,Mean absolute mean-|SHAP| difference (mV),0.725311
7,Median absolute mean-|SHAP| difference (mV),0.100406
8,Largest absolute rank change,4



Feature with largest absolute magnitude difference : current
Interventional mean |SHAP|                         : 79.238701 mV
Tree-path-dependent mean |SHAP|                    : 72.049477 mV
Absolute magnitude difference                      : 7.189225 mV

TOP-FEATURE MEMBERSHIP
Interventional top 5 : current, total_anode_stack_flow, temp_anode_endplate, cathode_pressure_diff, total_cathode_stack_flow
Tree-path top 5      : current, total_anode_stack_flow, temp_anode_endplate, cathode_pressure_diff, total_cathode_stack_flow

Interventional top 10: current, total_anode_stack_flow, temp_anode_endplate, cathode_pressure_diff, total_cathode_stack_flow, temp_anode_dewpoint_water, pressure_cathode_outlet, anode_temp_diff, cathode_dewpoint_offset, temp_cathode_dewpoint_water
Tree-path top 10     : current, total_anode_stack_flow, temp_anode_endplate, cathode_pressure_diff, total_cathode_stack_flow, temp_anode_dewpoint_water, anode_temp_diff, pressure_cathode_outlet, temp_cathode_dewpoint_w

In [ ]:
### 13.16.3 Feature-Dependence Sensitivity Summary

The sensitivity comparison indicated that the principal global attribution hierarchy was highly stable across the two TreeSHAP formulations. The interventional and tree-path-dependent rankings had a Spearman rank correlation of 0.976, while all five highest-ranked predictors and all ten highest-ranked predictors were retained under both formulations. In particular, current remained the dominant predictor, followed by total anode stack flow, anode endplate temperature and cathode pressure difference.

The exact attribution magnitudes were nevertheless formulation-dependent. For example, the mean absolute attribution of current decreased under the tree-path-dependent formulation, whereas the attribution assigned to total anode stack flow increased. Cathode pressure difference also showed a noticeable reduction. The expected model-output reference differed between the two formulations by approximately 4.98 mV.

These results indicate that the broad model-attribution structure is robust to the alternative TreeSHAP dependence assumption, while the precise allocation of contribution among individual predictors is not invariant. This distinction is particularly relevant because several operational and engineered PEMFC predictors are correlated or structurally related. Consequently, the SHAP results are interpreted primarily in terms of robust predictor hierarchy and model behaviour rather than as uniquely identifiable physical or causal effects.

The interventional TreeSHAP analysis remains the primary explanation framework because it was specified a priori with an explicit development-stage reference background. The tree-path-dependent analysis is retained as a sensitivity assessment and does not alter the frozen primary SHAP results.

In [50]:
# ============================================================
# 13.16.3 Final Feature-Dependence Sensitivity Summary
# ============================================================

final_formulation_sensitivity_summary = pd.DataFrame({
    "Measure": [
        "Primary SHAP formulation",
        "Sensitivity formulation",
        "Interventional expected value (V)",
        "Tree-path-dependent expected value (V)",
        "Expected-value difference (mV)",
        "Spearman rank correlation",
        "Top-5 feature overlap",
        "Top-10 feature overlap",
        "Mean absolute mean-|SHAP| difference (mV)",
        "Median absolute mean-|SHAP| difference (mV)",
        "Largest absolute rank change"
    ],

    "Result": [
        "Interventional TreeSHAP",
        "Tree-path-dependent TreeSHAP",
        f"{FINAL_SHAP_EXPECTED_VALUE:.6f}",
        f"{tree_path_expected_value:.6f}",
        f"{expected_value_difference_mV:.6f}",
        f"{spearman_rho:.6f}",
        f"{top5_overlap}/5",
        f"{top10_overlap}/10",
        f"{mean_absolute_magnitude_difference:.6f}",
        f"{median_absolute_magnitude_difference:.6f}",
        str(largest_absolute_rank_change)
    ]
})

print("=" * 78)
print("FINAL TREE-SHAP FORMULATION SENSITIVITY SUMMARY")
print("=" * 78)

display(
    final_formulation_sensitivity_summary
)

# ------------------------------------------------------------
# Dominant-feature stability
# ------------------------------------------------------------

dominant_feature_stability = (
    shap_formulation_comparison[
        [
            "Feature",
            "Interventional_Mean_Abs_SHAP_mV",
            "TreePath_Mean_Abs_SHAP_mV",
            "Magnitude_Difference_mV",
            "Interventional_Rank",
            "TreePath_Rank"
        ]
    ]
    .head(10)
    .copy()
)

print()
print("Top-10 predictor sensitivity:")
display(
    dominant_feature_stability
)

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

dominant_top5_identical = (
    top5_overlap == 5
)

dominant_top10_identical = (
    top10_overlap == 10
)

high_rank_agreement = (
    spearman_rho > 0.90
)

print()
print("=" * 78)
print("SENSITIVITY CONCLUSION")
print("=" * 78)

print(
    f"Identical top-5 membership       : "
    f"{dominant_top5_identical}"
)

print(
    f"Identical top-10 membership      : "
    f"{dominant_top10_identical}"
)

print(
    f"High overall rank agreement      : "
    f"{high_rank_agreement}"
)

print()
print(
    "Conclusion: The dominant SHAP attribution hierarchy is "
    "robust to the alternative TreeSHAP feature-dependence "
    "formulation, although individual attribution magnitudes "
    "show formulation sensitivity."
)

print()
print(
    "The interventional formulation remains the frozen primary "
    "SHAP analysis; tree-path-dependent results are retained "
    "only as robustness evidence."
)

FINAL TREE-SHAP FORMULATION SENSITIVITY SUMMARY


,Measure,Result
0,Primary SHAP formulation,Interventional TreeSHAP
1,Sensitivity formulation,Tree-path-dependent TreeSHAP
2,Interventional expected value (V),0.762257
3,Tree-path-dependent expected value (V),0.757277
4,Expected-value difference (mV),-4.980520
5,Spearman rank correlation,0.975940
6,Top-5 feature overlap,5/5
7,Top-10 feature overlap,10/10
8,Mean absolute mean-|SHAP| difference (mV),0.725311
9,Median absolute mean-|SHAP| difference (mV),0.100406



Top-10 predictor sensitivity:


,Feature,Interventional_Mean_Abs_SHAP_mV,TreePath_Mean_Abs_SHAP_mV,Magnitude_Difference_mV,Interventional_Rank,TreePath_Rank
0,current,79.238701,72.049477,-7.189225,1,1
1,total_anode_stack_flow,9.775657,12.546186,2.770530,2,2
2,temp_anode_endplate,6.044815,5.312314,-0.732502,3,3
3,cathode_pressure_diff,5.903888,3.545962,-2.357927,4,4
4,total_cathode_stack_flow,1.696977,1.462942,-0.234035,5,5
5,temp_anode_dewpoint_water,1.299153,1.377229,0.078076,6,6
6,pressure_cathode_outlet,1.229881,1.073323,-0.156558,7,8
7,anode_temp_diff,1.118551,1.094522,-0.024029,8,7
8,cathode_dewpoint_offset,0.904716,0.632892,-0.271825,9,10
9,temp_cathode_dewpoint_water,0.832284,0.882769,0.050486,10,9



SENSITIVITY CONCLUSION
Identical top-5 membership       : True
Identical top-10 membership      : True
High overall rank agreement      : True

Conclusion: The dominant SHAP attribution hierarchy is robust to the alternative TreeSHAP feature-dependence formulation, although individual attribution magnitudes show formulation sensitivity.

The interventional formulation remains the frozen primary SHAP analysis; tree-path-dependent results are retained only as robustness evidence.


In [ ]:
==============================================================================
FINAL TREE-SHAP FORMULATION SENSITIVITY SUMMARY
==============================================================================
Measure	Result
0	Primary SHAP formulation	Interventional TreeSHAP
1	Sensitivity formulation	Tree-path-dependent TreeSHAP
2	Interventional expected value (V)	0.762257
3	Tree-path-dependent expected value (V)	0.757277
4	Expected-value difference (mV)	-4.980520
5	Spearman rank correlation	0.975940
6	Top-5 feature overlap	5/5
7	Top-10 feature overlap	10/10
8	Mean absolute mean-|SHAP| difference (mV)	0.725311
9	Median absolute mean-|SHAP| difference (mV)	0.100406
10	Largest absolute rank change	4

Top-10 predictor sensitivity:
Feature	Interventional_Mean_Abs_SHAP_mV	TreePath_Mean_Abs_SHAP_mV	Magnitude_Difference_mV	Interventional_Rank	TreePath_Rank
0	current	79.238701	72.049477	-7.189225	1	1
1	total_anode_stack_flow	9.775657	12.546186	2.770530	2	2
2	temp_anode_endplate	6.044815	5.312314	-0.732502	3	3
3	cathode_pressure_diff	5.903888	3.545962	-2.357927	4	4
4	total_cathode_stack_flow	1.696977	1.462942	-0.234035	5	5
5	temp_anode_dewpoint_water	1.299153	1.377229	0.078076	6	6
6	pressure_cathode_outlet	1.229881	1.073323	-0.156558	7	8
7	anode_temp_diff	1.118551	1.094522	-0.024029	8	7
8	cathode_dewpoint_offset	0.904716	0.632892	-0.271825	9	10
9	temp_cathode_dewpoint_water	0.832284	0.882769	0.050486	10	9

==============================================================================
SENSITIVITY CONCLUSION
==============================================================================
Identical top-5 membership       : True
Identical top-10 membership      : True
High overall rank agreement      : True

Conclusion: The dominant SHAP attribution hierarchy is robust to the alternative TreeSHAP feature-dependence formulation, although individual attribution magnitudes show formulation sensitivity.

The interventional formulation remains the frozen primary SHAP analysis; tree-path-dependent results are retained only as robustness evidence.

In [51]:
# ============================================================
# 13.17 Consolidated SHAP Robustness Assessment
# ============================================================

print("=" * 78)
print("CONSOLIDATED SHAP ROBUSTNESS ASSESSMENT")
print("=" * 78)

# ------------------------------------------------------------
# 1. Background-size robustness
# ------------------------------------------------------------

# Final retained background = 500
# Diagnostic reference       = 1000

background_robustness = {
    "Assessment":
        "Background-size stability",

    "Primary_Configuration":
        "500 development observations",

    "Comparison":
        "vs 1,000-observation diagnostic reference",

    "Spearman_Rho":
        0.996992,

    "Top5_Overlap":
        "5/5",

    "Top10_Overlap":
        "10/10",

    "Mean_Abs_Difference_mV":
        0.061789,

    "Maximum_Difference_mV":
        0.518433,

    "Interpretation":
        "Global attribution structure stable"
}


# ------------------------------------------------------------
# 2. Explanation-sample-size convergence
# ------------------------------------------------------------

# Final retained sample      = 7,500
# Diagnostic reference       = 30,000

explanation_size_robustness = {
    "Assessment":
        "Explanation-sample convergence",

    "Primary_Configuration":
        "7,500 holdout observations",

    "Comparison":
        "vs 30,000-observation diagnostic reference",

    "Spearman_Rho":
        1.000000,

    "Top5_Overlap":
        "5/5",

    "Top10_Overlap":
        "10/10",

    "Mean_Abs_Difference_mV":
        0.029389,

    "Maximum_Difference_mV":
        0.352407,

    "Interpretation":
        "Complete global ranking reproduced"
}


# ------------------------------------------------------------
# 3. Repeated-seed robustness
# ------------------------------------------------------------

# Primary seed = 42
# Robustness seeds = 123 and 2026

seed_robustness_rows = [
    {
        "Assessment":
            "Repeated-seed stability",

        "Primary_Configuration":
            "Seed 42",

        "Comparison":
            "vs seed 123",

        "Spearman_Rho":
            0.995489,

        "Top5_Overlap":
            "5/5",

        "Top10_Overlap":
            "10/10",

        "Mean_Abs_Difference_mV":
            0.051464,

        "Maximum_Difference_mV":
            0.718341,

        "Interpretation":
            "Dominant attribution structure stable"
    },

    {
        "Assessment":
            "Repeated-seed stability",

        "Primary_Configuration":
            "Seed 42",

        "Comparison":
            "vs seed 2026",

        "Spearman_Rho":
            0.996992,

        "Top5_Overlap":
            "5/5",

        "Top10_Overlap":
            "10/10",

        "Mean_Abs_Difference_mV":
            0.043212,

        "Maximum_Difference_mV":
            0.455068,

        "Interpretation":
            "Dominant attribution structure stable"
    }
]


# ------------------------------------------------------------
# 4. TreeSHAP formulation sensitivity
# ------------------------------------------------------------

formulation_robustness = {
    "Assessment":
        "Feature-dependence sensitivity",

    "Primary_Configuration":
        "Interventional TreeSHAP",

    "Comparison":
        "vs tree-path-dependent TreeSHAP",

    "Spearman_Rho":
        float(spearman_rho),

    "Top5_Overlap":
        f"{top5_overlap}/5",

    "Top10_Overlap":
        f"{top10_overlap}/10",

    "Mean_Abs_Difference_mV":
        float(
            mean_absolute_magnitude_difference
        ),

    "Maximum_Difference_mV":
        float(
            shap_formulation_comparison[
                "Absolute_Magnitude_Difference_mV"
            ].max()
        ),

    "Interpretation":
        (
            "Hierarchy stable; exact attribution "
            "magnitudes formulation-sensitive"
        )
}


# ------------------------------------------------------------
# 5. Combine ranking/magnitude robustness evidence
# ------------------------------------------------------------

robustness_rows = [
    background_robustness,
    explanation_size_robustness,
    *seed_robustness_rows,
    formulation_robustness
]

consolidated_shap_robustness = pd.DataFrame(
    robustness_rows
)

print()
print("Global attribution robustness:")
print()

display(
    consolidated_shap_robustness
)


# ------------------------------------------------------------
# 6. Additivity evidence
# ------------------------------------------------------------

final_reconstruction_error = np.abs(
    (
        FINAL_SHAP_EXPECTED_VALUE
        + final_shap_values.sum(axis=1)
    )
    - final_explanation_predictions
)

additivity_robustness_summary = pd.DataFrame({
    "Measure": [
        "Explanation observations",
        "Mean absolute reconstruction error (mV)",
        "Median absolute reconstruction error (mV)",
        "95th percentile reconstruction error (mV)",
        "99th percentile reconstruction error (mV)",
        "Maximum reconstruction error (mV)",
        "Observations above 0.1 mV",
        "Observations above 0.5 mV",
        "Observations above 1.0 mV"
    ],

    "Result": [
        len(
            final_explanation_predictions
        ),

        np.mean(
            final_reconstruction_error
        ) * 1000,

        np.median(
            final_reconstruction_error
        ) * 1000,

        np.percentile(
            final_reconstruction_error,
            95
        ) * 1000,

        np.percentile(
            final_reconstruction_error,
            99
        ) * 1000,

        np.max(
            final_reconstruction_error
        ) * 1000,

        int(
            np.sum(
                final_reconstruction_error
                > 0.0001
            )
        ),

        int(
            np.sum(
                final_reconstruction_error
                > 0.0005
            )
        ),

        int(
            np.sum(
                final_reconstruction_error
                > 0.001
            )
        )
    ]
})

print()
print("=" * 78)
print("FINAL SHAP ADDITIVITY ROBUSTNESS")
print("=" * 78)

display(
    additivity_robustness_summary
)


# ------------------------------------------------------------
# 7. Final frozen configuration
# ------------------------------------------------------------

final_shap_configuration = pd.DataFrame({
    "Component": [
        "Explained model",
        "Primary SHAP formulation",
        "Reference population",
        "Background size",
        "Explanation population",
        "Explanation sample size",
        "900 h observations",
        "950 h observations",
        "1000 h observations",
        "Primary random seed",
        "Predictor count"
    ],

    "Final_Configuration": [
        "Frozen development-selected XGBoost model",
        "Interventional TreeSHAP",
        "Development stages 50–850 h",
        FINAL_BACKGROUND_N,
        "Later-stage holdout 900–1000 h",
        FINAL_EXPLANATION_N,
        int(
            final_explanation_allocation.loc[900]
        ),
        int(
            final_explanation_allocation.loc[950]
        ),
        int(
            final_explanation_allocation.loc[1000]
        ),
        FINAL_EXPLANATION_SEED,
        len(
            FROZEN_FEATURES
        )
    ]
})

print()
print("=" * 78)
print("FINAL FROZEN SHAP CONFIGURATION")
print("=" * 78)

display(
    final_shap_configuration
)


# ------------------------------------------------------------
# 8. Consolidated verification
# ------------------------------------------------------------

background_top5_stable = (
    background_robustness[
        "Top5_Overlap"
    ] == "5/5"
)

background_top10_stable = (
    background_robustness[
        "Top10_Overlap"
    ] == "10/10"
)

explanation_ranking_stable = (
    explanation_size_robustness[
        "Spearman_Rho"
    ] == 1.0
)

seed_top10_stable = all(
    row["Top10_Overlap"] == "10/10"
    for row in seed_robustness_rows
)

formulation_top5_stable = (
    top5_overlap == 5
)

formulation_top10_stable = (
    top10_overlap == 10
)

all_final_shap_finite = np.isfinite(
    final_shap_values
).all()

all_additivity_errors_finite = np.isfinite(
    final_reconstruction_error
).all()

correct_final_sample_size = (
    len(
        final_explanation_data
    )
    == FINAL_EXPLANATION_N
)

correct_final_background_size = (
    len(
        X_final_background
    )
    == FINAL_BACKGROUND_N
)

correct_final_stage_allocation = all(
    int(
        final_explanation_allocation.loc[
            stage
        ]
    ) == 2500

    for stage in [
        900,
        950,
        1000
    ]
)

print()
print("=" * 78)
print("CONSOLIDATED ROBUSTNESS VERIFICATION")
print("=" * 78)

print(
    f"Background top-5 stable          : "
    f"{background_top5_stable}"
)

print(
    f"Background top-10 stable         : "
    f"{background_top10_stable}"
)

print(
    f"Explanation ranking stable       : "
    f"{explanation_ranking_stable}"
)

print(
    f"Repeated-seed top-10 stable      : "
    f"{seed_top10_stable}"
)

print(
    f"Formulation top-5 stable         : "
    f"{formulation_top5_stable}"
)

print(
    f"Formulation top-10 stable        : "
    f"{formulation_top10_stable}"
)

print(
    f"All final SHAP values finite     : "
    f"{all_final_shap_finite}"
)

print(
    f"All additivity errors finite     : "
    f"{all_additivity_errors_finite}"
)

print(
    f"Final explanation size correct   : "
    f"{correct_final_sample_size}"
)

print(
    f"Final background size correct    : "
    f"{correct_final_background_size}"
)

print(
    f"Stage allocation 2500 each       : "
    f"{correct_final_stage_allocation}"
)

if not all([
    background_top5_stable,
    background_top10_stable,
    explanation_ranking_stable,
    seed_top10_stable,
    formulation_top5_stable,
    formulation_top10_stable,
    all_final_shap_finite,
    all_additivity_errors_finite,
    correct_final_sample_size,
    correct_final_background_size,
    correct_final_stage_allocation
]):

    raise ValueError(
        "Consolidated SHAP robustness verification failed."
    )

print()
print("VERIFICATION PASSED")

print(
    "The final SHAP configuration is supported by "
    "background-size, explanation-size, repeated-seed, "
    "additivity and feature-dependence sensitivity checks."
)

print(
    "These checks support stability of the dominant model-"
    "attribution hierarchy for this study; they do not "
    "establish causal or uniquely identifiable physical effects."
)

CONSOLIDATED SHAP ROBUSTNESS ASSESSMENT

Global attribution robustness:



,Assessment,Primary_Configuration,Comparison,Spearman_Rho,Top5_Overlap,Top10_Overlap,Mean_Abs_Difference_mV,Maximum_Difference_mV,Interpretation
0,Background-size stability,500 development observations,"vs 1,000-observation diagnostic reference",0.996992,5/5,10/10,0.061789,0.518433,Global attribution structure stable
1,Explanation-sample convergence,"7,500 holdout observations","vs 30,000-observation diagnostic reference",1.000000,5/5,10/10,0.029389,0.352407,Complete global ranking reproduced
2,Repeated-seed stability,Seed 42,vs seed 123,0.995489,5/5,10/10,0.051464,0.718341,Dominant attribution structure stable
3,Repeated-seed stability,Seed 42,vs seed 2026,0.996992,5/5,10/10,0.043212,0.455068,Dominant attribution structure stable
4,Feature-dependence sensitivity,Interventional TreeSHAP,vs tree-path-dependent TreeSHAP,0.975940,5/5,10/10,0.725311,7.189225,Hierarchy stable; exact attribution magnitudes...



FINAL SHAP ADDITIVITY ROBUSTNESS


,Measure,Result
0,Explanation observations,7500.000000
1,Mean absolute reconstruction error (mV),0.001263
2,Median absolute reconstruction error (mV),0.000194
3,95th percentile reconstruction error (mV),0.000555
4,99th percentile reconstruction error (mV),0.000713
5,Maximum reconstruction error (mV),4.656416
6,Observations above 0.1 mV,15.000000
7,Observations above 0.5 mV,1.000000
8,Observations above 1.0 mV,1.000000



FINAL FROZEN SHAP CONFIGURATION


,Component,Final_Configuration
0,Explained model,Frozen development-selected XGBoost model
1,Primary SHAP formulation,Interventional TreeSHAP
2,Reference population,Development stages 50–850 h
3,Background size,500
4,Explanation population,Later-stage holdout 900–1000 h
5,Explanation sample size,7500
6,900 h observations,2500
7,950 h observations,2500
8,1000 h observations,2500
9,Primary random seed,42



CONSOLIDATED ROBUSTNESS VERIFICATION
Background top-5 stable          : True
Background top-10 stable         : True
Explanation ranking stable       : True
Repeated-seed top-10 stable      : True
Formulation top-5 stable         : True
Formulation top-10 stable        : True
All final SHAP values finite     : True
All additivity errors finite     : True
Final explanation size correct   : True
Final background size correct    : True
Stage allocation 2500 each       : True

VERIFICATION PASSED
The final SHAP configuration is supported by background-size, explanation-size, repeated-seed, additivity and feature-dependence sensitivity checks.
These checks support stability of the dominant model-attribution hierarchy for this study; they do not establish causal or uniquely identifiable physical effects.


In [ ]:
## 13.18 Final SHAP Results and Reproducibility Export

Following completion of the primary SHAP analysis and all robustness assessments, the final numerical outputs are exported to reproducible CSV files. No model fitting, resampling, feature selection or SHAP recalculation is performed in this section.

The exported files preserve the frozen global feature-importance results, stage-wise attribution results, representative local explanations, robustness assessments, final SHAP configuration, additivity diagnostics and software environment. A manifest is also created to document the purpose of each exported file.

These exports provide a traceable numerical record from which dissertation tables, figures and appendix material can subsequently be prepared without rerunning the computationally expensive SHAP analyses.

In [52]:
# ============================================================
# 13.18.1 Export Frozen Core SHAP Results
# ============================================================

from pathlib import Path

print("=" * 78)
print("FINAL SHAP RESULTS AND REPRODUCIBILITY EXPORT")
print("=" * 78)

# ------------------------------------------------------------
# 1. Confirm export directory
# ------------------------------------------------------------

SHAP_EXPORT_DIR = Path(
    PROJECT_ROOT
) / "results" / "shap_analysis" / "exports"

SHAP_EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Export directory:\n{SHAP_EXPORT_DIR}"
)

# ------------------------------------------------------------
# 2. Prepare final global ranking
# ------------------------------------------------------------

final_global_shap_export = (
    pd.DataFrame({
        "Feature":
            FROZEN_FEATURES,

        "Mean_Abs_SHAP_V":
            np.mean(
                np.abs(final_shap_values),
                axis=0
            ),

        "Mean_Abs_SHAP_mV":
            np.mean(
                np.abs(final_shap_values),
                axis=0
            ) * 1000,

        "Mean_Signed_SHAP_V":
            np.mean(
                final_shap_values,
                axis=0
            ),

        "Mean_Signed_SHAP_mV":
            np.mean(
                final_shap_values,
                axis=0
            ) * 1000
    })
    .sort_values(
        "Mean_Abs_SHAP_V",
        ascending=False
    )
    .reset_index(drop=True)
)

final_global_shap_export.insert(
    0,
    "Rank",
    np.arange(
        1,
        len(final_global_shap_export) + 1
    )
)

total_mean_abs_shap = (
    final_global_shap_export[
        "Mean_Abs_SHAP_V"
    ].sum()
)

final_global_shap_export[
    "Relative_Mean_Abs_SHAP_Percent"
] = (
    final_global_shap_export[
        "Mean_Abs_SHAP_V"
    ]
    / total_mean_abs_shap
    * 100
)

# ------------------------------------------------------------
# 3. Prepare final stage-wise importance table directly
#    from the already frozen SHAP matrix
#
# This does NOT recompute SHAP. It only aggregates the
# existing final_shap_values by operating stage.
# ------------------------------------------------------------

stagewise_export_rows = []

for stage in [900, 950, 1000]:

    stage_mask = (
        final_explanation_data[
            STAGE_COLUMN
        ].to_numpy()
        == stage
    )

    stage_values = (
        final_shap_values[
            stage_mask
        ]
    )

    stage_mean_abs = np.mean(
        np.abs(stage_values),
        axis=0
    )

    stage_table = pd.DataFrame({
        "Operating_Hour":
            stage,

        "Feature":
            FROZEN_FEATURES,

        "Mean_Abs_SHAP_V":
            stage_mean_abs,

        "Mean_Abs_SHAP_mV":
            stage_mean_abs * 1000
    })

    stage_table[
        "Stage_Rank"
    ] = (
        stage_table[
            "Mean_Abs_SHAP_V"
        ]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(int)
    )

    stagewise_export_rows.append(
        stage_table
    )

final_stagewise_shap_export = pd.concat(
    stagewise_export_rows,
    ignore_index=True
)

final_stagewise_shap_export = (
    final_stagewise_shap_export
    .sort_values(
        [
            "Operating_Hour",
            "Stage_Rank"
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Prepare formulation sensitivity export
# ------------------------------------------------------------

final_formulation_sensitivity_export = (
    shap_formulation_comparison.copy()
)

# ------------------------------------------------------------
# 5. Prepare robustness export
# ------------------------------------------------------------

final_robustness_export = (
    consolidated_shap_robustness.copy()
)

# ------------------------------------------------------------
# 6. Prepare configuration export
# ------------------------------------------------------------

final_configuration_export = (
    final_shap_configuration.copy()
)

# ------------------------------------------------------------
# 7. Prepare additivity export
# ------------------------------------------------------------

final_additivity_export = (
    additivity_robustness_summary.copy()
)

# ------------------------------------------------------------
# 8. Export core tables
# ------------------------------------------------------------

export_objects = {
    "shap_global_feature_importance.csv":
        final_global_shap_export,

    "shap_stagewise_feature_importance.csv":
        final_stagewise_shap_export,

    "shap_feature_dependence_sensitivity.csv":
        final_formulation_sensitivity_export,

    "shap_consolidated_robustness.csv":
        final_robustness_export,

    "shap_final_configuration.csv":
        final_configuration_export,

    "shap_additivity_summary.csv":
        final_additivity_export
}

export_records = []

for filename, dataframe in export_objects.items():

    filepath = (
        SHAP_EXPORT_DIR
        / filename
    )

    dataframe.to_csv(
        filepath,
        index=False
    )

    export_records.append({
        "File":
            filename,

        "Rows":
            len(dataframe),

        "Columns":
            len(dataframe.columns),

        "Path":
            str(filepath)
    })

# ------------------------------------------------------------
# 9. Verify exported global ranking
# ------------------------------------------------------------

global_ranking_valid = (
    len(final_global_shap_export)
    == len(FROZEN_FEATURES)
)

global_features_valid = (
    set(
        final_global_shap_export[
            "Feature"
        ]
    )
    ==
    set(FROZEN_FEATURES)
)

global_values_finite = np.isfinite(
    final_global_shap_export[
        [
            "Mean_Abs_SHAP_V",
            "Mean_Abs_SHAP_mV",
            "Mean_Signed_SHAP_V",
            "Mean_Signed_SHAP_mV",
            "Relative_Mean_Abs_SHAP_Percent"
        ]
    ].to_numpy()
).all()

# ------------------------------------------------------------
# 10. Verify stage-wise export
# ------------------------------------------------------------

stagewise_rows_valid = (
    len(
        final_stagewise_shap_export
    )
    ==
    3 * len(FROZEN_FEATURES)
)

stagewise_stages_valid = (
    set(
        final_stagewise_shap_export[
            "Operating_Hour"
        ].unique()
    )
    ==
    {900, 950, 1000}
)

stagewise_feature_counts_valid = all(
    (
        final_stagewise_shap_export[
            final_stagewise_shap_export[
                "Operating_Hour"
            ] == stage
        ]["Feature"].nunique()
        ==
        len(FROZEN_FEATURES)
    )

    for stage in [
        900,
        950,
        1000
    ]
)

# ------------------------------------------------------------
# 11. Verify files exist
# ------------------------------------------------------------

all_export_files_exist = all(
    (
        SHAP_EXPORT_DIR
        / filename
    ).exists()

    for filename in export_objects
)

# ------------------------------------------------------------
# 12. Output
# ------------------------------------------------------------

export_summary = pd.DataFrame(
    export_records
)

print()
print("Exported core SHAP files:")
print()

display(
    export_summary
)

print()
print("Final global SHAP ranking:")
print()

display(
    final_global_shap_export
)

print()
print("=" * 78)
print("CORE EXPORT VERIFICATION")
print("=" * 78)

print(
    f"Global ranking has 20 predictors : "
    f"{global_ranking_valid}"
)

print(
    f"Global predictor membership valid: "
    f"{global_features_valid}"
)

print(
    f"Global values finite              : "
    f"{global_values_finite}"
)

print(
    f"Stage-wise table has 60 rows      : "
    f"{stagewise_rows_valid}"
)

print(
    f"Stage-wise stages correct         : "
    f"{stagewise_stages_valid}"
)

print(
    f"20 predictors present per stage   : "
    f"{stagewise_feature_counts_valid}"
)

print(
    f"All core export files exist       : "
    f"{all_export_files_exist}"
)

if not all([
    global_ranking_valid,
    global_features_valid,
    global_values_finite,
    stagewise_rows_valid,
    stagewise_stages_valid,
    stagewise_feature_counts_valid,
    all_export_files_exist
]):

    raise ValueError(
        "Core SHAP export verification failed."
    )

print()
print("VERIFICATION PASSED")

print(
    "Frozen core SHAP results have been exported "
    "without model refitting, resampling or SHAP recalculation."
)

FINAL SHAP RESULTS AND REPRODUCIBILITY EXPORT
Export directory:
C:\Users\usman\Desktop\PEMFC_Dissertation\results\shap_analysis\exports

Exported core SHAP files:



,File,Rows,Columns,Path
0,shap_global_feature_importance.csv,20,7,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
1,shap_stagewise_feature_importance.csv,60,5,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
2,shap_feature_dependence_sensitivity.csv,20,12,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
3,shap_consolidated_robustness.csv,5,9,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
4,shap_final_configuration.csv,11,2,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
5,shap_additivity_summary.csv,9,2,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...



Final global SHAP ranking:



,Rank,Feature,Mean_Abs_SHAP_V,Mean_Abs_SHAP_mV,Mean_Signed_SHAP_V,Mean_Signed_SHAP_mV,Relative_Mean_Abs_SHAP_Percent
0,1,current,0.079239,79.238701,-0.006389,-6.389181,71.260017
1,2,total_anode_stack_flow,0.009776,9.775657,-0.001310,-1.309827,8.791328
2,3,temp_anode_endplate,0.006045,6.044815,-0.005367,-5.367449,5.436152
3,4,cathode_pressure_diff,0.005904,5.903888,0.000018,0.018139,5.309416
4,5,total_cathode_stack_flow,0.001697,1.696977,-0.000298,-0.298355,1.526105
5,6,temp_anode_dewpoint_water,0.001299,1.299153,0.000221,0.221387,1.168339
6,7,pressure_cathode_outlet,0.001230,1.229881,-0.000057,-0.057076,1.106042
7,8,anode_temp_diff,0.001119,1.118551,0.000623,0.622663,1.005922
8,9,cathode_dewpoint_offset,0.000905,0.904716,0.000064,0.064493,0.813619
9,10,temp_cathode_dewpoint_water,0.000832,0.832284,0.000164,0.163965,0.748479



CORE EXPORT VERIFICATION
Global ranking has 20 predictors : True
Global predictor membership valid: True
Global values finite              : True
Stage-wise table has 60 rows      : True
Stage-wise stages correct         : True
20 predictors present per stage   : True
All core export files exist       : True

VERIFICATION PASSED
Frozen core SHAP results have been exported without model refitting, resampling or SHAP recalculation.


In [ ]:
### 13.18.2 Export of Representative Local Explanations

The three representative local explanations selected previously are exported using their verified original engineered-dataset indices. One observation is retained from each later durability stage (900, 950 and 1,000 h).

These observations were selected objectively as the explanation-sample cases whose absolute prediction errors were closest to the median absolute prediction error within their respective stages. The cases are not reselected in this section. Their original dataframe indices are used to recover the corresponding rows from the frozen 7,500-observation explanation sample and the already-computed SHAP matrix.

For each representative observation, the export records the durability stage, position within the frozen explanation sample, original dataframe index, measured voltage, XGBoost prediction, prediction error, predictor value and local SHAP contribution for all 20 predictors. No SHAP values are recalculated.

In [53]:
# ============================================================
# 13.18.2 Export Representative Local SHAP Explanations
# ============================================================

print("=" * 78)
print("REPRESENTATIVE LOCAL SHAP EXPORT")
print("=" * 78)

# ------------------------------------------------------------
# 1. Frozen representative observations
#
# These are the original dataframe indices already selected
# and verified in the local explanation analysis.
# ------------------------------------------------------------

REPRESENTATIVE_LOCAL_CASES = {
    900: 3201119,
    950: 3354208,
    1000: 3423797
}

# ------------------------------------------------------------
# 2. Recover each case from the frozen explanation sample
# ------------------------------------------------------------

local_summary_rows = []
local_contribution_rows = []

for stage, original_index in REPRESENTATIVE_LOCAL_CASES.items():

    matching_positions = np.where(
        final_explanation_data.index.to_numpy()
        == original_index
    )[0]

    if len(matching_positions) != 1:
        raise ValueError(
            f"Expected exactly one frozen explanation observation "
            f"for original index {original_index}, but found "
            f"{len(matching_positions)}."
        )

    global_position = int(
        matching_positions[0]
    )

    observed_stage = int(
        final_explanation_data.iloc[
            global_position
        ][STAGE_COLUMN]
    )

    if observed_stage != stage:
        raise ValueError(
            f"Stage mismatch for original index {original_index}: "
            f"expected {stage}, found {observed_stage}."
        )

    actual_voltage = float(
        final_explanation_data.iloc[
            global_position
        ][TARGET]
    )

    predicted_voltage = float(
        final_explanation_predictions[
            global_position
        ]
    )

    signed_error_v = (
        predicted_voltage
        - actual_voltage
    )

    absolute_error_v = abs(
        signed_error_v
    )

    # --------------------------------------------------------
    # Position within its stage-specific frozen subset
    # --------------------------------------------------------

    stage_positions = np.where(
        final_explanation_data[
            STAGE_COLUMN
        ].to_numpy()
        == stage
    )[0]

    stage_local_matches = np.where(
        stage_positions
        == global_position
    )[0]

    if len(stage_local_matches) != 1:
        raise ValueError(
            f"Could not uniquely determine stage-specific "
            f"position for stage {stage}."
        )

    stage_sample_position = int(
        stage_local_matches[0]
    )

    # --------------------------------------------------------
    # Additivity reconstruction for this local case
    # --------------------------------------------------------

    reconstructed_voltage = float(
        FINAL_SHAP_EXPECTED_VALUE
        + final_shap_values[
            global_position
        ].sum()
    )

    reconstruction_difference_v = (
        reconstructed_voltage
        - predicted_voltage
    )

    # --------------------------------------------------------
    # Summary row
    # --------------------------------------------------------

    local_summary_rows.append({
        "Operating_Hour":
            stage,

        "Explanation_Sample_Global_Position":
            global_position,

        "Stage_Specific_Sample_Position":
            stage_sample_position,

        "Original_DataFrame_Index":
            original_index,

        "Actual_Voltage_V":
            actual_voltage,

        "Predicted_Voltage_V":
            predicted_voltage,

        "Signed_Prediction_Error_mV":
            signed_error_v * 1000,

        "Absolute_Prediction_Error_mV":
            absolute_error_v * 1000,

        "SHAP_Expected_Value_V":
            FINAL_SHAP_EXPECTED_VALUE,

        "SHAP_Reconstructed_Prediction_V":
            reconstructed_voltage,

        "SHAP_Reconstruction_Difference_mV":
            reconstruction_difference_v * 1000
    })

    # --------------------------------------------------------
    # Full 20-feature local contribution table
    # --------------------------------------------------------

    for feature_position, feature in enumerate(
        FROZEN_FEATURES
    ):

        feature_value = float(
            X_final_explanation.iloc[
                global_position
            ][feature]
        )

        shap_value_v = float(
            final_shap_values[
                global_position,
                feature_position
            ]
        )

        local_contribution_rows.append({
            "Operating_Hour":
                stage,

            "Explanation_Sample_Global_Position":
                global_position,

            "Stage_Specific_Sample_Position":
                stage_sample_position,

            "Original_DataFrame_Index":
                original_index,

            "Feature":
                feature,

            "Feature_Value":
                feature_value,

            "SHAP_Value_V":
                shap_value_v,

            "SHAP_Value_mV":
                shap_value_v * 1000,

            "Absolute_SHAP_Value_mV":
                abs(shap_value_v) * 1000
        })


# ------------------------------------------------------------
# 3. Construct export tables
# ------------------------------------------------------------

representative_local_summary_export = pd.DataFrame(
    local_summary_rows
)

representative_local_contributions_export = pd.DataFrame(
    local_contribution_rows
)

representative_local_contributions_export[
    "Local_Absolute_Rank"
] = (
    representative_local_contributions_export
    .groupby(
        "Operating_Hour"
    )[
        "Absolute_SHAP_Value_mV"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

representative_local_contributions_export = (
    representative_local_contributions_export
    .sort_values(
        [
            "Operating_Hour",
            "Local_Absolute_Rank"
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Export
# ------------------------------------------------------------

local_summary_path = (
    SHAP_EXPORT_DIR
    / "shap_representative_local_cases_summary.csv"
)

local_contributions_path = (
    SHAP_EXPORT_DIR
    / "shap_representative_local_contributions.csv"
)

representative_local_summary_export.to_csv(
    local_summary_path,
    index=False
)

representative_local_contributions_export.to_csv(
    local_contributions_path,
    index=False
)


# ------------------------------------------------------------
# 5. Verification
# ------------------------------------------------------------

three_cases_present = (
    len(
        representative_local_summary_export
    )
    == 3
)

correct_stages_present = (
    set(
        representative_local_summary_export[
            "Operating_Hour"
        ]
    )
    ==
    {900, 950, 1000}
)

correct_indices_present = (
    set(
        representative_local_summary_export[
            "Original_DataFrame_Index"
        ]
    )
    ==
    set(
        REPRESENTATIVE_LOCAL_CASES.values()
    )
)

sixty_local_rows_present = (
    len(
        representative_local_contributions_export
    )
    ==
    3 * len(FROZEN_FEATURES)
)

twenty_features_per_case = all(
    (
        representative_local_contributions_export[
            representative_local_contributions_export[
                "Operating_Hour"
            ] == stage
        ][
            "Feature"
        ].nunique()
        ==
        len(FROZEN_FEATURES)
    )

    for stage in [
        900,
        950,
        1000
    ]
)

local_values_finite = np.isfinite(
    representative_local_contributions_export[
        [
            "Feature_Value",
            "SHAP_Value_V",
            "SHAP_Value_mV",
            "Absolute_SHAP_Value_mV"
        ]
    ].to_numpy()
).all()

local_files_exist = (
    local_summary_path.exists()
    and
    local_contributions_path.exists()
)


# ------------------------------------------------------------
# 6. Output
# ------------------------------------------------------------

print()
print("Representative local cases:")
print()

display(
    representative_local_summary_export
)

print()
print("Full local SHAP contributions:")
print()

display(
    representative_local_contributions_export
)

print()
print("=" * 78)
print("LOCAL EXPORT VERIFICATION")
print("=" * 78)

print(
    f"Exactly three local cases       : "
    f"{three_cases_present}"
)

print(
    f"Stages 900/950/1000 present     : "
    f"{correct_stages_present}"
)

print(
    f"Frozen original indices correct : "
    f"{correct_indices_present}"
)

print(
    f"Exactly 60 contribution rows    : "
    f"{sixty_local_rows_present}"
)

print(
    f"20 predictors per local case    : "
    f"{twenty_features_per_case}"
)

print(
    f"All numerical values finite     : "
    f"{local_values_finite}"
)

print(
    f"Both local export files exist   : "
    f"{local_files_exist}"
)

if not all([
    three_cases_present,
    correct_stages_present,
    correct_indices_present,
    sixty_local_rows_present,
    twenty_features_per_case,
    local_values_finite,
    local_files_exist
]):

    raise ValueError(
        "Representative local SHAP export verification failed."
    )

print()
print("VERIFICATION PASSED")

print(
    "The three previously selected representative observations "
    "and their complete 20-feature local SHAP contributions "
    "have been exported without SHAP recalculation."
)

REPRESENTATIVE LOCAL SHAP EXPORT

Representative local cases:



,Operating_Hour,Explanation_Sample_Global_Position,Stage_Specific_Sample_Position,Original_DataFrame_Index,Actual_Voltage_V,Predicted_Voltage_V,Signed_Prediction_Error_mV,Absolute_Prediction_Error_mV,SHAP_Expected_Value_V,SHAP_Reconstructed_Prediction_V,SHAP_Reconstruction_Difference_mV
0,900,811,289,3201119,0.685200,0.681279,-3.921294,3.921294,0.762257,0.681279,-0.000097
1,950,318,98,3354208,0.726200,0.730543,4.342719,4.342719,0.762257,0.730543,0.000442
2,1000,2691,900,3423797,0.847700,0.841734,-5.966127,5.966127,0.762257,0.841734,0.000085



Full local SHAP contributions:



,Operating_Hour,Explanation_Sample_Global_Position,Stage_Specific_Sample_Position,Original_DataFrame_Index,Feature,Feature_Value,SHAP_Value_V,SHAP_Value_mV,Absolute_SHAP_Value_mV,Local_Absolute_Rank
0,900,811,289,3201119,current,14.813900,-0.070068,-70.067958,70.067958,1
1,900,811,289,3201119,temp_anode_endplate,84.057602,-0.009037,-9.036605,9.036605,2
2,900,811,289,3201119,total_anode_stack_flow,0.207000,-0.008686,-8.685954,8.685954,3
3,900,811,289,3201119,cathode_pressure_diff,2.527103,0.005297,5.296952,5.296952,4
4,900,811,289,3201119,total_cathode_stack_flow,0.864000,0.002420,2.420000,2.420000,5
5,900,811,289,3201119,pressure_cathode_outlet,107.273025,-0.001335,-1.335034,1.335034,6
6,900,811,289,3201119,anode_temp_diff,-31.761028,0.001142,1.142150,1.142150,7
7,900,811,289,3201119,temp_anode_dewpoint_water,55.015720,-0.001117,-1.117237,1.117237,8
8,900,811,289,3201119,cathode_dewpoint_offset,5.754249,0.000902,0.902112,0.902112,9
9,900,811,289,3201119,temp_anode_outlet,38.057400,0.000616,0.615899,0.615899,10



LOCAL EXPORT VERIFICATION
Exactly three local cases       : True
Stages 900/950/1000 present     : True
Frozen original indices correct : True
Exactly 60 contribution rows    : True
20 predictors per local case    : True
All numerical values finite     : True
Both local export files exist   : True

VERIFICATION PASSED
The three previously selected representative observations and their complete 20-feature local SHAP contributions have been exported without SHAP recalculation.


In [ ]:
### 13.18.3 Reproducibility Manifest and Figure Audit

A final reproducibility manifest is created to catalogue the numerical SHAP outputs and graphical artefacts generated during the analysis. The manifest records file names, locations, file types and their analytical purpose.

The figure directory is also audited for PNG and SVG outputs. Existing vector-format figures are retained directly. Raster figures are not automatically converted into SVG because such conversion would not recreate genuine vector graphics. Where vector output is required, the original plotting code should instead be used to save the figure directly in SVG format.

This section performs only file inventory and documentation. It does not recompute SHAP values, regenerate model predictions, resample observations or alter any frozen analytical result.

In [54]:
# ============================================================
# 13.18.3 Reproducibility Manifest and Figure Audit
# ============================================================

print("=" * 78)
print("SHAP REPRODUCIBILITY MANIFEST AND FIGURE AUDIT")
print("=" * 78)

# ------------------------------------------------------------
# 1. Define figure directory
# ------------------------------------------------------------

SHAP_FIGURE_DIR = (
    Path(PROJECT_ROOT)
    / "figures"
    / "shap_analysis"
)

SHAP_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Expected numerical exports
# ------------------------------------------------------------

numerical_manifest_entries = [
    {
        "Category": "Numerical result",
        "File": "shap_global_feature_importance.csv",
        "Purpose":
            "Final global mean absolute and signed SHAP "
            "importance for all 20 predictors."
    },
    {
        "Category": "Numerical result",
        "File": "shap_stagewise_feature_importance.csv",
        "Purpose":
            "Stage-wise mean absolute SHAP importance for "
            "900, 950 and 1000 h."
    },
    {
        "Category": "Robustness",
        "File": "shap_feature_dependence_sensitivity.csv",
        "Purpose":
            "Interventional versus tree-path-dependent "
            "TreeSHAP comparison."
    },
    {
        "Category": "Robustness",
        "File": "shap_consolidated_robustness.csv",
        "Purpose":
            "Consolidated background-size, explanation-size, "
            "seed and formulation robustness evidence."
    },
    {
        "Category": "Configuration",
        "File": "shap_final_configuration.csv",
        "Purpose":
            "Frozen final SHAP analytical configuration."
    },
    {
        "Category": "Verification",
        "File": "shap_additivity_summary.csv",
        "Purpose":
            "Final SHAP prediction-reconstruction diagnostics."
    },
    {
        "Category": "Local explanation",
        "File": "shap_representative_local_cases_summary.csv",
        "Purpose":
            "Metadata and prediction information for the "
            "three representative local observations."
    },
    {
        "Category": "Local explanation",
        "File": "shap_representative_local_contributions.csv",
        "Purpose":
            "Complete 20-feature local SHAP contributions "
            "for the representative 900, 950 and 1000 h cases."
    }
]

# ------------------------------------------------------------
# 3. Add software-environment file if present
# ------------------------------------------------------------

environment_file = (
    Path(PROJECT_ROOT)
    / "results"
    / "shap_analysis"
    / "shap_software_environment.csv"
)

if environment_file.exists():

    numerical_manifest_entries.append({
        "Category":
            "Reproducibility",

        "File":
            environment_file.name,

        "Purpose":
            "Python and package versions used for the "
            "SHAP analysis."
    })

# ------------------------------------------------------------
# 4. Build numerical manifest with file verification
# ------------------------------------------------------------

numerical_manifest_rows = []

for entry in numerical_manifest_entries:

    if entry["File"] == environment_file.name:

        filepath = environment_file

    else:

        filepath = (
            SHAP_EXPORT_DIR
            / entry["File"]
        )

    numerical_manifest_rows.append({
        "Category":
            entry["Category"],

        "File":
            entry["File"],

        "File_Type":
            filepath.suffix.lower(),

        "Exists":
            filepath.exists(),

        "Size_KB":
            (
                filepath.stat().st_size / 1024
                if filepath.exists()
                else np.nan
            ),

        "Purpose":
            entry["Purpose"],

        "Path":
            str(filepath)
    })

numerical_manifest = pd.DataFrame(
    numerical_manifest_rows
)

# ------------------------------------------------------------
# 5. Audit all SHAP figures
# ------------------------------------------------------------

figure_extensions = {
    ".png",
    ".svg",
    ".pdf",
    ".jpg",
    ".jpeg"
}

figure_files = sorted(
    [
        path
        for path in SHAP_FIGURE_DIR.iterdir()
        if (
            path.is_file()
            and
            path.suffix.lower()
            in figure_extensions
        )
    ],
    key=lambda p: p.name.lower()
)

figure_manifest_rows = []

for filepath in figure_files:

    figure_manifest_rows.append({
        "Category":
            "Figure",

        "File":
            filepath.name,

        "File_Type":
            filepath.suffix.lower(),

        "Exists":
            True,

        "Size_KB":
            filepath.stat().st_size / 1024,

        "Purpose":
            "SHAP analytical figure generated in Notebook 13.",

        "Path":
            str(filepath)
    })

figure_manifest = pd.DataFrame(
    figure_manifest_rows
)

# ------------------------------------------------------------
# 6. Determine PNG/SVG coverage
# ------------------------------------------------------------

png_stems = {
    path.stem
    for path in figure_files
    if path.suffix.lower() == ".png"
}

svg_stems = {
    path.stem
    for path in figure_files
    if path.suffix.lower() == ".svg"
}

png_with_svg = sorted(
    png_stems
    & svg_stems
)

png_without_svg = sorted(
    png_stems
    - svg_stems
)

svg_without_png = sorted(
    svg_stems
    - png_stems
)

figure_format_audit = pd.DataFrame({
    "Measure": [
        "Total figure files",
        "PNG files",
        "SVG files",
        "PNG figures with matching SVG",
        "PNG figures without matching SVG",
        "SVG figures without matching PNG"
    ],

    "Count": [
        len(figure_files),
        len(png_stems),
        len(svg_stems),
        len(png_with_svg),
        len(png_without_svg),
        len(svg_without_png)
    ]
})

# ------------------------------------------------------------
# 7. Combine manifest
# ------------------------------------------------------------

if len(figure_manifest) > 0:

    final_reproducibility_manifest = pd.concat(
        [
            numerical_manifest,
            figure_manifest
        ],
        ignore_index=True
    )

else:

    final_reproducibility_manifest = (
        numerical_manifest.copy()
    )

# ------------------------------------------------------------
# 8. Export manifest
# ------------------------------------------------------------

manifest_path = (
    SHAP_EXPORT_DIR
    / "shap_reproducibility_manifest.csv"
)

final_reproducibility_manifest.to_csv(
    manifest_path,
    index=False
)

# ------------------------------------------------------------
# 9. Output
# ------------------------------------------------------------

print()
print("Numerical/reproducibility files:")
print()

display(
    numerical_manifest
)

print()
print("=" * 78)
print("FIGURE FORMAT AUDIT")
print("=" * 78)

display(
    figure_format_audit
)

if len(figure_manifest) > 0:

    print()
    print("Detected SHAP figures:")
    print()

    display(
        figure_manifest[
            [
                "File",
                "File_Type",
                "Size_KB",
                "Path"
            ]
        ]
    )

else:

    print()
    print(
        "No figure files were detected in the "
        "SHAP figure directory."
    )

# ------------------------------------------------------------
# 10. SVG audit details
# ------------------------------------------------------------

if png_without_svg:

    print()
    print(
        "PNG figures currently without matching SVG:"
    )

    for filename_stem in png_without_svg:
        print(
            f"  - {filename_stem}"
        )

else:

    print()
    print(
        "All detected PNG figure stems have matching SVG files."
    )

if svg_without_png:

    print()
    print(
        "SVG figures without matching PNG:"
    )

    for filename_stem in svg_without_png:
        print(
            f"  - {filename_stem}"
        )

# ------------------------------------------------------------
# 11. Final verification
# ------------------------------------------------------------

all_numerical_files_exist = bool(
    numerical_manifest[
        "Exists"
    ].all()
)

manifest_exists = (
    manifest_path.exists()
)

manifest_nonempty = (
    len(
        final_reproducibility_manifest
    ) > 0
)

all_manifest_paths_present = (
    final_reproducibility_manifest[
        "Path"
    ]
    .notna()
    .all()
)

print()
print("=" * 78)
print("FINAL EXPORT VERIFICATION")
print("=" * 78)

print(
    f"All expected numerical files exist : "
    f"{all_numerical_files_exist}"
)

print(
    f"Reproducibility manifest exists    : "
    f"{manifest_exists}"
)

print(
    f"Manifest contains records          : "
    f"{manifest_nonempty}"
)

print(
    f"All manifest paths recorded        : "
    f"{all_manifest_paths_present}"
)

if not all([
    all_numerical_files_exist,
    manifest_exists,
    manifest_nonempty,
    all_manifest_paths_present
]):

    raise ValueError(
        "Final SHAP export verification failed."
    )

print()
print("VERIFICATION PASSED")

print(
    "The frozen SHAP numerical outputs and available figures "
    "have been catalogued in the reproducibility manifest."
)

print(
    "No SHAP values, predictions or model parameters were "
    "recomputed or modified during this audit."
)

SHAP REPRODUCIBILITY MANIFEST AND FIGURE AUDIT

Numerical/reproducibility files:



,Category,File,File_Type,Exists,Size_KB,Purpose,Path
0,Numerical result,shap_global_feature_importance.csv,.csv,True,2.606445,Final global mean absolute and signed SHAP imp...,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
1,Numerical result,shap_stagewise_feature_importance.csv,.csv,True,4.128906,Stage-wise mean absolute SHAP importance for 9...,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
2,Robustness,shap_feature_dependence_sensitivity.csv,.csv,True,3.338867,Interventional versus tree-path-dependent Tree...,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
3,Robustness,shap_consolidated_robustness.csv,.csv,True,0.932617,"Consolidated background-size, explanation-size...",C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
4,Configuration,shap_final_configuration.csv,.csv,True,0.410156,Frozen final SHAP analytical configuration.,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
5,Verification,shap_additivity_summary.csv,.csv,True,0.443359,Final SHAP prediction-reconstruction diagnostics.,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
6,Local explanation,shap_representative_local_cases_summary.csv,.csv,True,0.700195,Metadata and prediction information for the th...,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
7,Local explanation,shap_representative_local_contributions.csv,.csv,True,7.100586,Complete 20-feature local SHAP contributions f...,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...
8,Reproducibility,shap_software_environment.csv,.csv,True,0.174805,Python and package versions used for the SHAP ...,C:\Users\usman\Desktop\PEMFC_Dissertation\resu...



FIGURE FORMAT AUDIT


,Measure,Count
0,Total figure files,0
1,PNG files,0
2,SVG files,0
3,PNG figures with matching SVG,0
4,PNG figures without matching SVG,0
5,SVG figures without matching PNG,0



No figure files were detected in the SHAP figure directory.

All detected PNG figure stems have matching SVG files.

FINAL EXPORT VERIFICATION
All expected numerical files exist : True
Reproducibility manifest exists    : True
Manifest contains records          : True
All manifest paths recorded        : True

VERIFICATION PASSED
The frozen SHAP numerical outputs and available figures have been catalogued in the reproducibility manifest.
No SHAP values, predictions or model parameters were recomputed or modified during this audit.


In [ ]:
## 13.19 Notebook 13 Summary and Interpretation Boundaries

### 13.19.1 Purpose of the Analysis

This notebook applied SHAP (SHapley Additive exPlanations) to the frozen XGBoost model selected previously through chronological development-stage validation. The purpose was not to perform further feature selection or improve predictive performance, but to interpret how the trained model used the 20 previously selected operational predictors when estimating PEMFC stack voltage at unseen later durability stages.

The primary explanation population consisted of observations from the independent 900, 950 and 1,000 h holdout stages. SHAP analysis was conducted only after model development, hyperparameter decisions and predictive evaluation had been frozen. Consequently, the holdout explanations were used for post-hoc model interpretation and were not fed back into model training, tuning or feature selection.

### 13.19.2 Final SHAP Configuration

The primary analysis used interventional TreeSHAP applied to the frozen development-selected XGBoost model.

The final configuration consisted of:

- 20 frozen predictors in the exact feature order expected by the trained XGBoost model.
- A 500-observation stage-balanced reference background sampled from the 50–850 h development period.
- A 7,500-observation stage-balanced explanation sample from the later-stage holdout period.
- 2,500 explanation observations each from 900, 950 and 1,000 h.
- Random seed 42 for the primary explanation sample.
- Interventional TreeSHAP as the primary feature-dependence formulation.
- Tree-path-dependent TreeSHAP as a secondary sensitivity analysis.

The interventional expected model output for the selected development reference background was 0.762257 V. This value represents the model's reference output under the selected SHAP background distribution; it must not be interpreted as a healthy-state, beginning-of-life or physically defined PEMFC reference voltage.

### 13.19.3 Global Model Attribution

Current was by far the largest global contributor to variation in the XGBoost voltage predictions, with a mean absolute SHAP magnitude of 79.239 mV. It represented approximately 71.26% of the total mean absolute SHAP magnitude across the 20 predictors.

The next most influential predictors were:

1. Current — 79.239 mV
2. Total anode stack flow — 9.776 mV
3. Anode endplate temperature — 6.045 mV
4. Cathode pressure difference — 5.904 mV
5. Total cathode stack flow — 1.697 mV
6. Anode dewpoint-water temperature — 1.299 mV
7. Cathode outlet pressure — 1.230 mV
8. Anode temperature difference — 1.119 mV
9. Cathode dewpoint offset — 0.905 mV
10. Cathode dewpoint-water temperature — 0.832 mV

The remaining predictors produced smaller mean absolute attribution magnitudes within the fitted model.

The dominance of current is consistent with the strong dependence of instantaneous stack voltage on electrical loading. The presence of reactant-flow, pressure and thermal/humidification variables among the subsequent influential predictors indicates that the fitted XGBoost model also used information describing the operating condition of the fuel-cell system when estimating voltage.

These results describe the predictive behaviour of the trained model. They do not establish that the variables with the largest SHAP values are necessarily the dominant physical causes of long-term PEMFC degradation.

### 13.19.4 Direction and Dependence of Model Contributions

The global beeswarm and dependence analyses demonstrated that SHAP contribution magnitude alone does not describe the direction of a predictor's effect.

Current showed the strongest dependence relationship with its SHAP contribution. Across the final explanation sample, increasing current was strongly associated with increasingly negative SHAP contributions (Spearman ρ = -0.937). This indicates that, relative to the selected model reference output, higher-current operating conditions generally pushed the XGBoost voltage prediction downward, whereas lower-current conditions generally pushed it upward.

Strong structured relationships were also observed for:

- Total anode stack flow: ρ = -0.940
- Anode endplate temperature: ρ = -0.889
- Cathode pressure difference: ρ = +0.949
- Anode dewpoint-water temperature: ρ = -0.816
- Cathode dewpoint offset: ρ = +0.922

These relationships demonstrate how the fitted model distributed predictions across the observed operating regime. They should not be interpreted as isolated causal effects because several PEMFC operating variables are correlated, physically coupled or algebraically derived from other predictors.

### 13.19.5 Attribution Stability Across Later Durability Stages

The dominant attribution hierarchy remained highly stable across the 900, 950 and 1,000 h explanation stages.

Current remained the highest-ranked predictor at all three stages, with mean absolute SHAP magnitudes of approximately:

- 900 h: 78.533 mV
- 950 h: 79.639 mV
- 1,000 h: 79.544 mV

Total anode stack flow remained second at all three stages. Anode endplate temperature and cathode pressure difference occupied the third and fourth positions, with only a small rank interchange at 1,000 h. Total cathode stack flow remained fifth.

Pairwise stage comparisons produced:

- 900 vs 950 h: Spearman ρ = 0.991
- 900 vs 1,000 h: Spearman ρ = 0.982
- 950 vs 1,000 h: Spearman ρ = 0.971

Top-5 membership was identical in every comparison, and top-10 membership was also preserved across all three stages.

This indicates that the XGBoost model continued to rely on a broadly consistent predictor hierarchy when applied to later durability stages. Importantly, this stability does not mean that the physical state of the PEMFC remained unchanged. The previously observed increase in XGBoost prediction error from 900 to 1,000 h and the stability of the SHAP ranking describe different properties of the analysis. SHAP ranking stability therefore should not be used to explain the increase in later-stage prediction error directly.

### 13.19.6 Representative Local Explanations

Three representative observations were examined to demonstrate how individual predictions were assembled from the common SHAP reference output. One observation was selected from each of the 900, 950 and 1,000 h stages using an objective rule: the observation whose absolute XGBoost prediction error was closest to the median absolute prediction error of its respective stage.

The representative observations were:

- 900 h — original dataframe index 3,201,119; actual voltage 0.685200 V; predicted voltage 0.681279 V.
- 950 h — original dataframe index 3,354,208; actual voltage 0.726200 V; predicted voltage 0.730543 V.
- 1,000 h — original dataframe index 3,423,797; actual voltage 0.847700 V; predicted voltage 0.841734 V.

The local explanations illustrate why SHAP contributions must be interpreted relative to the operating point. For example, current contributed approximately -70.07 mV to the representative 900 h prediction at 14.814 A, -22.76 mV to the representative 950 h prediction at 9.485 A, and +80.45 mV to the representative 1,000 h prediction at 1.763 A.

This change in sign does not indicate that ageing caused the physical effect of current to reverse. Instead, the three observations occupy different electrical operating conditions relative to the common SHAP reference distribution. The local waterfall plots therefore explain individual model predictions rather than stage-level degradation mechanisms.

### 13.19.7 Robustness of the SHAP Interpretation

The final explanation configuration was evaluated using several complementary robustness checks.

Background-size sensitivity showed that the retained 500-observation development background produced a highly similar global attribution hierarchy to the larger 1,000-observation diagnostic reference:

- Spearman ρ = 0.997
- Top-5 overlap = 5/5
- Top-10 overlap = 10/10

Explanation-sample convergence showed that 7,500 observations were sufficient to reproduce the complete feature ranking obtained using the 30,000-observation diagnostic sample:

- Spearman ρ = 1.000
- Top-5 overlap = 5/5
- Top-10 overlap = 10/10

Repeated sampling with seeds 123 and 2026 also preserved the dominant hierarchy relative to the primary seed 42 sample:

- Seed 42 vs 123: ρ = 0.995
- Seed 42 vs 2026: ρ = 0.997
- Top-5 and top-10 membership remained identical.

The first twelve feature ranks were unchanged across the three sampled explanation datasets.

SHAP additivity was also verified. Across the 7,500 primary explanation observations, the mean absolute difference between the direct XGBoost prediction and the SHAP reconstruction was approximately 0.0013 mV, while the 99th percentile was approximately 0.0007 mV. One isolated observation produced a larger discrepancy of approximately 4.656 mV and was retained transparently rather than removed.

Finally, interventional TreeSHAP was compared with tree-path-dependent TreeSHAP. The two formulations produced:

- Spearman rank correlation = 0.976
- Top-5 overlap = 5/5
- Top-10 overlap = 10/10

The dominant attribution hierarchy was therefore stable to the alternative TreeSHAP formulation. However, individual attribution magnitudes were formulation-sensitive. Current, for example, had a mean absolute attribution of 79.239 mV under the interventional formulation and 72.049 mV under the tree-path-dependent formulation.

The robustness evidence therefore supports the stability of the dominant model-attribution hierarchy for the purposes of this study, but does not imply that individual SHAP magnitudes are uniquely determined physical effects.

### 13.19.8 Interpretation Boundaries

The SHAP results must be interpreted within the predictive scope of the dissertation.

SHAP explains the behaviour of the fitted XGBoost model. It does not independently identify electrochemical degradation mechanisms.

Accordingly:

- High SHAP importance indicates strong influence on the model's predictions, not causal physical importance.
- Low SHAP importance does not demonstrate that a variable is physically unimportant to PEMFC operation or degradation.
- Positive and negative SHAP values indicate movement of a model prediction relative to the selected reference output; they do not directly represent beneficial or harmful degradation effects.
- Mean absolute SHAP values measure attribution magnitude and remove directional information.
- Correlated, coupled and derived predictors can redistribute attribution among related variables.
- Interventional TreeSHAP can evaluate combinations that do not perfectly preserve the physical or algebraic dependence structure of the PEMFC variables.
- Tree-path-dependent TreeSHAP provides a useful sensitivity comparison but does not constitute a fully conditional physical explanation.
- Stable feature rankings across durability stages do not establish causal stability of PEMFC mechanisms.
- SHAP does not convert the instantaneous stack-voltage prediction model into a degradation-trajectory forecasting or remaining-useful-life model.

Physical interpretation must therefore be developed by combining SHAP evidence with PEMFC electrochemical knowledge, the preceding exploratory and association analyses, the dynamic operating data, standardized polarization measurements and relevant published literature.

### 13.19.9 Contribution to the Dissertation

The SHAP analysis adds an explainability layer to the predictive modelling framework developed in this dissertation.

The chronological modelling analysis established whether earlier-stage operational data could predict stack voltage at unseen later durability stages. SHAP subsequently identified how the selected XGBoost model distributed predictive importance among electrical-load, reactant-flow, pressure and thermal/humidification variables when making those later-stage predictions.

The resulting evidence therefore supports two distinct but complementary questions:

Predictive modelling:
How accurately does the model estimate stack voltage under unseen later-stage operating conditions?

↓

Model explainability:
Which measured operating variables does the fitted model rely upon most strongly, and how do those variables move individual voltage predictions relative to the model's reference output?

These questions remain distinct from the physical degradation assessment provided by the standardized polarization measurements. The final dissertation discussion should therefore integrate, rather than conflate, three forms of evidence:

1. Predictive evidence — chronological Ridge, Random Forest and XGBoost performance.
2. Explainability evidence — SHAP attribution patterns from the selected XGBoost model.
3. Degradation evidence — dynamic voltage behaviour and standardized polarization measurements across durability stages.

Together, these provide a condition-aware, data-driven assessment of PEMFC voltage behaviour and long-term performance deterioration while maintaining a clear distinction between predictive association, model attribution and physical degradation evidence.

### 13.19.10 Notebook Completion

Notebook 13 completed the post-hoc explainability stage of the modelling workflow. The final XGBoost model and predictor set remained frozen throughout the SHAP analysis, and no SHAP result was used to modify model training, tuning, feature selection or holdout evaluation.

The primary global, dependence, stage-wise and local explanation results were generated and subjected to background-size, explanation-sample-size, repeated-seed, additivity and feature-dependence sensitivity checks. Final numerical outputs were exported to the project results directory to provide a reproducible evidence base for the dissertation Results, Discussion and appendices.

No further analytical SHAP computation is required.